# From Diminishing Returns to Entrenchment: A Unifying Theory of Epistasis along Adaptive Walks Revealed by Fisher's Geometric Model

In [ ]:
#Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from typing import List, Tuple, Dict,Callable
from scipy import integrate
from scipy.stats import gaussian_kde,norm
from scipy.special import erfc, erfcx,gammaln
from scipy.optimize import brentq
from matplotlib.transforms import blended_transform_factory


# I. Angular distribution in High dimension 

## I.1 Distribution of angle for isotropic mutation dpeending on n 

In [ ]:
"""
Figure S1a
Angular density of isotropic mutations in Fisher's Geometric Model.

For an isotropic mutation in n dimensions, the angle theta between the
mutational vector and any fixed reference direction (here, the direction
to the optimum) has marginal density

    f(theta) = sin^{n-2}(theta) / Z_n ,        theta in [0, pi],

with normalisation (Wallis integral)

    Z_n = int_0^pi sin^{n-2}(theta) dtheta
        = sqrt(pi) * Gamma((n-1)/2) / Gamma(n/2).

As n grows, the density concentrates around pi/2 (equatorial concentration).
The large-n result of Section 1.3 states that theta ~ N(pi/2, 1/n); this
Gaussian approximation is shown as dashed curves for comparison.

Output: a single figure of the theoretical angular distribution for
n = 3, 10, 100. No data required.
"""


# --------------------------------------------------------------------------
# Global style (publication)
# --------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 0.9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "mathtext.fontset": "cm",
})

# --------------------------------------------------------------------------
# Angular density and its large-n Gaussian approximation
# --------------------------------------------------------------------------
def log_wallis(n):
    """log of Z_n = int_0^pi sin^{n-2}(theta) dtheta."""
    return 0.5 * np.log(np.pi) + gammaln((n - 1) / 2.0) - gammaln(n / 2.0)


def angular_density(theta, n):
    """Exact marginal density f(theta) = sin^{n-2}(theta) / Z_n.

    Evaluated in log-space to remain numerically stable for large n.
    """
    with np.errstate(divide="ignore"):
        log_f = (n - 2) * np.log(np.sin(theta)) - log_wallis(n)
    return np.exp(log_f)


def gaussian_approx(theta, n):
    """Large-n approximation: theta ~ N(pi/2, 1/n)."""
    return np.sqrt(n / (2 * np.pi)) * np.exp(-n * (theta - np.pi / 2) ** 2 / 2.0)


# --------------------------------------------------------------------------
# Data
# --------------------------------------------------------------------------
theta = np.linspace(1e-6, np.pi - 1e-6, 3000)   # avoid log(0) at the poles
n_values = [3, 10, 100]
colors = {3: "#1f77b4", 10: "#d62728", 100: "#2ca02c"}   # blue / red / green

# --------------------------------------------------------------------------
# Figure
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.4, 4.4))

for n in n_values:
    # exact density (solid)
    ax.plot(theta, angular_density(theta, n),
            color=colors[n], lw=2.2, label=fr"$n = {n}$", zorder=3)
    # Gaussian approximation (thin dashed, same colour)
    ax.plot(theta, gaussian_approx(theta, n),
            color=colors[n], lw=1.1, ls="--", alpha=0.75, zorder=2)

# reference line at the equator theta = pi/2
ax.axvline(np.pi / 2, color="0.45", lw=0.8, ls=":", zorder=1)

# --------------------------------------------------------------------------
# Axes: centre on pi/2, mark pi/4, pi/2, 3pi/4
# --------------------------------------------------------------------------
ax.set_xlim(0, np.pi)
ax.set_ylim(bottom=0)
ax.set_xticks([0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi])
ax.set_xticklabels([r"$0$", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])
ax.set_xlabel(r"Mutation angle $\theta$")
ax.set_ylabel(r"Density $f(\theta)$")
ax.grid(axis="y", color="0.9", lw=0.6, zorder=0)

# --------------------------------------------------------------------------
# Legend: n values + a single entry for the Gaussian approximation
# --------------------------------------------------------------------------
handles, labels = ax.get_legend_handles_labels()
handles.append(Line2D([0], [0], color="0.45", lw=1.1, ls="--"))
labels.append(r"$\mathcal{N}(\pi/2,\, 1/n)$")
ax.legend(handles, labels, frameon=False, loc="upper right",
           title_fontsize=11)

fig.tight_layout()
fig.savefig("angular_density_FGM.pdf", bbox_inches="tight")
fig.savefig("angular_density_FGM.png", dpi=800, bbox_inches="tight")
plt.show()

## I.2 Distribution of $X=-\sqrt(n) \cos \theta $ for all mutations , beneficial mutations , or selected mutations in the haldane approximation 

In [ ]:
"""
Three mutational ensembles at fixed mutational norm in Fisher's Geometric Model.

Standardised projection toward the optimum:
    U = -sqrt(n) * cos(theta) ~ N(0, 1)     (central regime, isotropic mutations)

A mutation is beneficial iff U > x, where x = sqrt(n) * r / (2d) is the Fisher
parameter. Once the adaptive walk reaches the optimum, a beneficial substitution
is entrenched iff x < U < 2x.

Densities shown (illustrated at x = 1):
    - new mutations        f0(u) = phi(u)
    - beneficial           fB(u) = phi(u) / Phibar(x)              for u > x
    - fixed substitutions  fF(u) = (u - x) * phi(u) / D(x)         for u > x
      obtained under weak-selection (Haldane) weighting p_fix ~ 2s, with the
      selection coefficient s proportional to the angular excess U - x, and
      D(x) = phi(x) - x * Phibar(x) the normalising constant.
    The beneficial and fixed densities are drawn only over their support u > x.

Mean standardised projections (shown as dashed vertical lines):
    - beneficial ensemble  Lambda(x) = phi(x) / Phibar(x)      (inverse Mills ratio)
    - fixed ensemble       M(x)      = Phibar(x) / D(x)
Fixation weighting biases substitutions toward the optimum, so M(x) > Lambda(x).

Figure only; no external data required. Do not run automatically.
"""


# --------------------------------------------------------------------------
# Densities and summary statistics
# --------------------------------------------------------------------------
def phi(u):
    """Standard normal density."""
    return norm.pdf(u)

def phibar(x):
    """Upper-tail probability 1 - Phi(x)."""
    return norm.sf(x)

def D(x):
    """Normalising constant of the fixed-substitution density: phi(x) - x*(1 - Phi(x))."""
    return phi(x) - x * phibar(x)

def f_new(u):
    """New mutations: standard normal density."""
    return phi(u)

def f_ben(u, x):
    """Beneficial mutations: standard normal truncated to U > x.

    Returns NaN for u <= x so that no spurious vertical segment is drawn at the
    truncation edge; the curve is displayed only over its support.
    """
    return np.where(u > x, phi(u) / phibar(x), np.nan)

def f_fix(u, x):
    """Fixed substitutions: size-biased by (U - x) under weak-selection weighting.

    Returns NaN for u <= x (support u > x only), as for the beneficial density.
    """
    return np.where(u > x, (u - x) * phi(u) / D(x), np.nan)

def lambda_mills(x):
    """Mean projection of beneficial mutations, E[U | U > x] = phi(x) / Phibar(x)."""
    return phi(x) / phibar(x)

def M_fixed(x):
    """Mean projection of fixed substitutions, E_fix[U | x] = Phibar(x) / D(x)."""
    return phibar(x) / D(x)

# --------------------------------------------------------------------------
# Parameters and grid
# --------------------------------------------------------------------------
x = 1.0
u = np.linspace(-3.5, 5.0, 3000)

# Colour code (matched to the three ensembles, print-safe)
COL_NEW = "#1f77b4"   # blue   : new mutations
COL_BEN = "#e08214"   # orange : beneficial
COL_FIX = "#2ca02c"   # green  : fixed
COL_BAND = "#9ecae1"  # light blue : entrenchment interval

# --------------------------------------------------------------------------
# Figure
# --------------------------------------------------------------------------

# Global style 
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 0.9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "mathtext.fontset": "cm",
})


fig, ax = plt.subplots(figsize=(7.0, 4.4))


# Densities (beneficial and fixed drawn only over their support u > x)
ax.plot(u, f_new(u),    color=COL_NEW, lw=2.3, zorder=3, label="New mutations")
ax.plot(u, f_ben(u, x), color=COL_BEN, lw=2.3, zorder=3,
        label="Conditioned on being beneficial")
ax.plot(u, f_fix(u, x), color=COL_FIX, lw=2.3, zorder=3,
        label="Conditioned on fixation")


# Mean standardised projections: dashed vertical lines, Lambda(x) < M(x)
lam, Mx = lambda_mills(x), M_fixed(x)

ax.axvline(1, color="black", lw=2, ls="-", zorder=3,
           label=r"x=1")

ax.axvline(lam, color=COL_BEN, lw=1.7, ls="--", zorder=2,
           label=r"$\Lambda(x)$: mean projection, beneficial")
ax.axvline(Mx, color=COL_FIX, lw=1.7, ls="--", zorder=2,
           label=r"$M(x)$: mean projection, fixed")

# --------------------------------------------------------------------------
# Axes, legend, title
# --------------------------------------------------------------------------
ax.set_xlim(-3.5, 5.0)
ax.set_ylim(0, 1.62)
ax.set_xticks(np.arange(-3, 6, 1))
ax.set_xlabel(r"Standardised projection toward the optimum "
              r"$U = -\sqrt{n}\,\cos\theta$")
ax.set_ylabel("Density")
#ax.set_title(rf"Three mutational ensembles at fixed mutational norm " rf"($x = {x:.0f}$)", fontsize=12)

ax.legend(frameon=False, loc="upper left", fontsize=9.5)

fig.tight_layout()
fig.savefig("three_ensembles_FGM.pdf", bbox_inches="tight")
fig.savefig("three_ensembles_FGM.png", dpi=300, bbox_inches="tight")
plt.show()

## I.3 Function $\Lambda(x)$ and $M (x)$ and sign change 

In [ ]:
"""
Sign-change threshold x* of the mean epistasis between successive substitutions.

The mean pairwise epistasis changes sign where the mean standardised projection
of the incoming mutation crosses the reversal boundary y = 2x. The relevant mean
depends on the conditioning ensemble:

    without fixation (beneficial only):   Lambda(x) = phi(x) / Phibar(x)   (inverse Mills)
    with fixation (2s-weighted):          M(x)      = 1 / (Lambda(x) - x)

so the two thresholds solve

    x*_nofix :  Lambda(x) = 2x        (expected ~ 0.612)
    x*_fix   :  M(x)      = 2x        (expected ~ 0.925)

Fixation weighting biases substitutions toward the optimum, so M(x) > Lambda(x)
and the threshold shifts upward: x*_fix > x*_nofix.

Standalone; no external cell required. Do not run automatically.
"""


# ── Mean standardised projection of the incoming mutation ─────────────────────
def lambda_mills(x):
    """Inverse Mills ratio Lambda(x) = phi(x) / Phibar(x), the mean projection of
    beneficial mutations (no fixation weighting).

    Stable rewrite via the scaled complementary error function:
        Lambda(x) = sqrt(2/pi) / erfcx(x / sqrt(2)),
    exact for all x >= 0 with no tail underflow.
    """
    return np.sqrt(2.0 / np.pi) / erfcx(x / np.sqrt(2.0))

def M_fixed(x):
    """Mean projection of fixed substitutions, M(x) = 1 / (Lambda(x) - x).

    Equivalent to Phibar(x) / D(x) with D(x) = phi(x) - x*Phibar(x); the compact
    form is used here. Lambda(x) > x for all x >= 0, so the denominator is positive.
    """
    return 1.0 / (lambda_mills(x) - x)

# ── Sign-change thresholds: mean projection meets the reversal boundary y = 2x ─
def x_star(with_fixation, bracket=(1e-3, 3.0), xtol=1e-12):
    """Return x* solving mean_projection(x) = 2x on `bracket`.

    with_fixation : bool
        False -> Lambda(x) = 2x  (beneficial ensemble, geometric baseline)
        True  -> M(x)      = 2x  (fixation-weighted ensemble)
    """
    mean_proj = M_fixed if with_fixation else lambda_mills
    g = lambda x: mean_proj(x) - 2.0 * x        # increasing through zero on the bracket
    return brentq(g, *bracket, xtol=xtol)

# ── Report ────────────────────────────────────────────────────────────────────
x_star_nofix = x_star(with_fixation=False)   # beneficial:  Lambda(x) = 2x
x_star_fix   = x_star(with_fixation=True)    # fixed:       M(x)      = 2x

print(f"x*  without fixation (Lambda = 2x) : {x_star_nofix:.12f}")
print(f"x*  with fixation    (M      = 2x) : {x_star_fix:.12f}")
print(f"upward shift due to fixation weighting: "
      f"{x_star_fix - x_star_nofix:.12f}")

In [ ]:
"""
Lambda(x) and M(x) in Fisher's Geometric Model as functions of the Fisher parameter x.

Standardised projection: U = -sqrt(n) * cos(theta) ~ N(0, 1) in the central regime.
A mutation is beneficial iff U > x, with x = sqrt(n) * r / (2d).

Two functions of x:
    - Lambda(x) = phi(x) / Phibar(x)          (inverse Mills ratio)
                = mean standardised projection of the beneficial ensemble.
    - M(x)      = 1 / (Lambda(x) - x)
                = mean standardised projection of the fixed ensemble under
                  weak-selection (Haldane) weighting p_fix ~ 2s. Algebraically
                  identical to Phibar(x) / D(x), with D(x) = phi(x) - x * Phibar(x).

Sign-change thresholds of mean epistasis are the intersections with the line y = 2x:
    - x_B: solves Lambda(x) = 2x   (geometric baseline, x* ~ 0.612)
    - x_F: solves M(x)      = 2x   (fixation-weighted, x*_sel ~ 0.925)
Since M(x) > Lambda(x), fixation weighting shifts the threshold to a larger x_F.
The two thresholds are reported below the x-axis, stacked to avoid overlap.

Figure only; no external data required. Do not run automatically.
"""



# --------------------------------------------------------------------------
# Functions of x
# --------------------------------------------------------------------------
def phi(x):
    """Standard normal density."""
    return norm.pdf(x)

def phibar(x):
    """Upper-tail probability 1 - Phi(x)."""
    return norm.sf(x)

def lambda_mills(x):
    """Inverse Mills ratio Lambda(x) = phi(x) / Phibar(x)."""
    return phi(x) / phibar(x)

def M_fixed(x):
    """M(x) = 1 / (Lambda(x) - x), equal to Phibar(x) / D(x) with
    D(x) = phi(x) - x*Phibar(x)."""
    return 1.0 / (lambda_mills(x) - x)

# --------------------------------------------------------------------------
# Sign-change thresholds: intersections with y = 2x
# --------------------------------------------------------------------------
x_B = brentq(lambda x: lambda_mills(x) - 2 * x, 0.1, 3.0)   # ~ 0.612 (beneficial)
x_F = brentq(lambda x: M_fixed(x) - 2 * x, 0.1, 3.0)        # ~ 0.925 (fixed)

# --------------------------------------------------------------------------
# Grid
# --------------------------------------------------------------------------
x = np.linspace(0.0, 5.0, 1500)

# Colour code, consistent with the ensemble figure
COL_BEN = "#e08214"   # orange : beneficial ensemble
COL_FIX = "#2ca02c"   # green  : fixed ensemble

# --------------------------------------------------------------------------
# Figure
# --------------------------------------------------------------------------
# Global style 

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.linewidth": 0.9,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "mathtext.fontset": "cm",
})


fig, ax = plt.subplots(figsize=(6.6, 4.6))

# Reference lines y = x (asymptote) and y = 2x (threshold line)
ax.plot(x, x, color="0.55", lw=1.1, ls="--", zorder=1, label=r"$y=x$ (asymptote)")
ax.plot(x, 2 * x, color="0.15", lw=1.1, ls="-.", zorder=1, label=r"$y=2x$")

# The two functions
ax.plot(x, lambda_mills(x), color=COL_BEN, lw=2.3, zorder=3,
        label=r"$\Lambda(x)=\varphi(x)/\overline{\Phi}(x)$")
ax.plot(x, M_fixed(x), color=COL_FIX, lw=2.3, zorder=3,
        label=r"$M(x)=1/\left(\Lambda(x)-x\right)$")

# --------------------------------------------------------------------------
# Intersection points x_B and x_F: marker + short dotted guide to the axis.
# Threshold values are stacked below the x-axis (x_B above x_F) so the two
# nearby labels never overlap. x in data coordinates, y in axis fraction.
# --------------------------------------------------------------------------
trans = blended_transform_factory(ax.transData, ax.transAxes)
for xv, col in [(x_B, COL_BEN), (x_F, COL_FIX)]:
    ax.plot([xv, xv], [0, 2 * xv], color=col, lw=1.0, ls=":", zorder=2)
    ax.plot(xv, 2 * xv, "o", color="black", ms=5, zorder=5)

ax.annotate(fr"$x_B\approx{x_B:.3f}$", xy=(x_B, 0), xycoords=trans,
            xytext=(0, -22), textcoords="offset points",
            ha="center", va="top", fontsize=10, color=COL_BEN)
ax.annotate(fr"$x_F\approx{x_F:.3f}$", xy=(x_F, 0), xycoords=trans,
            xytext=(0, -37), textcoords="offset points",
            ha="center", va="top", fontsize=10, color=COL_FIX)

# --------------------------------------------------------------------------
# Axes, legend
# --------------------------------------------------------------------------
ax.set_xlim(0, 5)
ax.set_ylim(0, 5.5)
ax.set_xlabel(r"Fisher parameter $x$", labelpad=34)
ax.set_ylabel(r"Function value")
ax.legend(frameon=False, loc="upper left", fontsize=10)

fig.tight_layout()
fig.savefig("lambda_M_FGM.pdf", bbox_inches="tight")
fig.savefig("lambda_M_FGM.png", dpi=300, bbox_inches="tight")
plt.show()

# II. Sample mutation x following the posterior distribution $\mathbb{P}(x|\text{beneficial})$ and $\mathbb{P}(x|\text{fixed})$

## II.1 Rejection sampling

In [ ]:
# ============================================================
# Kimura fixation probability (haploid population)
# ============================================================


def kimura_pfix(s, Ne):
    """
    Exact diffusion fixation probability (diploid genic, initial freq 1/(2Ne)):
        pi(s) = (1 - e^{-2s}) / (1 - e^{-4 Ne s})
    Limits: s->0 : 1/(2Ne) ; s>0, Ne s>>1 : ~2s ; s<0 : ~2|s|/ (e^{4Ne|s|}-1) (drift-limited).
    Stable & vectorised for any sign of s. If your simulator uses a different
    convention, replace this function (it is the only place pi(s) is defined).
    """
    s = np.asarray(s, float)
    num = -np.expm1(np.clip(-2.0 * s,      -700.0, 700.0))
    den = -np.expm1(np.clip(-4.0 * Ne * s, -700.0, 700.0))
    return np.clip(np.where(np.abs(den) < 1e-300, 1.0 / (2.0 * Ne), num / den), 0.0, 1.0)



def draw_mutation_by_rejection(z:np.ndarray, sigma_m:float, Ne: float=1e7, max_attempts: int=10**6,kimura=True,only_beneficial:bool=True,exact=True)->np.ndarray:

    """
    Draw a mutation vector using naive rejection sampling 
    
   
    Parameters
    ----------
    z : np.ndarray
        Current phenotype, shape (n,)
    sigma_m : float
        Std of mutational effect
    Ne : float
        Effective population size
    max_attempts : int
        Max attempts before failing
    kimura : bool
        Whether to use Kimura fixation probability
    only_beneficial : bool
        Whether to accept only beneficial mutations

    Returns
    -------
    x : np.ndarray
        Mutation vector, shape (n,)
    nb_of_attempts : int
        Number of attempts taken to draw a valid mutation
    """
    z_norm = np.linalg.norm(z)
    u= z / z_norm #Unit vector from vect(z)
    n = len(z)
    nb_of_attempts:int = 0
    accepted:bool= False
    if not only_beneficial and not kimura: #just draw any mutation, no rejection
        mutation = np.random.normal(0, sigma_m, size=n)
        return mutation, 1
    else:
        while not accepted and nb_of_attempts <= max_attempts:
            # --- propose mutation
            mutation = np.random.normal(0, sigma_m, size=n)
            nb_of_attempts += 1
            #log_selection_coefficientp1(z, mutation) = log(1+s) = log(W_mut / W_res)=-(mutation,z)-||mutation||^2/2
            logsp1=-np.dot(mutation, z) - 0.5 * np.linalg.norm(mutation)**2
            if only_beneficial and logsp1 <= 0:
                continue
            if kimura and exact:
                s=np.expm1(logsp1)
                pfix = kimura_pfix(s, Ne)
                if np.random.rand() < pfix:
                    accepted = True
            elif kimura and not exact:
                # Approximate fixation probability for small s: P_fix ≈ 2s ≈ 2*(-delta)
                #Only accept mutations with negative delta = positive s
                pfix=2*logsp1 
                if np.random.rand() < pfix:
                    accepted = True
            else:# without Kimura fixation probability, just accept any  mutation
                accepted = True
        if not accepted:
            raise RuntimeError("Failed to draw mutation after max_attempts")
        return mutation, nb_of_attempts

In [ ]:
# Example usage
z=np.zeros(20)
z[0] = 1  # Example phenotype in 3D
sigma_m = 0.1  # Std of mutational effect
Ne = 1e7  # Effective population size
draw_mutation_by_rejection(z, sigma_m, Ne,only_beneficial=False,kimura=True,exact=True)

## II.2 Rosenblatt sampler

In [ ]:

# ============================================================================
# Unified direct sampler of FIXED substitutions in the isotropic Gaussian
# Fisher Geometric Model, by inverse-CDF (inverse transform) sampling.
#
# Works over the FULL sphere in (r, u = cos theta), u in (-1, -R/2D), so it handles
# only beneficial fixations. Exact, independent draws;
# Cost O(1) in n (the n-2 transverse directions are integrated analytically,
# contributing the (1-u^2)^{(n-3)/2} factor). Valid arbitrarily close to the optimum.
#
# OPTIONS
# -------
# probability_of_fixation (the fixation weight pi(s), up to a constant):
#     "1"          pi = 1                      every mutation fixes (no selection filter)
#     "2s"         pi = 2 s                    Haldane linear   (beneficial: pi=0 for s<=0)
#     "expm1(2s)"  pi = 1-e^{-2s} (~2s)      exponential form (beneficial: pi=0 for s<=0)
#     "exact"      pi = (1-e^{-2s})/(1-e^{-4 Ne s})   full Kimura, BOTH signs, uses Ne
# Ne              : effective population size  (only used by "exact"; the drift knob
#                   that sets the deleterious-fixation rate in the |Ne s| ~ 1 regime)
##
# Log-fitness effect:  s = -d r u - r^2/2   (u = cos theta; beneficial <=> s > 0 <=> u < -r/2d)
# Mutation vector from (r, u) about the unit radial axis e (points away from optimum):
#     radial     = r u e             (u<0 -> toward optimum)
#     transverse = r sqrt(1-u^2) * (uniform unit vector orthogonal to e)
# ============================================================================


# ----------------------------------------------------------------- pi(s ; Ne)
def kimura_pfix(s, Ne):
    """
    Exact diffusion fixation probability (diploid genic, initial freq 1/(2Ne)):
        pi(s) = (1 - e^{-2s}) / (1 - e^{-4 Ne s})
    Limits: s->0 : 1/(2Ne) ; s>0, Ne s>>1 : ~2s ; s<0 : ~2|s|/ (e^{4Ne|s|}-1) (drift-limited).
    Stable & vectorised for any sign of s. If your simulator uses a different
    convention, replace this function (it is the only place pi(s) is defined).
    """
    s = np.asarray(s, float)
    num = -np.expm1(np.clip(-2.0 * s,      -700.0, 700.0))
    den = -np.expm1(np.clip(-4.0 * Ne * s, -700.0, 700.0))
    return np.clip(np.where(np.abs(den) < 1e-300, 1.0 / (2.0 * Ne), num / den), 0.0, 1.0)


def _fixation_weight(s, Ne, prob):
    """Return the (non-negative) fixation weight pi(s) for the chosen option."""
    if   prob == "1":         w = np.ones_like(s)
    elif prob == "2s":        w = 2.0 * s
    elif prob == "expm1(2s)": w = -np.expm1(-2.0 * s)          # 1-exp(-2s)
    elif prob == "exact":     w = kimura_pfix(s, Ne)
    else:
        raise ValueError("probability_of_fixation must be '1', '2s', 'expm1(2s)' or 'exact'")
    return np.clip(w, 0.0, None)                             # weights must be >= 0


# ----------------------------------------------------------- inverse-CDF tables
# ----------------------------------------------------------- inverse-CDF tables

def _ru_grid(d, sigma, n, Ne, prob, n_r=800, n_u=800, cap=12.0):
    """
    Construct lookup tables for numerical inverse-transform sampling of fixed
    mutations in (r, u), with r = ||delta|| and u = cos(theta).

    For a phenotype at distance d from the optimum, the target density is

        f_fix(r,u | d) ∝ f_mut(r,u) * pi(s(r,u)),

    with
        Delta(r,u) = -d*r*u - r^2/2,
        s(r,u) = exp(Delta) - 1,

    and, for isotropic Gaussian mutations,

        f_mut(r,u) ∝ r^(n-1) exp[-r^2/(2 sigma^2)]
                     (1-u^2)^((n-3)/2).

    The beneficial domain
        0 < r < 2d,  -1 < u < -r/(2d)
    is reparameterized by
        v = (u + 1) / (1 - r/(2d)) in (0,1),
    so that the whole angular grid is used inside the admissible domain.
    The Jacobian du/dv = 1 - r/(2d) is included in the density.

    The function returns the radial grid, the angular sampling coordinate,
    the marginal CDF of r, and the conditional angular CDF given r.
    The actual inverse transform is performed in `_sample_ru`.
    """


    if d <= 0:
        raise ValueError("No strictly beneficial mutation exists at d <= 0.")
    r_max = min(sigma * (np.sqrt(n) + cap), 2.0 * d * (1.0 - 1e-12))
    r = (np.arange(n_r) + 0.5) / n_r * r_max # n_r radial midpoints in (0, r_max)
    v = (np.arange(n_u) + 0.5) / n_u # n_u angular midpoints in (0, 1)

    R, V = np.meshgrid(r, v, indexing='ij')
    u_c = -R / (2.0 * d) #Geometric constraint: u < -r/(2d) for beneficial mutations
    U = -1.0 + V * (1.0 + u_c) #Change of variable: v = (u + 1) / (1 - r/(2d)) in (0,1)
    jac = 1.0 + u_c


    Delta = -d * R * U - 0.5 * R**2 #Log-fitness effect: Delta = log(1+s) = -d r u - r^2/2
    s = np.expm1(Delta) #Selection coefficient: s = exp(Delta) - 1
    w = _fixation_weight(s, Ne, prob) #Fixation weight: pi(s)

    log_base = (
        (n - 1) * np.log(R)
        - R**2 / (2.0 * sigma**2)
        + ((n - 3) / 2.0) * np.log(np.maximum(1.0 - U**2, 1e-300))
    )

    log_base += np.log(np.maximum(jac, 1e-300))

    log_w = np.full_like(w, -np.inf)
    log_w = np.log(np.maximum(w, 1e-300))

    log_f = log_base + log_w
    finite = np.isfinite(log_f)

    if not np.any(finite):
        raise ValueError("Zero fixation-weighted density on the whole grid.")

    f = np.zeros_like(log_f)
    f[finite] = np.exp(log_f[finite] - np.max(log_f[finite])) # renormalize to avoid overflow

    # Marginal CDF of r
    cdf_r = np.cumsum(f.sum(axis=1))
    cdf_r /= cdf_r[-1] # Normalisation 

    # Conditional CDF of the angular coordinate given r
    cdf_v = np.cumsum(f, axis=1)
    cdf_v /= np.maximum(cdf_v[:, -1:], 1e-300)

    return r, v, cdf_r, cdf_v


def _sample_ru(d, sigma, n, N, rng, Ne, prob, n_r=800, n_u=800, cap=12.0):
    """
    Sample N independent beneficial mutation pairs (r, u) from the
    fixation-weighted distribution corresponding to a single distance d
    from the optimum.

    `_ru_grid` constructs the marginal CDF of r and the conditional CDF
    of the transformed angular coordinate v given r, with

        v = (u + 1) / (1 - r/(2d)) in (0,1).

    Sampling proceeds as

        R = F_R^{-1}(X1),
        V = F_{V|R}^{-1}(X2),

    where X1 and X2 are independent Uniform(0,1) variables. The physical
    angular coordinate is then reconstructed as

        u = -1 + v * (1 - r/(2d)).

    Uniform sub-cell jitter is added in r and v so that samples are not
    restricted to grid midpoints.
    """

    r, v, cdf_r, cdf_v = _ru_grid(
        d, sigma, n, Ne, prob,
        n_r=n_r, n_u=n_u, cap=cap
    )

    # Sample r from its marginal CDF
    ri = np.clip(
        np.searchsorted(cdf_r, rng.random(N)),
        0, len(r) - 1
    )

    # Sample v conditionally on the sampled r
    vi = np.clip(
        (cdf_v[ri] < rng.random(N)[:, None]).sum(axis=1),
        0, len(v) - 1
    )

    # Uniform jitter within the selected grid cells
    dr = r[1] - r[0]
    dv = v[1] - v[0]

    r_out = r[ri] + (rng.random(N) - 0.5) * dr
    v_out = v[vi] + (rng.random(N) - 0.5) * dv

    # Keep jittered values inside the beneficial parameter domain
    r_out = np.clip(r_out, 1e-12, 2.0 * d * (1.0 - 1e-12))
    v_out = np.clip(v_out, 1e-12, 1.0 - 1e-12)

    # Transform back from v to the physical angular coordinate u = cos(theta)
    u_out = -1.0 + v_out * (1.0 - r_out / (2.0 * d))

    return r_out, u_out



def _uniform_orthogonal(e, rng):
    """Unit vectors uniform on the sphere orthogonal to each unit row of e (N, n)."""
    g = rng.normal(size=e.shape)
    g -= np.einsum('ij,ij->i', g, e)[:, None] * e
    return g / np.linalg.norm(g, axis=1, keepdims=True)


def _mutation_from_ru(r, u, e, rng):
    """Full n-dim mutation vectors from (r, u) about unit radial axes e (N, n)."""
    radial     = (r * u)[:, None] * e
    transverse = (r * np.sqrt(np.clip(1 - u**2, 0, 1)))[:, None] * _uniform_orthogonal(e, rng)
    return radial + transverse


# ------------------------------------------------------------- public samplers
def sample_fixed_mutations(
z,
    sigma,
    N,
    rng,
    Ne=1e3,
    probability_of_fixation="exact",
):
    """
    Sample N fixation-weighted beneficial mutation vectors from the current
    phenotype z.

    The distance to the optimum is computed as

        d = ||z||,

    and the associated radial unit vector is

        e = z / ||z||.

    The same current phenotype is used for all N sampled mutations, so the
    radial direction e is repeated N times before reconstructing the full
    n-dimensional mutation vectors from their sampled (r,u) coordinates.

    Parameters
    ----------
    z : array-like, shape (n,)
        Current phenotype relative to the optimum.

    sigma : float
        Per-trait standard deviation of the isotropic Gaussian mutation model.

    N : int
        Number of mutation vectors to sample.

    rng : numpy.random.Generator
        Random-number generator.

    Ne : float
        Effective population size used in the fixation probability.

    probability_of_fixation : str
        Fixation-probability model passed to `_sample_ru`.

    Returns
    -------
    delta : ndarray, shape (N, n)
        N independent fixation-weighted beneficial mutation vectors.
    """

    z = np.asarray(z, dtype=float)

    if z.ndim != 1:
        raise ValueError("z must be a one-dimensional phenotype vector.")

    n = len(z)
    d = np.linalg.norm(z)

    if d <= 0:
        raise ValueError("No strictly beneficial mutation exists at the optimum.")

    # Unit radial direction of the current phenotype
    e0 = z / d

    # Same current radial axis for all N sampled mutations
    e = np.broadcast_to(e0, (N, n))

    # Sample radial length and angle relative to the current phenotype
    r, u = _sample_ru(
        d,
        sigma,
        n,
        N,
        rng,
        Ne,
        probability_of_fixation,
    )

    # Reconstruct the full n-dimensional mutation vectors
    return _mutation_from_ru(r, u, e, rng)



def sample_adaptive_walks_cdf(z, sigma, n_steps, n_rep=20000, rng=None,
                          Ne=1e3, probability_of_fixation="exact"):
    """
    Generate n_rep independent adaptive walks starting from the same phenotype z.

    All walks share the same initial phenotype, so the first substitution is sampled
    jointly for all repetitions. After this first step, each walk has its own current
    phenotype and therefore its own distance and radial direction relative to the
    optimum. Subsequent substitutions are sampled separately for each walk.

    Parameters
    ----------
    z : array-like, shape (n,)
        Initial phenotype relative to the optimum.
    sigma : float
        Per-trait standard deviation of the isotropic Gaussian mutation model.
    n_steps : int
        Number of fixed beneficial substitutions per walk.
    n_rep : int
        Number of independent adaptive walks.
    rng : numpy.random.Generator, optional
        Random-number generator.
    Ne : float
        Effective population size used in the fixation probability.
    probability_of_fixation : str
        Fixation-probability model passed to `sample_fixed_mutations`.

    Returns
    -------
    phenotypes : ndarray, shape (n_rep, n_steps + 1, n)
        Phenotypic states along each walk. phenotypes[:, 0, :] contains the
        common initial phenotype.
    mutations : ndarray, shape (n_rep, n_steps, n)
        Fixed mutation vectors along each walk.
    """

    rng = rng or np.random.default_rng()
    z = np.asarray(z, dtype=float)

    if z.ndim != 1:
        raise ValueError("z must be a one-dimensional phenotype vector.")
    if np.linalg.norm(z) <= 0:
        raise ValueError("Adaptive walks cannot start exactly at the optimum.")

    n = len(z)
    phenotypes = np.empty((n_rep, n_steps + 1, n))
    mutations = np.empty((n_rep, n_steps, n))
    phenotypes[:, 0, :] = z

    if n_steps == 0:
        return phenotypes, mutations

    # First step: all walks start from the same phenotype, so all mutations
    # can be sampled in one vectorized call.
    delta = sample_fixed_mutations(
        z, sigma, n_rep, rng,
        Ne=Ne, probability_of_fixation=probability_of_fixation
    )
    mutations[:, 0, :] = delta
    phenotypes[:, 1, :] = z + delta

    # Later steps: each walk has its own current phenotype.
    for step in range(1, n_steps):
        for rep in range(n_rep):
            z_current = phenotypes[rep, step, :]
            delta = sample_fixed_mutations(
                z_current, sigma, 1, rng,
                Ne=Ne, probability_of_fixation=probability_of_fixation
            )[0]
            mutations[rep, step, :] = delta
            phenotypes[rep, step + 1, :] = z_current + delta

    return phenotypes, mutations

def sample_two_successive_mutations_cdf(z, sigma, n_rep=20000, rng=None, Ne=1e7,
                                        probability_of_fixation="exact", nbins=96):
    """
    Sample two successive fixed beneficial mutations from n_rep adaptive walks
    starting from the same phenotype z.

    The first mutation is sampled in bulk. After this first substitution, the
    post-step distances d1 differ across replicates. They are grouped into
    `nbins`, and one inverse-CDF table is constructed per bin for the second
    substitution. Within each bin, all mutations are sampled jointly using the
    mean distance of that bin.

    Returns
    -------
    delta1, delta2 : ndarray, shape (n_rep, n)
        First and second fixed mutations.
    z1 : ndarray, shape (n_rep, n)
        Phenotypes after the first substitution.
    """
    rng = rng or np.random.default_rng()
    z = np.asarray(z, dtype=float)
    n = len(z)

    # First substitution: common initial phenotype, hence one bulk draw
    delta1 = sample_fixed_mutations(
        z, sigma, n_rep, rng, Ne=Ne,
        probability_of_fixation=probability_of_fixation
    )
    z1 = z[None, :] + delta1
    d1 = np.linalg.norm(z1, axis=1)
    e1 = z1 / d1[:, None]
    delta2 = np.empty_like(delta1)

    # Bin the distances reached after the first substitution
    edges = np.linspace(d1.min(), d1.max() + 1e-12, nbins + 1)
    bins = np.clip(np.digitize(d1, edges) - 1, 0, nbins - 1)

    # One CDF table per distance bin; mutations are sampled in bulk within bins
    for b in range(nbins):
        idx = np.flatnonzero(bins == b)
        if idx.size == 0:
            continue
        d_bin = d1[idx].mean()
        r, u = _sample_ru(d_bin, sigma, n, idx.size, rng, Ne, probability_of_fixation)
        delta2[idx] = _mutation_from_ru(r, u, e1[idx], rng)

    return delta1, delta2, z1

Appraise robustness of discretization of $d_i$ ( distance after the first step) 

In [ ]:
def compare_distance_binning(z, sigma, nbins_list=(12, 24, 48, 96),
                             n_rep=500, Ne=1e3, seed=1234):
    """
    Compare mean and variance of consecutive epistasis obtained with distance
    binning against the exact per-replicate second-step sampler.
    """
    rng = np.random.default_rng(seed)
    _,mut = sample_adaptive_walks_cdf(
        z, sigma,n_steps=2, n_rep=n_rep, rng=rng, Ne=Ne
    )
    d1,d2=mut[:, 0, :], mut[:, 1, :]
    eps_ref = -np.einsum("ij,ij->i", d1, d2)
    mean_ref, var_ref = eps_ref.mean(), eps_ref.var(ddof=1)

    print(f"Exact:    E[eps]={mean_ref:.6g}, Var[eps]={var_ref:.6g}")

    for nbins in nbins_list:
        rng = np.random.default_rng(seed + nbins)
        d1, d2, _ = sample_two_successive_mutations_cdf(
            z, sigma, n_rep=n_rep, rng=rng, Ne=Ne, nbins=nbins
        )
        eps = -np.einsum("ij,ij->i", d1, d2)
        mean, var = eps.mean(), eps.var(ddof=1)

        print(
            f"{nbins:3d} bins: "
            f"E[eps]={mean:.6g} ({(mean-mean_ref)/abs(mean_ref):+.2%}), "
            f"Var[eps]={var:.6g} ({(var-var_ref)/var_ref:+.2%})"
        )

z = np.zeros(5)
z[0] = 1.0

compare_distance_binning(
    z=z,
    sigma=0.1,
    nbins_list=(12, 24, 48, 96),
    n_rep=500,
    Ne=1e7
)

## II.3 MCMC - MH algorithm

In [ ]:
class MetropolisHastingsAR:
    """
    Metropolis-Hastings sampler for the (a, r) reduced problem.

    The original high-dimensional sampling problem is reduced using symmetry:
        x = a * u + r * w
    where:
        - u is the direction of z
        - w is uniform on the (d-1)-sphere orthogonal to u

    We sample (a, r) in a 2D disk with density:

        pi(a,r) ∝ (-delta) * exp(-(a^2 + r^2)/(2σ^2)) * r^(d-2)

    with:
        delta = a^2 + r^2 + 2 a ||z||

    Constraint:
        delta < 0  (inside disk)
    """

    def __init__(self):
        self.sigma = None
        self.target_log_density = None
        self.nsteps = None
        self.a0 = None
        self.r0 = None 
        self.posterior_ar = None #After burnin and thinning, store (a, r) samples here
        self.last_sample_x= None
        self.x_mut=None
        self.acceptance_rate = None
        self.thinning = 1
        self.labels = ["a", "r"]
        self.alpha_a_prop = None
        self.alpha_r_prop = None
        self.z_norm=None
        self.z=None
        self.Ne=None
        self.all_samples_ar=None #All (a, r) samples without burnin and thinning, for diagnostics
        self.prop_burnin=None
        self.burnin=None
        self.kimura=None
        self.only_beneficial=None
        self.forward=True
        self.exact=None

    # --------------------------------------------------
    # Problem setup
    # --------------------------------------------------

    def set_problem(self, z: np.ndarray, sigma: float,Ne: float=1e7,kimura:bool=True,only_beneficial:bool=True,forward:bool=True,exact:bool=True):
        """
        Define the problem parameters.

        Parameters
        ----------
        z : ndarray
            Initial vector in R^n
        sigma : float
            Mutational sd 
        n : int
            Dimension of the problem
        kimura : bool
            Wether to use Kimura fixation probability in log_pi or just keep favorable mutations
        only_beneficial : bool
            Whether to accept only beneficial mutations in log_pi or also slightly deleterious ones (with Kimura fixation probability)
        
        """
        self.z = np.array(z)
        self.z_norm = np.linalg.norm(z)
        self.u = self.z / self.z_norm #unit vector in direction of z
        self.sigma = sigma
        self.n = len(z)
        self.only_beneficial = only_beneficial # Whether to accept only beneficial mutations in log_pi or both favorable and deleterious ones (depending onimura fixation probability)
        self.kimura = kimura # Fixation probability term in log_pi or not (only beneficial mutations or all mutations)
        self.Ne = Ne
        self.exact=exact # Kimura fixation probability exact or approximate (2s)
        

    def log_pi(self, a: float, r: float) -> float:
        """
        Log-density of (a,r) distribution for fixed mutations.

        Parameters
        ----------
        a : float
            Projection along z direction.
        r : float
            Norm of orthogonal component.

        Returns
        -------
        float
            Log-density (unnormalized). Returns -inf if outside support.
        
        Notes
        -----
        - delta = a^2 + r^2 + 2 a ||z||
        - Support: inside the disk: (a + ||z||)^2 + r^2 < ||z||^2
                r > 0
        - Gaussian term: exp(-(a^2 + r^2)/(2 sigma^2))
        - Jacobian term for polar coordinates: r^(n-2)
        """

        # ---------------------------
        # Geometric constraint
        # ---------------------------

        #principle : 2 conditions -> (i,j) in Only beneficial x kimura fixation probability
        # (0,0): accept all mutations
        # (0,1): accept all mutations but with Kimura fixation probability
        # (1,0): accept only beneficial mutations without Kimura fixation probability
        # (1,1) : accept only beneficial mutations with Kimura fixation probability


        delta = a**2 + r**2 + 2 * a * self.z_norm
        
        if self.only_beneficial:
            # Only beneficial mutations
            if delta >= 0:
                return -np.inf

        if self.kimura and self.exact:
        # ---------------------------
        # Fixation probability / selection term . 
        #  kimura is True-> else just keep favorable/all mutations(depending on only_beneficial)
        # ---------------------------
        #Expm1: Calculate exp(x) - 1 for all elements in the array.
        # S = exp(-delta/2) - 1, numerically stable

            S = np.expm1(-delta / 2) # = exp(-delta/2) - 1
            Ne=self.Ne       
            NeS=Ne*S    
            # Exact fixation probability for large Ne: P_fix = 1 - exp(-2*S)
            # Use expm1 for numerical stability
            # Protect against numerical issues
            #epsilon = 1e-100
            
            if abs(NeS) < 1e-2:  # For very small s, use neutral approximation 1/Ne
                num=1
                denom=2*Ne
                S=0
            if NeS < -1e2:  # Avoid overflow in exp(-4Ne s) for large negative s
                num=0  # Approximation for large negative s
                denom=1
            elif NeS > 1e2: # Avoid overflow in exp(-4Ne s) for large positive s
                num=-np.expm1(-2 * S)  # = 1 - exp(- 2S)
                denom=1.0  # Approximation for large positive s
            else:  # For very small s, use expm1 for neutral approximation
                denom =-np.expm1(-4 * NeS)  # = 1 - exp(-4Ne s)
                num = -np.expm1(-2 * S)  # = 1 - exp(-4s)
            
            if S<0:
                num=-num
                denom=-denom
            log_selection = np.log(num) - np.log(denom)
        elif self.kimura and not self.exact:
            # Approximate fixation probability for small s: P_fix ≈ 2s ≈ 2*(-delta)
            #Only accept mutations with negative delta = positive s
            #factor 2 is not important since it will be absorbed in the normalization constant
            
            if delta >= 0:
                log_selection = -np.inf  
            else:
                log_selection = np.log(-delta)  
        else:
            log_selection = 0 #just keep any mutations, no Kimura term
            
        # ---------------------------
        # Gaussian term - mutation distribution
        # ---------------------------
        log_gaussian = -(a**2 + r**2) / (2 * self.sigma**2)

        # ---------------------------
        # Jacobian for orthogonal subspace (polar coordinates)
        # ---------------------------
        log_jacobian = (self.n - 2) * np.log(r)

        return log_selection + log_gaussian + log_jacobian


    def initialize(self, nsteps: int = 5000, thinning: int = 1,prop_burnin:float=0.2,alpha_a_prop:float=0.6, alpha_r_prop:float=0.6):
        """
        Initialize chain.

        Parameters
        ----------
        nsteps : int
        burnin : int
        thinning : int
        """
        self.nsteps = nsteps
        self.prop_burnin=prop_burnin
        self.burnin = int(self.prop_burnin * self.nsteps)
        self.thinning = thinning
        self.alpha_a_prop = alpha_a_prop
        self.alpha_r_prop = alpha_r_prop

        #INITIALISATION
        
        # Smart initialization ( should work both for forward and backward cases)
        #Carreful r cannot be 0 for follpwing calcutation
        self.a = max(-self.sigma*np.sqrt(self.n)/max(1e-100, self.z_norm), -self.z_norm) # Worts case chose a to go at optimum
        #E(a)=-sqrt(n)*sigma/2z (Allen Orr)
        
        #r2<=|a|*(2*z_norm-|a|) for delta to be negative)
        self.r = min(
            0.9*np.sqrt(np.abs(self.a)* (2 * self.z_norm - np.abs(self.a))),
            self.sigma * np.sqrt(self.n - 1)
        )


    # --------------------------------------------------
    def sample(self, verbose:bool=True):
        """
        Gibbs-like Metropolis-Hastings sampler:
            - update a | r
            - update r | a (via log r)

        This allows independent tuning of proposals and better mixing.
        """
        
        samples = [] #store (a, r) samples without thinning before burnin
        

        a, r = self.a, self.r

        # --- Step sizes (initial)
        step_a = min(self.alpha_a_prop * self.sigma, self.alpha_a_prop * self.z_norm)
        r_typ = np.sqrt(self.n - 1) * self.sigma
        step_logr = np.log(r_typ) * self.alpha_r_prop


        logp_current = self.log_pi(a, r)

        accepted_a = 0
        accepted_r = 0

        for i in range(self.nsteps):

            # ==================================================
            # 1. UPDATE a | r
            # ==================================================
            a_prop = a + step_a * np.random.randn()
            logp_prop = self.log_pi(a_prop, r)

            if np.log(np.random.rand()) < (logp_prop - logp_current):
                a = a_prop
                logp_current = logp_prop
                accepted_a += 1

            # ==================================================
            # 2. UPDATE r | a   (log-scale)
            # ==================================================
            log_r = np.log(r)
            log_r_prop = log_r + step_logr * np.random.randn()
            r_prop = np.exp(log_r_prop)

            logp_prop = self.log_pi(a, r_prop)

            # Jacobian correction (IMPORTANT)
            log_accept = (
                logp_prop - logp_current
                + log_r_prop - log_r
            )

            if np.log(np.random.rand()) < log_accept:
                r = r_prop
                logp_current = logp_prop
                accepted_r += 1

            # ==================================================
            # STORE
            # ==================================================
            #if i >= self.burnin and i % self.thinning == 0: samples.append([a, r])
            
            samples.append(([a, r],i)) #store all samples for diagnostics
            
            if verbose:
            # progress
                if i % max(1, self.nsteps // 1000) == 0:
                    sys.stdout.write(f"\rProgress: {100 * i / self.nsteps:.1f}%")
                    sys.stdout.flush()

        if verbose:
            print("\nSampling complete.")
            

        posterior_samples = [vec for vec, idx in samples if idx >= self.burnin and (idx - self.burnin) % self.thinning == 0]
        all_samples = [vec for vec, idx in samples]
        self.all_samples_ar = np.array(all_samples)
        self.posterior_ar = np.array(posterior_samples)

        # separate acceptance rates
        self.acceptance_rate_a = accepted_a / self.nsteps
        self.acceptance_rate_r = accepted_r / self.nsteps

    # --------------------------------------------------

    def run_with_diagnostics(self, max_retries=4,verbose=True):
        """
        Run MH with automatic diagnostics and relaunch if needed.

        Criteria for relaunch:
        - Acceptance rate for a or r < 0.075 or > 0.45

        Parameters
        ----------
        max_retries : int
            Maximum number of retries if diagnostics fail.
        verbose : bool
            Whether to print progress information.


        """
        original_nsteps=self.nsteps
        original_burnin=self.prop_burnin
        if verbose:
            print(f"initial nsteps: {self.nsteps}")
        
        self.nsteps = int(0.05 * original_nsteps)
        self.prop_burnin = 0.8
        self.burnin = int(self.prop_burnin * self.nsteps)
    
        for attempt in range(1, max_retries+1):
            if verbose:
                print(f"\n--- Sampling attempt {attempt} ---")
            if verbose:
                print(f"Adjusting parameters for retry: nsteps={self.nsteps}")
            self.sample(verbose=verbose)

            a_chain = self.posterior_ar[:,0]
            r_chain = self.posterior_ar[:,1]
            
            
            ess_a = self.effective_sample_size(a_chain)
            ess_r = self.effective_sample_size(r_chain)

            # --- sécurisation ESS (a) ---
            if not np.isfinite(ess_a) or ess_a <= 1e-12:
                ess_a = 1.0
            ess_a = min(ess_a, self.nsteps)  # upper bound raisonnable

            # --- sécurisation ESS (r) ---
            if not np.isfinite(ess_r) or ess_r <= 1e-12:
                ess_r = 1.0
            ess_r = min(ess_r, self.nsteps)

            # --- calcul tau sécurisé ---
            tau_a = self.nsteps / ess_a
            tau_r = self.nsteps / ess_r

            # optionnel : borne finale (évite valeurs absurdes)
            tau_a = max(1.0, min(tau_a, self.nsteps))
            tau_r = max(1.0, min(tau_r, self.nsteps))
            

            if verbose:
                print(f"Acceptance a = {self.acceptance_rate_a:.3f}")
                print(f"Acceptance r = {self.acceptance_rate_r:.3f}")                
                print(f"Autocorrelation times: tau_a = {tau_a:.1f}, tau_r = {tau_r:.1f} for a thinning of {self.thinning}")
            ratio = max(tau_a, tau_r) / self.thinning
            #tau_refusal=((ratio < 0.5) or (ratio > 2)) and (self.thinning>100)
            
            # Check criteria
            if (self.acceptance_rate_a < 0.075) or (self.acceptance_rate_r < 0.075) or (self.acceptance_rate_a >0.45) or (self.acceptance_rate_r > 0.45):
                if verbose:
                    print("Warning: diagnostics out of range. Adjusting parameters and relaunching.")
                
                                # --- adapt a
                if self.acceptance_rate_a < 0.075:
                    self.alpha_a_prop *= 0.7
                    if verbose:
                        print(f"Reducing proposal step for a: alpha_a_prop={self.alpha_a_prop:.2f}")
                elif self.acceptance_rate_a > 0.45:
                    self.alpha_a_prop *= 1.3
                    if verbose:
                        print(f"Increasing proposal step for a: alpha_a_prop={self.alpha_a_prop:.2f}")

                # --- adapt r
                if self.acceptance_rate_r < 0.075:
                    self.alpha_r_prop *= 0.7
                    if verbose:
                        print(f"Reducing proposal step for r: alpha_r_prop={self.alpha_r_prop:.2f}")
                elif self.acceptance_rate_r > 0.45:
                    self.alpha_r_prop *= 1.3
                    if verbose:
                        print(f"Increasing proposal step for r: alpha_r_prop={self.alpha_r_prop:.2f}")


            else:
                if verbose:
                    print("Diagnostics OK. Chain accepted.")
                self.nsteps = original_nsteps  # reset nsteps for future runs
                self.prop_burnin=original_burnin
                self.burnin = int(self.prop_burnin * self.nsteps)
                self.thinning = int(max(1,min(100, max(tau_a, tau_r))))

                if verbose:
                    print(f"Adjusted thinning = {self.thinning}")
                
                if verbose: 
                    print(f"Final proposal parameters: alpha_a_prop={self.alpha_a_prop:.2f}, alpha_r_prop={self.alpha_r_prop:.2f},nsteps={self.nsteps}, burnin={self.burnin}, thinning={self.thinning}")
                self.sample(verbose=verbose)
                break
        else:
            if verbose:
                print("Warning: maximum retries reached. Chain may be suboptimal.")
            self.nsteps = original_nsteps  # reset nsteps for future runs
            self.prop_burnin=original_burnin
            self.burnin = int(self.prop_burnin * self.nsteps)   
            self.thinning = int(max(1,min(100, max(tau_a, tau_r))))
            if verbose:
                print(f"Adjusted thinning = {self.thinning}")
                     
            if verbose: 
                print(f"Final proposal parameters: alpha_a_prop={self.alpha_a_prop:.2f}, alpha_r_prop={self.alpha_r_prop:.2f},nsteps={self.nsteps}, burnin={self.burnin}, thinning={self.thinning}")
            self.sample(verbose=verbose)

        
    def sample_x(self):
        """
        Reconstruct a single mutation vector x in R^n from the last (a, r) sample.
        """
        a = self.posterior_ar[-1, 0]
        r = self.posterior_ar[-1, 1]

        # sample orthogonal direction
        v = np.random.randn(self.n)
        v -= v.dot(self.u) * self.u
        v /= np.linalg.norm(v)

        # final mutation vector
        x = a * self.u + r * v

        # store without overwriting method
        self.last_sample_x = x
        return x
    
    def sample_multiple_x(self):
        """
        Reconstruct multiple mutation vectors x in R^n from (a, r) sample.
        Vectors are supposed to be independent due to thinning and burnin, so we can sample multiple x's from the posterior (a, r) samples.
        """
        a = self.posterior_ar[:, 0]
        r = self.posterior_ar[:, 1]
        size=len(a)
        sample=[]
        for i in range(size):
            a=self.posterior_ar[i, 0]
            r=self.posterior_ar[i, 1]
            # sample orthogonal direction
            v = np.random.randn(self.n)
            v -= v.dot(self.u) * self.u
            v /= np.linalg.norm(v)
        # final mutation vector
            x = a * self.u + r * v
            sample.append(x)
        self.x_mut=np.array(sample)
    # --------------------------------------------------

        
    def plot_chains(self):
        """Plot chains of a and r.
        Color red for burnin, blue for post-burnin and green for thinned samples.
        Useful for diagnostics and visualizing convergence."""

        a_chain = self.all_samples_ar[:, 0]
        r_chain = self.all_samples_ar[:, 1]
        n = len(a_chain)
        # Burn-in
        burnin = getattr(self, "burnin", None)
        if burnin is None:
            burnin = int(getattr(self, "prop_burnin", 0) * n)

        thinning = getattr(self, "thinning", 1)

        fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

        # a
        axs[0].plot(a_chain[:burnin], color="red", label="burn-in")
        axs[0].plot(range(burnin, n), a_chain[burnin:], color="blue", label="post burn-in")
        axs[0].plot(range(burnin, n, thinning), a_chain[burnin:n:thinning],
                    color="green", label="thinned")
        axs[0].set_ylabel("a")
        axs[0].legend()

        # r
        axs[1].plot(r_chain[:burnin], color="red", label="burn-in")
        axs[1].plot(range(burnin, n), r_chain[burnin:], color="blue", label="post burn-in")
        axs[1].plot(range(burnin, n, thinning), r_chain[burnin:n:thinning],
                    color="green", label="thinned")
        axs[1].set_ylabel("r")
        axs[1].set_xlabel("Iteration")
        axs[1].legend()

        plt.tight_layout()
        plt.show()
            

        
    def autocorr(self, x, max_lag=200):
        """
        Compute the autocorrelation function of a 1D array x up to max_lag.
        Returns an array of autocorrelations for lags 0 to max_lag-1 
        Parameters
        ----------
        x : array-like
            Input 1D array.
        max_lag : int
            Maximum lag to compute autocorrelation for.
        Returns
        -------
        acf : np.ndarray
            Autocorrelation function values for lags 0 to max_lag-1.
        """


        x = np.asarray(x)
        x = x - x.mean()
        n = len(x)

        var = np.var(x)
        acf = []

        for lag in range(max_lag):
            if lag == 0:
                acf.append(1.0)
            else:
                if(n!=lag):
                    c=np.dot(x[:-lag], x[lag:]) / (n - lag)
                else:
                    c=np.dot(x[:-lag], x[lag:]) / (n+1 - lag)
                acf.append(c / var)

        return np.array(acf)
    
    def effective_sample_size(self, x):
        """
        Compute the effective sample size (ESS) of a 1D array x using the autocorrelation function.
        """
        acf = self.autocorr(x, max_lag=500)

        # Geyer truncation (important)
        s = 0.0
        for k in range(1, len(acf)-1, 2):
            pair_sum = acf[k] + acf[k+1]
            if pair_sum < 0:
                break
            s += pair_sum

        tau = 1 + 2 * s
        return len(x) / tau
    
    def plot_delta(self):
        """
        Plot the distribution of delta=-2*(mut,z)-||mut||^2  and selection coefficient S for the posterior samples.
        """


        if self.forward:
            delta = self.posterior_ar[:,0]**2 + self.posterior_ar[:,1]**2 \
                    + 2*self.posterior_ar[:,0]*self.z_norm
            S=np.expm1(-delta / 2) # = exp(-delta/2) - 1
        else:
            delta = self.posterior_ar[:,0]**2 + self.posterior_ar[:,1]**2 - 2*self.posterior_ar[:,0]*self.z_norm
            S=np.expm1(delta / 2) # = exp(-delta/2) - 1
            
        fig, axs = plt.subplots(2, 1, figsize=(10, 6))

        axs[0].hist(delta, bins=50)
        axs[0].set_title("Distribution of delta")
        axs[1].hist(S, bins=50)
        axs[1].set_title("Distribution of S")
        plt.show()
    
    def diagnostics(self):
        """ 
        Print acceptance rates and effective sample sizes for a and r.
        Useful for checking if the sampler is mixing well and if the thinning is appropriate.
        Useful for debugging and tuning the sampler.
        """

        a = self.posterior_ar[:, 0]
        r = self.posterior_ar[:, 1]
        ESS_a = self.effective_sample_size(a)
        ESS_r = self.effective_sample_size(r)
        
        print("Acceptance rate a:", self.acceptance_rate_a)
        print("Acceptance rate r:", self.acceptance_rate_r)
        print("ESS a:", ESS_a)
        print("ESS r:", ESS_r)
        print("taux a: ",self.nsteps / ESS_a)
        print("taux r: ",self.nsteps / ESS_r)

        
    def plot_diagnostics(self):
        """
        Plot traces and histograms for a and log(r).
        Useful for visualizing the posterior distribution and checking for convergence.
        """
        a = self.posterior_ar[:, 0]
        r = np.log(self.posterior_ar[:, 1])

        fig, axs = plt.subplots(2, 2, figsize=(10, 6))

        axs[0,0].plot(a)
        axs[0,0].set_title("Trace a")

        axs[1,0].plot(r)
        axs[1,0].set_title("Trace log(r)")

        axs[0,1].hist(a, bins=50)
        axs[0,1].set_title("Histogram a")

        axs[1,1].hist(r, bins=50)
        axs[1,1].set_title("Histogram log(r)")

        plt.tight_layout()
        plt.show()

### II.3.1 Testing of MH sampler

n=3\
nsteps=1 000 000 \
Kimura correction

In [ ]:
#Jeux de test Backward
z=np.zeros(10) # Example phenotype in 32
z[0] = 1
MH=MetropolisHastingsAR()
MH.set_problem(z, sigma=1,forward=True,only_beneficial=True,kimura=True,Ne=1e5,exact=False)
MH.initialize(nsteps=1000000, thinning=20,prop_burnin=0.3,alpha_r_prop=1, alpha_a_prop=1)
MH.run_with_diagnostics(max_retries=5,verbose=True)
MH.plot_diagnostics()
MH.plot_chains()
MH.plot_delta()


n=10\
nsteps=1 000 000 \
No Kimura correction

In [ ]:
#Jeux de test Backward
z=np.zeros(3) # Example phenotype in 32
z[0] = 1
MH=MetropolisHastingsAR()
MH.set_problem(z, sigma=0.05,forward=True,kimura=False,only_beneficial=True,exact=True)
MH.initialize(nsteps=1000000, thinning=20,prop_burnin=0.3,alpha_r_prop=1, alpha_a_prop=1)
MH.run_with_diagnostics(max_retries=5,verbose=True)
MH.plot_diagnostics()
MH.plot_chains()
MH.plot_delta()


## II.4 Comparison of 3 samplers 

In [ ]:
"""
Figure:  Validation of the three samplers of fixed substitutions.
 
Problem
-------
Starting from a phenotype z_0 at distance d = ||z_0|| from the optimum, we need
independent draws of the next substitution delta under two target laws:
 
    (no fixation weighting)  q_B(delta)  ∝ q(delta) · 1{Delta_delta(z_0) > 0}
    (fixation weighting)     q_F(delta)  ∝ q(delta) · 1{Delta_delta(z_0) > 0} · pi(s)
 
with q = N(0, sigma^2 I_n) the isotropic Gaussian proposal,
Delta_delta(z) = -<delta, z> - ||delta||^2 / 2 the log-fitness effect,
s = exp(Delta) - 1 the selection coefficient, and
pi(s) = (1 - e^{-2s}) / (1 - e^{-4 Ne s}) the Kimura fixation probability

 
Three samplers are compared:
    1. Rejection sampling   — exact by construction; used as the reference.
                              Draw delta ~ q, keep it if beneficial, then (with
                              fixation weighting) keep it with probability pi(s)
                              (`draw_mutation_by_rejection`).
    2. Rosenblatt sampler   — inverse-CDF sampling of (R, cos Theta), the n-2
                              transverse directions being integrated analytically
                              (`sample_fixed_mutations`). O(1) cost in n.
    3. MCMC (Metropolis–Hastings) on the reduced coordinates (a, r), with a the
                              projection on z_0/||z_0|| and r the transverse norm
                              (`MetropolisHastingsAR`).
 
Figure layout (2 rows x 4 columns, full page width)
---------------------------------------------------
    Row 1 (A–D): beneficial mutations, no fixation weighting  (pi = 1)
    Row 2 (E–H): fixed substitutions, Kimura fixation weighting
    Columns    : <delta, z_0>,  Theta / pi,  s,  R = ||delta||
 
The reference (rejection) is drawn as a filled grey histogram; the Rosenblatt
and MCMC samples are drawn as outlined step histograms on the same bins, so
that any discrepancy appears as a line departing from the grey area.
Two-sample Kolmogorov–Smirnov tests (Rosenblatt vs rejection, MCMC vs
rejection) are printed and written to a CSV file for the figure legend.
Rejection and Rosenblatt draws are i.i.d., so the exact KS p-value applies.
MCMC draws are thinned but remain weakly autocorrelated, which would make the
naive p-value anti-conservative; for MCMC the p-value is therefore computed
from the asymptotic Kolmogorov distribution with the effective sample size
(ESS) of the chain in place of the nominal number of draws.
 
Dependencies defined in earlier cells
-------------------------------------
    draw_mutation_by_rejection(z, sigma_m, Ne, max_attempts, kimura,
                               only_beneficial, exact)
                                                — rejection sampler (reference)
    sample_fixed_mutations(z, sigma, N, rng, Ne, probability_of_fixation)
                                                — Rosenblatt sampler
    MetropolisHastingsAR                        — MCMC sampler
 
Samples are cached to disk (CACHE_FILE) so the figure can be restyled without
rerunning the Markov chains. Delete the cache to force recomputation.
"""
 
from __future__ import annotations
 
import csv
import os
from dataclasses import dataclass
from typing import Dict, List, Tuple
 
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from scipy.stats import ks_2samp, kstwobign
 
FloatArray = NDArray[np.float64]
 
# =============================================================================
# 1. Parameters
# =============================================================================
N_DIM: int = 16                  # phenotypic complexity n
D0: float = 1.0                  # distance to the optimum d = ||z_0||
SIGMA: float = 0.025              # per-trait mutational SD; sigma_tilde = sigma*sqrt(n) = 0.1
NE: float = 1e5                  # effective population size 
N_SAMPLES: int = 20_000          # draws per sampler and per scenario (rejection, Rosenblatt)
MH_NSTEPS: int = 2_000_000       # MCMC chain length (before burn-in and thinning)
MH_BURNIN: float = 0.3           # fraction of the chain discarded as burn-in
N_BINS: int = 30                 # histogram bins per panel
SEED: int = 2026                 # master seed (all three samplers are reproducible)
CACHE_FILE: str = "figS2_sampler_validation_samples.npz"
OUT_STEM: str = "figS2_sampler_validation"
 
# Derived quantities, reported in the legend
SIGMA_TILDE: float = SIGMA * np.sqrt(N_DIM)               # sigma_tilde = sqrt(E[R^2])
X_FISHER: float = np.sqrt(N_DIM) * SIGMA_TILDE / (2 * D0)  # Fisher parameter x
 
 
@dataclass(frozen=True)
class Scenario:
    """One target distribution (one row of the figure)."""
    key: str             # short identifier used in the cache and CSV
    title: str           # row title shown on the figure
    with_fixation: bool  # True -> weight beneficial mutations by pi(s)
 
 
SCENARIOS: Tuple[Scenario, ...] = (
    Scenario("nofix", "Beneficial mutations (no fixation weighting)", False),
    Scenario("fix", "Fixed substitutions (with fixation probability)", True),
)
 
 
@dataclass(frozen=True)
class Sampler:
    """Visual encoding of one sampler."""
    key: str
    label: str
    color: str
    linestyle: str
 
 
REFERENCE = Sampler("rejection", "Rejection sampling (reference)", "#bdbdbd", "-")
TESTED: Tuple[Sampler, ...] = (
    Sampler("rosenblatt", "Rosenblatt (inverse CDF)", "#0072b5", "-"),
    Sampler("mcmc", "MCMC (Metropolis–Hastings)", "#bc3c29", "--"),
)
 
 
@dataclass(frozen=True)
class Statistic:
    """One summary statistic of a mutation (one column of the figure)."""
    key: str
    xlabel: str
 
 
STATISTICS: Tuple[Statistic, ...] = (
    Statistic("projection", r"$\langle \delta,\, \mathbf{z}_0 \rangle$"),
    Statistic("theta", r"$\Theta\,/\,\pi$"),
    Statistic("s", r"Selection coefficient $s$"),
    Statistic("norm", r"$R = \|\delta\|$"),
)
 
 
# =============================================================================
# 2. Samplers
# =============================================================================
def sample_rejection(
    z: FloatArray,
    sigma: float,
    n_samples: int,
    ne: float,
    with_fixation: bool,
    seed: int,
    max_attempts: int = 100_000,
) -> Tuple[FloatArray, float]:
    """Rejection sampler of beneficial (optionally fixed) mutations.
 
    Calls `draw_mutation_by_rejection` — the very function used in the
    simulations — `n_samples` times, so that it is the code being validated.
    Each call proposes delta ~ N(0, sigma^2 I_n) until one is accepted:
    beneficial only (with_fixation=False), or beneficial and fixed with the
    exact Kimura probability pi(s) (with_fixation=True). Accepted draws are
    therefore i.i.d. from the target law.
 
    The function draws from the global NumPy RNG, seeded here for reproducibility.
 
    Parameters
    ----------
    z : array, shape (n,)
        Current phenotype z_0 (optimum at the origin).
    sigma : float
        Per-trait standard deviation of the Gaussian mutational proposal.
    n_samples : int
        Number of accepted mutations to return.
    ne : float
        Effective population size, used only when `with_fixation` is True.
    with_fixation : bool
        False -> accept every beneficial mutation (target q_B).
        True  -> accept a beneficial mutation with probability pi(s) (target q_F).
    seed : int
        Seed of the global NumPy RNG.
    max_attempts : int
        Maximum number of proposals per accepted draw before failing.
 
    Returns
    -------
    delta : array, shape (n_samples, n)
        Accepted mutation vectors.
    acceptance_rate : float
        Fraction of proposals accepted (cost of the reference sampler).
    """
    np.random.seed(seed)
    z = np.asarray(z, dtype=float)
    delta = np.empty((n_samples, z.size))
    n_proposed = 0
    for k in range(n_samples):
        delta[k], n_attempts = draw_mutation_by_rejection(
            z, sigma, Ne=ne, max_attempts=max_attempts,
            kimura=with_fixation, only_beneficial=True, exact=True,
        )
        n_proposed += n_attempts
    return delta, n_samples / n_proposed
 
 
def sample_rosenblatt(
    z: FloatArray,
    sigma: float,
    n_samples: int,
    rng: np.random.Generator,
    ne: float,
    with_fixation: bool,
) -> FloatArray:
    """Rosenblatt (inverse-CDF) sampler; thin wrapper around `sample_fixed_mutations`."""
    return sample_fixed_mutations(
        z, sigma, n_samples, rng, Ne=ne,
        probability_of_fixation="exact" if with_fixation else "1",
    )
 
 
def sample_mcmc(
    z: FloatArray,
    sigma: float,
    ne: float,
    with_fixation: bool,
    n_steps: int,
    burnin: float,
    seed: int,
) -> Tuple[FloatArray, Dict[str, float]]:
    """Metropolis–Hastings sampler on (a, r); thin wrapper around `MetropolisHastingsAR`.
 
    The class draws from the global NumPy RNG, which is therefore seeded here
    for reproducibility. The proposal scales are tuned automatically by
    `run_with_diagnostics`, which also sets the thinning to the estimated
    integrated autocorrelation time (capped at 100).
 
    Returns
    -------
    delta : array, shape (n_kept, n)
        Mutation vectors reconstructed from the retained (a, r) states, with a
        uniformly random transverse direction.
    diagnostics : dict
        Acceptance rates of the two Gibbs-like updates and the final thinning.
    """
    np.random.seed(seed)
    mh = MetropolisHastingsAR()
    mh.set_problem(z, sigma=sigma, Ne=ne, kimura=with_fixation,
                   only_beneficial=True, exact=True)
    mh.initialize(nsteps=n_steps, thinning=20, prop_burnin=burnin,
                  alpha_a_prop=1.0, alpha_r_prop=1.0)
    mh.run_with_diagnostics(max_retries=5, verbose=False)
    mh.sample_multiple_x()
    diagnostics = {
        "acceptance_a": float(mh.acceptance_rate_a),
        "acceptance_r": float(mh.acceptance_rate_r),
        "thinning": float(mh.thinning),
        "n_kept": float(mh.x_mut.shape[0]),
    }
    return np.asarray(mh.x_mut, dtype=float), diagnostics
 
 
# =============================================================================
# 3. Summary statistics of a set of mutations
# =============================================================================
def summary_statistics(delta: FloatArray, z: FloatArray) -> Dict[str, FloatArray]:
    """Compute the four statistics displayed in the figure.
 
    Parameters
    ----------
    delta : array, shape (N, n)
        Mutation vectors.
    z : array, shape (n,)
        Background phenotype z_0.
 
    Returns
    -------
    dict with keys
        "projection" : <delta, z_0>         (negative for beneficial mutations)
        "theta"      : Theta / pi, Theta the angle between delta and z_0 (in [1/2, 1])
        "s"          : s = exp(Delta) - 1, Delta = -<delta, z_0> - ||delta||^2 / 2
        "norm"       : R = ||delta||
    """
    projection = delta @ z
    norm = np.linalg.norm(delta, axis=1)
    cos_theta = np.clip(projection / (norm * np.linalg.norm(z)), -1.0, 1.0)
    log_gain = -projection - 0.5 * norm**2
    return {
        "projection": projection,
        "theta": np.arccos(cos_theta) / np.pi,
        "s": np.expm1(log_gain),
        "norm": norm,
    }
 
 
# =============================================================================
# 4. Compute (or load cached samples)
# =============================================================================
def compute_samples(cache: str = CACHE_FILE) -> Dict[str, Dict[str, FloatArray]]:
    """Run the three samplers for both scenarios; return samples[scenario][sampler].
 
    The cache is reused only if the stored parameters match the current ones.
    """
    params = np.array([N_DIM, D0, SIGMA, NE, N_SAMPLES, MH_NSTEPS, MH_BURNIN, SEED])
    if cache and os.path.exists(cache):
        with np.load(cache) as f:
            if np.array_equal(f["params"], params):
                print(f"Loaded cached samples from {cache}")
                return {sc.key: {k: f[f"{sc.key}_{k}"]
                                 for k in (REFERENCE.key, *(t.key for t in TESTED))}
                        for sc in SCENARIOS}
 
    z0 = np.zeros(N_DIM)
    z0[0] = D0
    samples: Dict[str, Dict[str, FloatArray]] = {}
    to_save: Dict[str, FloatArray] = {"params": params}
 
    for i, sc in enumerate(SCENARIOS):
        rng = np.random.default_rng([SEED, i])
        print(f"\n[{sc.key}] {sc.title}")
 
        delta_rej, acc = sample_rejection(z0, SIGMA, N_SAMPLES, NE, sc.with_fixation,
                                          seed=SEED + 100 + i)
        print(f"  rejection   : {len(delta_rej)} draws, acceptance rate = {acc:.3e}")
 
        delta_ros = sample_rosenblatt(z0, SIGMA, N_SAMPLES, rng, NE, sc.with_fixation)
        print(f"  Rosenblatt  : {len(delta_ros)} draws")
 
        delta_mh, diag = sample_mcmc(z0, SIGMA, NE, sc.with_fixation,
                                     MH_NSTEPS, MH_BURNIN, seed=SEED + i)
        print(f"  MCMC        : {int(diag['n_kept'])} draws "
              f"(acceptance a = {diag['acceptance_a']:.2f}, "
              f"r = {diag['acceptance_r']:.2f}, thinning = {int(diag['thinning'])})")
 
        samples[sc.key] = {REFERENCE.key: delta_rej, "rosenblatt": delta_ros, "mcmc": delta_mh}
        for k, v in samples[sc.key].items():
            to_save[f"{sc.key}_{k}"] = v
 
    if cache:
        np.savez_compressed(cache, **to_save)
    return samples
 
 
def effective_sample_size(chain: FloatArray) -> float:
    """ESS of a 1-D chain, with the estimator used to tune the MCMC sampler.
 
    Geyer initial-positive-sequence truncation of the autocorrelation function
    (`MetropolisHastingsAR.effective_sample_size`), bounded to [1, len(chain)].
    """
    ess = MetropolisHastingsAR().effective_sample_size(np.asarray(chain, dtype=float))
    return float(np.clip(ess, 1.0, chain.size)) if np.isfinite(ess) else 1.0
 
 
def ks_test(sample: FloatArray, reference: FloatArray,
            n_eff: float | None = None) -> Tuple[float, float]:
    """Two-sample KS distance D and p-value.
 
    If `n_eff` is None, both samples are i.i.d. and the exact p-value of
    `scipy.stats.ks_2samp` is returned. Otherwise `sample` is autocorrelated
    and the p-value uses the asymptotic Kolmogorov law with the effective
    sample size: P(K > D * sqrt(n_eff * m / (n_eff + m))), m = len(reference).
    """
    result = ks_2samp(sample, reference)
    if n_eff is None:
        return float(result.statistic), float(result.pvalue)
    m = reference.size
    lam = result.statistic * np.sqrt(n_eff * m / (n_eff + m))
    return float(result.statistic), float(kstwobign.sf(lam))
 
 
def ks_table(stats: Dict[str, Dict[str, Dict[str, FloatArray]]],
             outfile: str = f"{OUT_STEM}_KS.csv") -> None:
    """Print and save KS tests of each tested sampler against the reference."""
    rows: List[List[str]] = [["scenario", "statistic", "sampler",
                              "n_draws", "n_effective", "KS_D", "p_value"]]
    print("\nTwo-sample KS test against the rejection reference")
    print(f"  {'scenario':8s} {'statistic':11s} {'sampler':11s} "
          f"{'n':>6s} {'n_eff':>7s} {'D':>7s} {'p':>7s}")
    for sc in SCENARIOS:
        for st in STATISTICS:
            ref = stats[sc.key][REFERENCE.key][st.key]
            for smp in TESTED:
                x = stats[sc.key][smp.key][st.key]
                n_eff = effective_sample_size(x) if smp.key == "mcmc" else None
                D, p = ks_test(x, ref, n_eff)
                n_eff_out = n_eff if n_eff is not None else float(x.size)
                rows.append([sc.key, st.key, smp.key, str(x.size),
                             f"{n_eff_out:.0f}", f"{D:.4f}", f"{p:.3g}"])
                print(f"  {sc.key:8s} {st.key:11s} {smp.key:11s} "
                      f"{x.size:6d} {n_eff_out:7.0f} {D:7.4f} {p:7.3g}")
    with open(outfile, "w", newline="") as fh:
        csv.writer(fh).writerows(rows)
 
 
# =============================================================================
# 5. Plot
# =============================================================================
def _density_histogram(values: FloatArray, edges: FloatArray) -> FloatArray:
    """Histogram normalised by the TOTAL sample size (not only in-range draws).
 
    Values outside [edges[0], edges[-1]] are simply not shown, instead of being
    piled into the edge bins (no clipping artefact).
    """
    counts, _ = np.histogram(values, bins=edges)
    return counts / (values.size * np.diff(edges))
 
 
def _bin_edges(pooled: FloatArray, key: str, n_bins: int) -> FloatArray:
    """Common bins for the three samplers, set from pooled quantiles."""
    lo, hi = np.quantile(pooled, [0.002, 0.998])
    if key == "theta":                       # beneficial mutations: Theta > pi/2
        lo = max(lo, 0.5)
    if key in ("s", "norm"):                 # non-negative quantities
        lo = max(lo, 0.0)
    return np.linspace(lo, hi, n_bins + 1)
 
 
def plot_figure(stats: Dict[str, Dict[str, Dict[str, FloatArray]]],
                outfile: str = OUT_STEM) -> None:
    """Draw the 2 x 4 validation figure and save it as PDF and TIFF."""
    mpl.rcParams.update({
        "font.family": "serif",
        "mathtext.fontset": "cm",
        "font.size": 8,
        "axes.labelsize": 8.5,
        "axes.titlesize": 8.5,
        "axes.linewidth": 0.7,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.minor.visible": True,
        "ytick.minor.visible": True,
        "xtick.minor.size": 1.5,
        "ytick.minor.size": 1.5,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.frameon": False,
        "legend.fontsize": 8,
    })
 
    one_mm = 1 / 25.4
    fig = plt.figure(figsize=(178 * one_mm, 112 * one_mm), layout="constrained")
    subfigs = fig.subfigures(len(SCENARIOS), 1, hspace=0.04)
    panel_letters = iter("ABCDEFGH")
 
    for subfig, sc in zip(subfigs, SCENARIOS):
        subfig.suptitle(sc.title, x=0.01, ha="left", fontsize=9, fontweight="bold")
        axes = subfig.subplots(1, len(STATISTICS))
 
        for j, (ax, st) in enumerate(zip(axes, STATISTICS)):
            data = stats[sc.key]
            pooled = np.concatenate([data[k][st.key] for k in
                                     (REFERENCE.key, *(t.key for t in TESTED))])
            edges = _bin_edges(pooled, st.key, N_BINS)
 
            # Reference: filled grey histogram
            ax.stairs(_density_histogram(data[REFERENCE.key][st.key], edges), edges,
                      fill=True, color=REFERENCE.color, lw=0, zorder=1)
            # Tested samplers: outlined step histograms on the same bins
            for smp in TESTED:
                ax.stairs(_density_histogram(data[smp.key][st.key], edges), edges,
                          color=smp.color, ls=smp.linestyle, lw=1.1, zorder=3)
 
            ax.set_xlim(edges[0], edges[-1])
            ax.set_ylim(bottom=0)
            ax.set_xlabel(st.xlabel)
            if j == 0:
                ax.set_ylabel("Density")
            ax.text(-0.04, 1.03, next(panel_letters), transform=ax.transAxes,
                    fontsize=10, fontweight="bold", ha="right", va="bottom")
 
    # Single legend shared by all panels
    handles = [mpl.patches.Patch(facecolor=REFERENCE.color, label=REFERENCE.label)]
    handles += [mpl.lines.Line2D([0], [0], color=s.color, ls=s.linestyle, lw=1.1,
                                 label=s.label) for s in TESTED]
    fig.legend(handles=handles, loc="outside lower center", ncol=3)
 
    fig.savefig(f"{outfile}.pdf", bbox_inches="tight")
    fig.savefig(f"{outfile}.tiff", dpi=1200, bbox_inches="tight",
                pil_kwargs={"compression": "tiff_lzw"})
    fig.savefig(f"{outfile}.png", dpi=300, bbox_inches="tight")   # quick preview
    plt.close(fig)
    print(f"\nSaved: {outfile}.pdf / .tiff / .png")
 
 
# =============================================================================
# 6. Driver
# =============================================================================
if __name__ == "__main__" or "get_ipython" in globals():
    print(f"n = {N_DIM}, d = {D0}, sigma = {SIGMA} (sigma_tilde = {SIGMA_TILDE:.3f}), "
          f"x = {X_FISHER:.3f}, Ne = {NE:.0e}")
    z_ref = np.zeros(N_DIM)
    z_ref[0] = D0
    samples = compute_samples()
    stats = {sc.key: {k: summary_statistics(v, z_ref) for k, v in samples[sc.key].items()}
             for sc in SCENARIOS}
    ks_table(stats)
    plot_figure(stats)
 

# III  Entrenchment probability $\mathcal{E}_{i,\infty}$

## III.1 Probability of lying in IS depending on x

In [ ]:
"""
Figure S2 — Probability of entrenchment as a function of x = n*sigma/(2d).

Theory:  P_entrenched(x) = [Phi(2x) - Phi(x)] / [1 - Phi(x)]  (black curve)

Log-log axes. Legends placed outside the axes (right side), not clipped.
Points with identical x values are jittered horizontally to avoid overlap.
y-axis lower limit set from the minimum simulated value.
"""

from scipy.stats import norm

# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":         "serif",
    "mathtext.fontset":    "dejavuserif",
    "font.size":           11,
    "axes.labelsize":      12,
    "axes.linewidth":      0.8,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.major.width":   0.8,
    "ytick.major.width":   0.8,
    "legend.frameon":      False,
    "legend.fontsize":     9,
})

# ── Palette ───────────────────────────────────────────────────────────────────
COLOURS = {
    0.005: "#0072b5",
    0.05:  "#bc3c29",
    0.5:   "#20854e",
}
MARKERS = {5: "o", 10: "s", 100: "^"}
MS = 5
CAPSIZE = 3

# ── Parameters ────────────────────────────────────────────────────────────────
N_DIMS        = [5, 10, 100]
SIGMA_OVER_2D = [0.005, 0.05, 0.5] #sti/2d = sigma*sqrt(n)/(2d)
D             = 1
NE            = 1e5
X_STAR        = 0.612003180962481
X_MAX         = 6
X_MIN         = 0.005
N_DRAWS       =30_000
RNG           = np.random.default_rng(4)


JITTER_LOG = {5: -0.04, 10: 0.0, 100: +0.04}

# --------------------------------------------------------------------------
# Functions of x
# --------------------------------------------------------------------------
def phi(x):
    """Standard normal density."""
    return norm.pdf(x)

def phibar(x):
    """Upper-tail probability 1 - Phi(x)."""
    return norm.sf(x)

def lambda_mills(x):
    """Inverse Mills ratio Lambda(x) = phi(x) / Phibar(x)."""
    return phi(x) / phibar(x)

def M_fixed(x):
    """M(x) = 1 / (Lambda(x) - x), equal to Phibar(x) / D(x) with
    D(x) = phi(x) - x*Phibar(x)."""
    return 1.0 / (lambda_mills(x) - x)

# --------------------------------------------------------------------------
# probability of fixation for an array of beneficial mutations 
# --------------------------------------------------------------------------

def proba_fixation(delta:np.ndarray)->np.ndarray:
    """Compute the probability of fixation for an array of beneficial mutations.
    delta: array of shape (n_beneficial, n) representing beneficial mutations.
    Delta_{delta}(z)=-<mut, z> - ||mut||^2/2 < 0 for beneficial mutations.
    S=exp(Delta)-1
    pi(S)=1-exp(-2S) ~ 2S for small S
    Returns: array of shape (n_beneficial,) with the probability of fixation for each mutation.
    """
    # Compute the selection coefficient s for each beneficial mutation
    Delta_log_fitness= -np.sum(delta**2, axis=1) / 2 - delta[:, 0] * D  # Delta_{delta}(z) = -<mut, z> - ||mut||^2/2
    S=np.expm1(Delta_log_fitness)  # S = exp(Delta) - 1
    return -np.expm1(-2*S)  # pi(S) = 1 - exp(-2S)


# ── Theory ────────────────────────────────────────────────────────────────────
def p_entrenched_theory_without_fixation(x):
    sf_x  = phibar(x)
    sf_2x = phibar(2 * x)
    return np.where(sf_x > 1e-300, (sf_x - sf_2x) / sf_x, 1.0)

def p_entrenched_theory_with_fixation(x):
    num1 = phibar(x) - phibar(2 * x)
    num2= phi(x)-phi(2*x)
    factor=M_fixed(x)/phibar(x)
    return np.where(phibar(x) > 1e-300, factor*(num2-x*num1) , 1.0)

# ── Wilson CI ─────────────────────────────────────────────────────────────────
def wilson_ci(k, n, z_alpha=1.96):
    p_hat  = k / n
    denom  = 1 + z_alpha**2 / n
    centre = (p_hat + z_alpha**2 / (2 * n)) / denom
    half   = z_alpha * np.sqrt(p_hat * (1 - p_hat) / n +
                              z_alpha**2 / (4 * n**2)) / denom
    lo = np.clip(centre - half, 1e-6, 1 - 1e-6)
    hi = np.clip(centre + half, 1e-6, 1 - 1e-6)
    return np.clip(p_hat, 1e-6, 1 - 1e-6), lo, hi

# ── Simulation ────────────────────────────────────────────────────────────────

#######ROSENBLATT #########

def simulate_p_entrenched_without_fixation_rosenblatt(n, sigma_over_2d, d, n_draws,rng):
    """Simulate the probability of entrenchment."""
    sigma   = sigma_over_2d * 2 * d * (1/np.sqrt(n) )
    x_param = np.sqrt(n)*sigma_over_2d
    z=np.zeros(n)
    z[0]=d
    n_entrenched = 0
    n_fixed = 0
    _,mut=sample_adaptive_walks_cdf(z, sigma, n_steps=1, n_rep=n_draws, rng=RNG,Ne=NE, probability_of_fixation="1")
    d_fix=mut[:,0,:]
    ent = (np.sum(d_fix**2, axis=1) + d_fix[:, 0] * d) > 0
    n_entrenched += int(ent.sum()) # count the entrenched mutations in this batch
    p_hat, lo, hi = wilson_ci(n_entrenched, n_draws)
    return  x_param, p_hat, lo, hi


def simulate_p_entrenched_with_fixation_rosenblatt(n, sigma_over_2d, d, n_draws,rng):
    """Simulate the probability of entrenchment."""
    sigma   = sigma_over_2d * 2 * d * (1/np.sqrt(n) )
    x_param = np.sqrt(n)*sigma_over_2d
    z=np.zeros(n)
    z[0]=d

    n_entrenched = 0
    n_fixed = 0
    _,mut=sample_adaptive_walks_cdf(z, sigma, n_steps=1, n_rep=n_draws, rng=RNG, Ne=NE, probability_of_fixation="exact")
    d_fix=mut[:,0,:]
    ent = (np.sum(d_fix**2, axis=1) + d_fix[:, 0] * d) > 0
    n_entrenched += int(ent.sum()) # count the entrenched mutations in this batch
    p_hat, lo, hi = wilson_ci(n_entrenched, n_draws)
    return  x_param, p_hat, lo, hi




In [ ]:
#WITHOUT FIXATION
# ── Run simulations  ───────────────────────────────────────────────────────
results = {}
for s2d in SIGMA_OVER_2D:
    results[s2d] = {}
    for n in N_DIMS:
        print(f"Simulating P_entrenched for n={n}, sigma/(2d)={s2d}...")
        x, p, lo, hi = simulate_p_entrenched_without_fixation_rosenblatt(n, s2d, D, N_DRAWS, RNG)
        results[s2d][n] = (x, p, lo, hi)

p_all = [results[s2d][n][1] for s2d in SIGMA_OVER_2D for n in N_DIMS
         if results[s2d][n][1] is not None]
y_min = 10 ** (np.floor(np.log10(min(p_all))) - 0.01)

# ── Figure ────────────────────────────────────────────────────────────────────
ONE_MM = 1 / 25.4
fig = plt.figure(figsize=(160 * ONE_MM, 90 * ONE_MM))

# Axes principal
ax = fig.add_axes([0.12, 0.12, 0.58, 0.80])  # position contrôlée

# ── Plot ──────────────────────────────────────────────────────────────────────
x_th = np.linspace(X_MIN, X_MAX, 600)
ax.plot(x_th, p_entrenched_theory_without_fixation(x_th),
        color="black", lw=1.8)

#ax.axvline(X_STAR, color="black", lw=0.7, ls=":")

for s2d in SIGMA_OVER_2D:
    col = COLOURS[s2d]
    for n in N_DIMS:
        x, p, lo, hi = results[s2d][n]
        if p is None:
            continue

        mk = MARKERS[n]
        x_plot = x * 10 ** JITTER_LOG[n]
        yerr = np.array([[p - lo], [hi - p]])

        ax.errorbar(x_plot, p,
                    yerr=yerr,
                    fmt=mk,
                    color=col,
                    markersize=MS,
                    capsize=CAPSIZE,
                    lw=0.9)

# ── Axes config ───────────────────────────────────────────────────────────────
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$x = \sqrt{n}\tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$1-P_{\mathrm{IS}}$")
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(10**(-4), 1.1)

# ── Légende collée EXACTEMENT à droite ───────────────────────────────────────
pos = ax.get_position()

ax_leg = fig.add_axes([
    pos.x1,          # ← collé au bord droit de l'axe
    pos.y0,
    0.25,            # largeur de la colonne légende
    pos.height
])
ax_leg.axis("off")

legend_colour = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=COLOURS[s2d], markersize=MS,
               label=rf"$\tilde{{\sigma}}/2d={s2d}$")
    for s2d in SIGMA_OVER_2D
]

legend_marker = [
    plt.Line2D([0], [0], marker=mk, color="gray",
               markersize=MS, linestyle="none",
               label=rf"$n={n}$")
    for n, mk in MARKERS.items()
]

leg1 = ax_leg.legend(handles=legend_colour,
                     loc="upper left")

leg2 = ax_leg.legend(handles=legend_marker,
                     loc="lower left")

ax_leg.add_artist(leg1)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig("figS2_P_entrenched_nofixation.pdf", bbox_inches="tight")
fig.savefig("figS2_P_entrenched_nofixation.tiff", dpi=1200, bbox_inches="tight")

plt.close(fig)

In [ ]:
#WITH FIXATION

# ── Run simulations  ───────────────────────────────────────────────────────
results = {}
for s2d in SIGMA_OVER_2D:
    results[s2d] = {}
    for n in N_DIMS:
        print(f"Simulating P_entrenched for n={n}, sigma/(2d)={s2d}...")
        x, p, lo, hi = simulate_p_entrenched_with_fixation_rosenblatt(n, s2d, D, N_DRAWS, RNG)
        results[s2d][n] = (x, p, lo, hi)

p_all = [results[s2d][n][1] for s2d in SIGMA_OVER_2D for n in N_DIMS
         if results[s2d][n][1] is not None]
y_min = 10 ** (np.floor(np.log10(min(p_all))) - 0.01)

# ── Figure ────────────────────────────────────────────────────────────────────
ONE_MM = 1 / 25.4
fig = plt.figure(figsize=(160 * ONE_MM, 90 * ONE_MM))

# Axes principal
ax = fig.add_axes([0.12, 0.12, 0.58, 0.80])  # position contrôlée

# ── Plot ──────────────────────────────────────────────────────────────────────
x_th = np.linspace(X_MIN, X_MAX, 600)
ax.plot(x_th, p_entrenched_theory_with_fixation(x_th),
        color="black", lw=1.8)

ax.plot(x_th, p_entrenched_theory_without_fixation(x_th),
        color="black", lw=1, ls="--")

#ax.axvline(X_STAR, color="black", lw=0.7, ls=":")

for s2d in SIGMA_OVER_2D:
    col = COLOURS[s2d]
    for n in N_DIMS:
        x, p, lo, hi = results[s2d][n]
        if p is None:
            continue

        mk = MARKERS[n]
        x_plot = x * 10 ** JITTER_LOG[n]
        yerr = np.array([[p - lo], [hi - p]])

        ax.errorbar(x_plot, p,
                    yerr=yerr,
                    fmt=mk,
                    color=col,
                    markersize=MS,
                    capsize=CAPSIZE,
                    lw=0.9)

# ── Axes config ───────────────────────────────────────────────────────────────
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$x = \sqrt{n}\tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$1-P_{\mathrm{IS}}$")
ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(10**(-5), 1.1)

# ── Légende collée EXACTEMENT à droite ───────────────────────────────────────
pos = ax.get_position()

ax_leg = fig.add_axes([
    pos.x1,          # ← collé au bord droit de l'axe
    pos.y0,
    0.25,            # largeur de la colonne légende
    pos.height
])
ax_leg.axis("off")

legend_colour = [
    plt.Line2D([0], [0], marker="o", color="w",
               markerfacecolor=COLOURS[s2d], markersize=MS,
               label=rf"$\tilde{{\sigma}}/2d={s2d}$")
    for s2d in SIGMA_OVER_2D
]

legend_marker = [
    plt.Line2D([0], [0], marker=mk, color="gray",
               markersize=MS, linestyle="none",
               label=rf"$n={n}$")
    for n, mk in MARKERS.items()
]

leg1 = ax_leg.legend(handles=legend_colour,
                     loc="upper left")

leg2 = ax_leg.legend(handles=legend_marker,
                     loc="lower left")

ax_leg.add_artist(leg1)

# ── Save ──────────────────────────────────────────────────────────────────────
fig.savefig("figS2_P_entrenched_fixation.pdf", bbox_inches="tight")
fig.savefig("figS2_P_entrenched_fixation.tiff", dpi=1200, bbox_inches="tight")

plt.close(fig)

## III.2 Probability of lying in IS depending on y 

In [ ]:
"""
 Probability of entrenchment versus relative distance to the optimum.

Problem   P_entrenched, the probability that a fixed beneficial substitution is
          entrenched, is predicted to depend on the phenotype only through the
          composite parameter x = sqrt(n) * sigma_tilde / (2d).

Approach  For n in {5, 10, 100} we sweep sigma_tilde/(2d) on a log grid. At each
          point, fixed beneficial substitutions are drawn with the MH sampler under
          Kimura fixation weighting and scored for entrenchment. The analytical
          prediction with fixation weighting is overlaid.

Result    Simulations (markers) match the analytical curves (lines) over the whole
          range, confirming the x-scaling. The three dimensions are read from the
          dash pattern and marker shape only; the figure is monochrome.

External dependencies (earlier cells):
    phi, phibar, M_fixed   standard-normal density, upper tail, fixed-mean M(x)
    MetropolisHastingsAR   adaptive-rejection Metropolis-Hastings sampler

Compute and rendering are separated: simulation output is cached to disk so the
figure can be restyled without rerunning the Markov chains. Do not run automatically.
"""



# ── Global style (harmonised with the other panels: Computer Modern maths) ────
mpl.rcParams.update({
    "font.family":         "serif",
    "mathtext.fontset":    "cm",       # matches Figs S1a / three-ensembles
    "font.size":           11,
    "axes.labelsize":      12,
    "axes.linewidth":      0.8,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.major.width":   0.8,
    "ytick.major.width":   0.8,
    "legend.frameon":      False,
    "legend.fontsize":     9,
})

# ── Constants ────────────────────────────────────────────────────────────────
ONE_MM   = 1 / 25.4
D        = 1.0            # distance to optimum d; the sweep variable is sigma_tilde/(2d)
N_DRAWS  =30_000         # fixed beneficial substitutions scored per grid point
MS       = 5             # marker size
CAPSIZE  = 3             # error-bar cap size
X_CUTOFF = 6.0           # skip x > X_CUTOFF 
NE       = 1e5           # effective population size for fixation weighting
CACHE_B  = "fig1b_data.npz"

# ── Per-dimension visual encoding: one line style + one marker per n ──────────
#    Monochrome figure: n is read from the dash pattern and the marker shape only.
N_VALUES = [5, 10, 100]
STYLE = {
    5:   dict(marker="o", linestyle="-"),                # solid  + circle
    10:  dict(marker="s", linestyle=(0, (5, 2))),        # dashed + square
    100: dict(marker="^", linestyle=(0, (1, 1.4))),      # dotted + triangle
}

# ── Analytical prediction (with fixation weighting) ──────────────────────────
def p_entrenched_theory_with_fixation(x):
    """P_entrenched(x) under weak-selection (Kimura) fixation weighting.

    P(x) = M(x)/Phibar(x) * [ (phi(x) - phi(2x)) - x * (Phibar(x) - Phibar(2x)) ],
    with M(x) the fixed-substitution mean projection and Phibar the upper tail.
    """
    tail   = phibar(x) - phibar(2 * x)
    dens   = phi(x)   - phi(2 * x)
    factor = M_fixed(x) / phibar(x)
    return np.where(phibar(x) > 1e-300, factor * (dens - x * tail), 1.0)

# ── Simulation (kept: this part is validated) ────────────────────────────────
def wilson_ci(k, n_obs, z_alpha=1.96):
    """Wilson score 95% CI for a proportion k / n_obs; returns (p_hat, lo, hi)."""
    p_hat  = k / n_obs
    denom  = 1 + z_alpha**2 / n_obs
    centre = (p_hat + z_alpha**2 / (2 * n_obs)) / denom
    half   = z_alpha * np.sqrt(p_hat * (1 - p_hat) / n_obs
                              + z_alpha**2 / (4 * n_obs**2)) / denom
    p_hat = float(np.clip(p_hat, 0, 1))
    lo    = min(float(np.clip(centre - half, 0, 1)), p_hat)
    hi    = max(float(np.clip(centre + half, 0, 1)), p_hat)
    return p_hat, lo, hi


def simulate_p_entrenched_with_fixation_rosenblatt(n, sigma_over_2d, d, n_draws,rng):
    """Simulate the probability of entrenchment."""
    sigma   = sigma_over_2d * 2 * d * (1/np.sqrt(n) )
    x_param = np.sqrt(n)*sigma_over_2d
    z=np.zeros(n)
    z[0]=d

    n_entrenched = 0
    n_fixed = 0
    _,mut=sample_adaptive_walks_cdf(z, sigma, n_steps=1, n_rep=n_draws, rng=RNG,Ne=NE, probability_of_fixation="exact")
    d_fix=mut[:,0,:]
    ent = (np.sum(d_fix**2, axis=1) + d_fix[:, 0] * d) > 0
    n_entrenched += int(ent.sum()) # count the entrenched mutations in this batch
    p_hat, lo, hi = wilson_ci(n_entrenched, n_draws)
    return  x_param, p_hat, lo, hi


# ── 1. Compute (or load cached simulation output) ────────────────────────────
def compute_panel_b(n_values, st_grid, d, n_draws, x_cutoff=X_CUTOFF, cache=CACHE_B):
    """Return {n: (p, lo, hi)} aligned to st_grid; NaN where x > x_cutoff (skipped).

    Output is cached to `cache` so the figure can be restyled without rerunning
    the chains. The cache is reused only if its grid matches st_grid; delete the
    file to force recomputation.
    """
    if cache and os.path.exists(cache):
        with np.load(cache) as f:
            if np.array_equal(f["st_grid"], st_grid):
                return {n: (f[f"p_{n}"], f[f"lo_{n}"], f[f"hi_{n}"]) for n in n_values}

    data, to_save = {}, {"st_grid": st_grid}
    for n in n_values:
        p  = np.full(st_grid.size, np.nan)
        lo = np.full_like(p, np.nan)
        hi = np.full_like(p, np.nan)
        for i, st in enumerate(st_grid):
            if np.sqrt(n) * st > x_cutoff:          # rejection too costly: leave NaN
                continue
            # Pass st = sigma_tilde/(2d) directly; the sampler rescales it internally.
            _,p[i], lo[i], hi[i] = simulate_p_entrenched_with_fixation_rosenblatt(n, st, d, n_draws, RNG)
            print(f"n={n:>3d}  sigma_tilde/2d={st:.3g}  x={np.sqrt(n)*st:.3g}  "
                  f"P={p[i]:.3f}", flush=True)
        data[n] = (p, lo, hi)
        to_save.update({f"p_{n}": p, f"lo_{n}": lo, f"hi_{n}": hi})
    if cache:
        np.savez(cache, **to_save)
    return data

# ── 2. Plot ──────────────────────────────────────────────────────────────────
def plot_panel_b(st_sim, st_theory, data, style,
                 outfile="fig1b_Pent_vs_sigmatilde_fixation"):
    """Draw Panel b from precomputed data. Monochrome; n encoded by line + marker."""
    fig, ax = plt.subplots(figsize=(85 * ONE_MM, 75 * ONE_MM))

    for n, sty in style.items():
        # Analytical curve (with fixation), black, distinct dash pattern
        ax.plot(st_theory,
                p_entrenched_theory_with_fixation(np.sqrt(n) * st_theory),
                color="k", linestyle=sty["linestyle"], lw=1.4, zorder=3)

        # Simulation points with Wilson 95% CI, black, distinct marker
        p, lo, hi = data[n]
        m = np.isfinite(p)
        if m.any():
            yerr = np.vstack([p[m] - lo[m], hi[m] - p[m]])
            ax.errorbar(st_sim[m], p[m], yerr=yerr,
                        fmt=sty["marker"], color="k",
                        markerfacecolor="white", markeredgecolor="k",
                        ecolor="k", elinewidth=0.8, capsize=CAPSIZE,
                        markersize=MS, linestyle="none", zorder=5)

    # Combined legend (line + marker per n), top left
    handles = [Line2D([0], [0], color="k", marker=sty["marker"],
                      linestyle=sty["linestyle"], markerfacecolor="white",
                      markeredgecolor="k", markersize=MS, lw=1.4,
                      label=fr"$n = {n}$")
               for n, sty in style.items()]
    ax.legend(handles=handles, loc="upper left", handlelength=2.8)

    # Axes
    ax.set_xscale("log")
    ax.set_xlabel(r"Relative distance to optimum $\tilde{\sigma}/2d$", labelpad=4)
    ax.set_ylabel(r"1-$P_{\mathrm{IS}}$", labelpad=4)
    ax.set_xlim(st_sim[0] * 0.8, 1.05)
    ax.set_ylim(0, 1.02)

    fig.tight_layout(pad=0.5)
    fig.savefig(f"{outfile}.pdf",  format="pdf",  bbox_inches="tight")
    fig.savefig(f"{outfile}.tiff", format="tiff", dpi=1200, bbox_inches="tight")
    plt.close(fig)

# ── Driver ───────────────────────────────────────────────────────────────────
ST_SIM_B = np.logspace(-2, -0.3, 12)   # sigma_tilde/(2d): 12 points in [1e-2, ~0.50]
ST_TH_B  = np.logspace(-2,  0.0, 500)  # smooth theory grid in [1e-2, 1]

data_b = compute_panel_b(N_VALUES, ST_SIM_B, D, N_DRAWS)   # cached to CACHE_B
plot_panel_b(ST_SIM_B, ST_TH_B, data_b, STYLE)

## III.3 Generalized epistasis $E_{1,\infty}$ versus $x$.

In [ ]:
"""
 Generalized epistasis E_{1,inf} versus x = sqrt(n) * sigma_tilde / (2d).

Problem   E_{1,inf} = <delta, z_0 + delta> is the epistasis between the first
          substitution and the optimum. We test the Regime I predictions
          (Supp. Section 5) against simulation under fixation weighting.

Approach  sigma_tilde = 0.1 is fixed. For each target x we vary d
          (d = sigma_tilde*sqrt(n)/(2x)) so that all n reach the same x, with
          sigma = sigma_tilde/sqrt(n). Fixed beneficial substitutions are drawn
          with the MH sampler (Kimura weighting) and their E_{1,inf} distribution
          is shown as a violin, jittered by n.

Theory (fixation ensemble, Regime I; d*sigma = sigma_tilde^2/(2x)):
    E_F [E_{1,inf}] = (sti^2 / 2x) * (2x - M(x))          [eq. ES_fix_regimeI]
    Var_F[E_{1,inf}]/(d^2 sigma^2) = 2 + x*M(x) - M(x)^2  [eq. VarS_fix_regimeI]
  so  SD_F[E_{1,inf}] = (sti^2 / 2x) * sqrt(2 + x*M(x) - M(x)^2).
    (Constant 2 with M for the fixed ensemble; 1 with Lambda for beneficial-only.)

Sign change of the expectation at x_F* solving M(x) = 2x  (~ 0.9254).


"""

import os
from scipy.optimize import brentq
from matplotlib.transforms import blended_transform_factory

# ── Global style (Computer Modern maths, harmonised with the other panels) ────
mpl.rcParams.update({
    "font.family":         "serif",
    "mathtext.fontset":    "cm",
    "font.size":           11,
    "axes.labelsize":      12,
    "axes.linewidth":      0.8,
    "xtick.direction":     "in",
    "ytick.direction":     "in",
    "xtick.major.size":    4,
    "ytick.major.size":    4,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.minor.size":    2,
    "ytick.minor.size":    2,
    "xtick.major.width":   0.8,
    "ytick.major.width":   0.8,
    "legend.frameon":      False,
    "legend.fontsize":     9,
})

# ── Constants ────────────────────────────────────────────────────────────────
ONE_MM  = 1 / 25.4
N_DRAWS = 30_000
X_MIN, X_MAX = 0.08, 3.0
STI     = 0.1                             # fixed mutational norm sigma_tilde
N_DIMS  = [5, 10, 100]
X_SIMS  = [0.1, 0.3, 0.612, 1.0, 2.0]  # target x values
NE=1e5   
CACHE   = "figS3_data.npz"

# ── Visual encoding ──────────────────────────────────────────────────────────
COLOURS = {5: "#e69f00", 10: "#56b4e9", 100: "#cc79a7"}   # orange / sky blue / purple
MARKERS = {5: "o", 10: "s", 100: "^"}
JITTER_LOG   = {5: -0.07, 10: 0.0, 100: +0.07}   # offset in log10(x); non-overlapping
VIOLIN_WIDTH = 0.032                             # half-width in log10(x) units

# ── Functions of x ───────────────────────────────────────────────────────────
def phi(x):     return norm.pdf(x)
def phibar(x):  return norm.sf(x)

def lambda_mills(x):
    """Inverse Mills ratio Lambda(x) = phi(x) / Phibar(x) (beneficial mean projection)."""
    return phi(x) / phibar(x)

def M_fixed(x):
    """Fixed-substitution mean projection M(x) = 1 / (Lambda(x) - x)."""
    return 1.0 / (lambda_mills(x) - x)

def e1inf_theory(x, sti=STI):
    """E_F[E_{1,inf}] = (sti^2/2x)*(2x - M(x)) = sti^2*(1 - M(x)/2x)  (Supp. eq. ES_fix)."""
    return sti**2 * (1.0 - M_fixed(x) / (2.0 * x))

def e1inf_std_theory(x, sti=STI):
    """SD_F[E_{1,inf}] = (sti^2/2x)*sqrt(2 + x*M(x) - M(x)^2)  (Supp. eq. VarS_fix).

    The constant is 2 for the fixation ensemble (paired with M), not 1 (which
    pairs with Lambda for the beneficial-only ensemble). Clamped at 0 for safety.
    """
    M_x = M_fixed(x)
    var = np.maximum(2.0 + x * M_x - M_x**2, 0.0)
    return (sti**2 / (2.0 * x)) * np.sqrt(var)

# Sign-change threshold of the expectation under fixation: M(x) = 2x
x_F = brentq(lambda x: M_fixed(x) - 2.0 * x, 0.1, 3.0)   # ~ 0.9254

# ── Simulation (validated; unchanged apart from annotation) ──────────────────
def simulate_e1inf(n, x_target, sti, n_draws):
    """Draw fixed beneficial substitutions and return (x_actual, E_{1,inf} array).

    sigma = sti/sqrt(n); d = sti*sqrt(n)/(2*x_target) so that x_actual = x_target.
    E_{1,inf} = ||delta||^2 + d * delta_parallel  (parallel = component 0).
    """
    sigma = sti / np.sqrt(n)
    d     = sti * np.sqrt(n) / (2.0 * x_target)
    x_actual = n * sigma / (2.0 * d)
    z=np.zeros(n)
    z[0]=d
    _,mut=sample_adaptive_walks_cdf(z, sigma, n_steps=1, n_rep=n_draws, rng=RNG,Ne=NE, probability_of_fixation="exact")
    d_fix=mut[:,0,:]
    e_vals = np.sum(d_fix**2, axis=1) + d * d_fix[:, 0]   # ||delta||^2 + d*delta_par
    return x_actual,e_vals

# ── 1. Compute (or load cached samples) ──────────────────────────────────────
def compute_figS3(n_dims, x_sims, sti, n_draws, cache=CACHE):
    """Return sim_data[n][x_target] = (x_actual, samples); cached to disk.

    Reused only if (n_dims, x_sims, n_draws, sti) match the stored run; delete
    the file to force recomputation.
    """
    key = np.array([len(n_dims), len(x_sims), n_draws, sti], dtype=float)
    if cache and os.path.exists(cache):
        with np.load(cache) as f:
            if np.array_equal(f["key"], key):
                return {n: {x: (float(f[f"xa_{n}_{i}"]), f[f"e_{n}_{i}"])
                            for i, x in enumerate(x_sims)} for n in n_dims}

    sim_data, to_save = {}, {"key": key}
    for n in n_dims:
        sim_data[n] = {}
        for i, x_t in enumerate(x_sims):
            print(f"Simulating n={n}, x_target={x_t:.3f} ...", flush=True)
            x_actual, vals = simulate_e1inf(n, x_t, sti, n_draws)
            sim_data[n][x_t] = (x_actual, vals)
            to_save[f"xa_{n}_{i}"] = x_actual
            to_save[f"e_{n}_{i}"]  = vals
            print(f"  x_actual={x_actual:.4f}  mean={vals.mean():.4f}  "
                  f"theory={e1inf_theory(x_actual):.4f}")
    if cache:
        np.savez(cache, **to_save)
    return sim_data

# ── 2. Plot ──────────────────────────────────────────────────────────────────

# ── 2. Plot (updated: theory anchors at the true x + x_F* label at the bottom) ──
def plot_figS3(sim_data, outfile="figS3_E1inf_violin"):
    fig = plt.figure(figsize=(160 * ONE_MM, 105 * ONE_MM))
    ax  = fig.add_axes([0.12, 0.13, 0.55, 0.82])

    # Theory mean and +/- 2 SD band
    x_th = np.logspace(np.log10(X_MIN), np.log10(X_MAX), 800)
    mu, sd = e1inf_theory(x_th), e1inf_std_theory(x_th)
    ax.plot(x_th, mu, color="black", lw=1.8, zorder=3, label="Theory")
    ax.fill_between(x_th, mu - 2 * sd, mu + 2 * sd,
                    color="black", alpha=0.10, zorder=2)

    # Reference lines
    ax.axhline(0.0, color="black", lw=0.6, ls="-",  zorder=1)
    ax.axvline(x_F, color="black", lw=0.8, ls=":",  zorder=2)
    trans = blended_transform_factory(ax.transData, ax.transAxes)  # x data, y axes
    ax.text(x_F * 1.05, 0.04, r"$x_{F}^{*}$", transform=trans,      # bottom of the axis
            fontsize=9, ha="left", va="bottom")
    
    x_eval = np.array(
        sorted({
            x_actual
            for n in N_DIMS
            for x_actual, _ in sim_data[n].values()
        }),
        dtype=float,
    )

    y_eval = e1inf_theory(x_eval)

    tick_half_height: float = 0.0075

    for x_value, y_value in zip(x_eval, y_eval):
        ax.plot(
            [x_value, x_value],
            [y_value - tick_half_height, y_value + tick_half_height],
            color="black",
            lw=1.0,
            solid_capstyle="butt",
            zorder=3,
        )

    # Violins, jittered by n; marker = sample MEAN (matches the theory curve)
    for n in N_DIMS:
        col = COLOURS[n]
        for x_actual, vals in sim_data[n].values():
            x_plot = x_actual * 10 ** JITTER_LOG[n]
            w_lin  = x_plot * (10 ** VIOLIN_WIDTH - 1)

            y_lo, y_hi = np.percentile(vals, [1.0, 99.0])   # central 98% support
            y_grid = np.linspace(y_lo, y_hi, 300)
            dens   = gaussian_kde(vals, bw_method="scott")(y_grid)
            dens   = dens / dens.max() * w_lin

            ax.fill_betweenx(y_grid, x_plot - dens, x_plot + dens,
                             color=col, alpha=0.50, zorder=4)
            ax.plot(x_plot - dens, y_grid, color=col, lw=0.4, zorder=5)
            ax.plot(x_plot + dens, y_grid, color=col, lw=0.4, zorder=5)
            ax.plot(x_plot, vals.mean(), marker=MARKERS[n], color="white",
                    markersize=4.0, markeredgecolor=col, markeredgewidth=0.9,
                    zorder=6)

    # Anchors on the theory curve at the true (un-jittered) x being evaluated.
    # Violins are offset by the jitter; these dots recover the exact x of each cluster.
# Small vertical ticks on the theory curve at the true, un-jittered x values.

    ax.set_xscale("log")
    ax.set_xlabel(r"$x = \tilde{\sigma}\sqrt{n}\,/\,(2d)$", labelpad=4)
    ax.set_ylabel(r"$\mathcal{E}_{1,\infty}$", labelpad=4)
    ax.set_xlim(X_MIN, X_MAX)

    # Legend on separate axes to the right
    pos    = ax.get_position()
    ax_leg = fig.add_axes([pos.x1 + 0.01, pos.y0, 0.30, pos.height])
    ax_leg.axis("off")
    handles_n = [plt.Line2D([0], [0], marker=MARKERS[n], color="w",
                            markerfacecolor=COLOURS[n], markeredgecolor=COLOURS[n],
                            markersize=6, label=rf"$n = {n}$") for n in N_DIMS]
    handles_th = [
        plt.Line2D(
            [0], [0],
            color="black",
            lw=1.5,
            label="Theory (mean)",
        ),
        plt.Line2D(
            [0], [0],
            color="black",
            marker="|",
            linestyle="none",
            markersize=7,
            markeredgewidth=1.0,
            label=r"Evaluated $x$",
        ),
        plt.Rectangle(
            (0, 0),
            1,
            1,
            fc="black",
            alpha=0.15,
            label=r"$\pm 2\,\mathrm{SD}$",
        ),
    ]
    leg1 = ax_leg.legend(handles=handles_n, loc="upper left", borderaxespad=0,
                         handletextpad=0.4, title=r"$\tilde{\sigma} = 0.1$ fixed")
    ax_leg.legend(handles=handles_th, loc="lower left", borderaxespad=0,
                  handletextpad=0.4)
    ax_leg.add_artist(leg1)

    fig.savefig(f"{outfile}.pdf",  format="pdf",  bbox_inches="tight")
    fig.savefig(f"{outfile}.tiff", format="tiff", dpi=1200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {outfile}.pdf / .tiff")


# ── Driver ───────────────────────────────────────────────────────────────────
sim_data = compute_figS3(N_DIMS, X_SIMS, STI, N_DRAWS)   # cached to CACHE
plot_figS3(sim_data)

# IV.  Adaptative walk : Generator

## IV.1 Function based on rejection sampling and MCMC

In [ ]:

# ============================================================
# Adaptive walk simulation (NumPy tensor version)
# ============================================================

def simulate_adaptive_walk(
        n:int=50,
        sigma_m:float=0.05,
        Ne:int=10**7,
        steps:int=40,
        replicates:int=10,
        initial_size:float=1.0,
        MH_size:int=15000,
        kimura:bool=True,
        only_beneficial:bool=True,
        exact=True,
        single_first_mutation:bool=False,
        verbose:bool=True)->np.ndarray:
    """
    Simulate adaptive walks in Fisher geometric model.Forward in time version.

    Returns
    -------
    Z : numpy array

        Shape:
        (replicates, steps+1, n)

        Z[r, t] is the phenotype vector of replicate r at step t.
    """

    # tensor storing phenotype trajectories
    Z = np.zeros((replicates, steps + 1, n))

    # initial condition Z(0) = (1,0,0,...)
    Z[:, 0, 0] = initial_size
    k_initial=0 #index for the initial step for the adaptive walk (0 if no single first mutation, 1 if single first mutation simulated as bulk for all replicates)
    alpha_a_prop=1
    alpha_r_prop=1
    if single_first_mutation: # draw a single first mutation for all replicates
        nb_rejection_trials:int=1000
        z = Z[0, 0].copy()
        try:
            mutation,_= draw_mutation_by_rejection(z, sigma_m=sigma_m, Ne=Ne, max_attempts=nb_rejection_trials,kimura=kimura,only_beneficial=only_beneficial, exact=exact)
            #If the rejection sampling fails, we switch to MH sampling for the first mutation
            #If the rejection sampling works we do not use first mutation for the other replicates, we draw them independently
        except RuntimeError:
            mutation_x=MetropolisHastingsAR()
            mutation_x.set_problem(z=z,sigma=sigma_m,kimura=kimura,only_beneficial=only_beneficial,Ne=Ne,exact=exact)
            mutation_x.initialize(nsteps=max(MH_size,replicates*200), thinning=40,prop_burnin=0.2,alpha_a_prop=alpha_a_prop, alpha_r_prop=alpha_r_prop)
            mutation_x.run_with_diagnostics(max_retries=8,verbose=False)
            mutation_x.sample_x()
            mutation_x.sample_multiple_x()
            mutation = mutation_x.x_mut[0:replicates,:]
            #Uptade proposal parameters for next time if needed
            alpha_r_prop=mutation_x.alpha_r_prop
            alpha_a_prop=mutation_x.alpha_a_prop  
            Z[:,1,:] = mutation
            k_initial=1

    for r in range(replicates):
        z = Z[r, k_initial].copy()
        nb_rejection_trials:int=1000
        for step in range(k_initial+1, steps + 1):
            try:
                mutation,_= draw_mutation_by_rejection(z, sigma_m=sigma_m, Ne=Ne, max_attempts=nb_rejection_trials,kimura=kimura,only_beneficial=only_beneficial, exact=exact)
                z_mut = z + mutation
                z= z_mut
            except RuntimeError:
                    if verbose:
                        print(f"replicate {r}, step {step} Warning: failed to draw mutation by rejection after {nb_rejection_trials} attempts. Switching to MH sampling for mutation generation.")
                    mutation_x=MetropolisHastingsAR()
                    mutation_x.set_problem(z=z,sigma=sigma_m,kimura=kimura,only_beneficial=only_beneficial,Ne=Ne,exact=exact)
                    mutation_x.initialize(nsteps=MH_size, thinning=20,prop_burnin=0.2,alpha_a_prop=alpha_a_prop, alpha_r_prop=alpha_r_prop)
                    mutation_x.run_with_diagnostics(max_retries=8,verbose=False)
                    mutation_x.sample_x()
                    mutation = mutation_x.last_sample_x
                    #Uptade proposal parameters for next time if needed
                    alpha_r_prop=mutation_x.alpha_r_prop
                    alpha_a_prop=mutation_x.alpha_a_prop  
                    z_mut = z + mutation  
                    z = z_mut 
            Z[r, step] = z
    return Z


### IV.1.2 Utilitary functions for walks

In [ ]:


def extract_mutational_vectors(Z: np.ndarray) -> np.ndarray:
    """
    Extract mutational vectors from phenotype trajectories.

    Parameters
    ----------
    Z : ndarray
        Shape (replicates, steps+1, n)

    Returns
    -------
    X : ndarray
        Shape (replicates, steps, n)

        X[r, t] = z_{t+1} - z_t = x_{t+1} (math indexing)
        So Python index t corresponds to mutation step t+1
        X[:, k-1, :] = x_k = z_k - z_{k-1}

    """
    return Z[:, 1:, :] - Z[:, :-1, :]


def extract_mutational_size(X:np.ndarray)->np.ndarray:
    """
    Extract mutational sizes from mutational vectors.

    Returns
    -------
    S : numpy array

        Shape:
        (replicates, steps)

        S[:, k-1] = ||x_k||
    """
    return np.linalg.norm(X, axis=2)



def extract_distance_to_optimum(Z:np.ndarray)->np.ndarray:
    """
    Extract distance to optimum from phenotype trajectories.

    Returns
    -------
    D : numpy array

        Shape:
        (replicates, steps+1)

        D[r, t] is the distance to optimum of the phenotype Z[r, t].
    """
    return np.linalg.norm(Z, axis=2)





# ==============================
# 4. Scalar product utility
# ==============================

def dot_product(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Row-wise dot product.
    """
    return np.einsum('rn,rn->r', a, b)


# ==============================
# compute angle between vectors
# ==============================
def compute_angle(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Compute angle in radians between vectors a and b for each replicate.

    Parameters
    ----------
    a : ndarray
        Shape (replicates, n)

    b : ndarray
        Shape (replicates, n)

    Returns
    -------
    angles : ndarray 
        Shape (replicates,)

        angles[r] is the angle in radians between a[r] and b[r].
        beetween 0 and pi.
    """
    dot_prod = dot_product(a, b)
    norm_a = np.linalg.norm(a, axis=1)
    norm_b = np.linalg.norm(b, axis=1)
    cos_theta = dot_prod / (norm_a * norm_b + 1e-12)  # add small value to avoid division by zero
    cos_theta = np.clip(cos_theta, -1.0, 1.0)  # clip to valid range for arccos
    angles = np.arccos(cos_theta)
    return angles



# ==============================
# 5. Compute E_ij (math indexing)
# ==============================

def compute_Eij(Z: np.ndarray, i: int, j: int) -> np.ndarray:
    """
    i, j in {1,...,steps}  (math indexing)
    i,j are mutation steps (1-based index) ( 1,2,...,steps)
    Compute E_ij for contingency / entrenchment analysis.

    E_ij =
        <x_i , z_{i-1} - z_{j-1}>  if i >= j
        <x_i , z_i     - z_j    >  if i < j
    """
    X = extract_mutational_vectors(Z)

    # x_i
    mut = X[:, i-1, :]

    if i >= j:
        delta = Z[:, i-1, :] - Z[:, j-1, :]
    else:
        delta = Z[:, i, :] - Z[:, j, :]

    return dot_product(mut, delta)


# ==============================
# 6. Table E_ij for fixed i
# ==============================

def Eij_table_fixed_i(Z: np.ndarray, i: int) -> np.ndarray:
    """
    i in {1,...,steps}

    Returns E_ij for all j = 1..steps

    Compute E_ij for all j at fixed i (contingency / entrenchment analysis).

    Mathematical definitions
    ------------------------
    We consider a phenotypic trajectory:
        z_0, z_1, ..., z_T

    with mutations:
        x_k = z_k - z_{k-1},   for k = 1,...,T

    The quantity E_ij is defined as:

        E_ij =
            <x_i , z_{i-1} - z_{j-1}>   if i >= j
            <x_i , z_i     - z_j    >   if i < j

    Interpretation:
    - i >= j : projection onto past genetic background (contingency)
    - i < j  : projection onto future genetic background (entrenchment)


    Parameters
    ----------
    Z : ndarray
        Phenotype tensor of shape (replicates, steps+1, n)

        Z[:, k, :] corresponds to z_k in mathematical notation
        (k = 0,...,steps)

    i : int
        Fixed mutation index (1-based indexing)

        i in {1,...,steps}
        corresponds to mutation x_i = z_i - z_{i-1}


    Returns
    -------
    E_table : ndarray
        Shape (replicates, steps)

        E_table[r, j-1] = E_ij for replicate r and mutation index j

        where j runs from 1 to steps (1-based indexing)


    Notes
    -----
    - Internally, Python uses 0-based indexing:
        x_i is stored as X[:, i-1, :]
        z_k is stored as Z[:, k, :]

    """
    
    
    replicates, steps_plus1, n = Z.shape
    steps = steps_plus1 - 1

    if i < 1 or i > steps:
        raise ValueError(f"i must be in [1, {steps}]")

    X = extract_mutational_vectors(Z)
    mut = X[:, i-1, :]

    E_table = np.zeros((replicates, steps))

    for j in range(1, steps+1):

        if i >= j:
            delta = Z[:, i-1, :] - Z[:, j-1, :]
        else:
            delta = Z[:, i, :] - Z[:, j, :]

        E_table[:, j-1] = np.einsum('rn,rn->r', mut, delta)

    return E_table


In [ ]:

def plot_mutation_and_distance(Z: np.ndarray):
    """
    Plot the evolution of mutation size |x(t)| and distance to optimum |Z(t)|
    for each replicate.

    Parameters
    ----------
    Z : ndarray
        Phenotype tensor with shape

            (replicates, steps+1, n)
    """

    # ------------------------------------------------
    # Extract mutational vectors and their sizes
    # ------------------------------------------------

    X = extract_mutational_vectors(Z)

    mutation_sizes = extract_mutational_size(X)

    # ------------------------------------------------
    # Extract distances to optimum
    # ------------------------------------------------

    distances = extract_distance_to_optimum(Z)

    replicates = Z.shape[0]
    steps = mutation_sizes.shape[1]

    t_mut = np.arange(1, steps+1)
    t_dist = np.arange(0, steps+1)


    # generate colors automatically
    colors = plt.cm.viridis(np.linspace(0, 1, replicates))

    # ====================================================
    # Plot mutation sizes |x(t)|
    # ====================================================

    plt.figure(figsize=(7,5))

    for r in range(replicates):

        plt.plot(
            t_mut,
            mutation_sizes[r],
            marker="o",
            linestyle="-",
            color=colors[r],
            label=f"replicate {r}"
        )

    plt.xlabel("Step t")
    plt.ylabel("Mutation size |x(t)|")
    plt.title("Evolution of mutation size during adaptive walk")

    #plt.legend()
    plt.grid(True)

    plt.show()

    # ====================================================
    # Plot distance to optimum |Z(t)|
    # ====================================================

    plt.figure(figsize=(7,5))

    for r in range(replicates):

        plt.plot(
            t_dist,
            distances[r],
            marker="o",
            linestyle="-",
            color=colors[r],
            label=f"replicate {r}"
        )

    plt.xlabel("Step t")
    plt.ylabel("Distance to optimum |Z(t)|")
    plt.title("Distance to optimum during adaptive walk")

    #  plt.legend()
    plt.grid(True)

    plt.show()


In [ ]:

def plot_angle(Z: np.ndarray):
    """
    Plot the evolution of the angle between:
    - subsequent mutation vectors x(t) and x(t+1)
    - mutation vector x(t) and current phenotype Z(t-1) (phenotype before mutation t)
    - phenotype Z(t) and initial phenotype Z(0)
    - first mutation x(1) and x(t)

    Parameters
    ----------
    Z : ndarray
        Phenotype tensor with shape (replicates, steps+1, n)
        Z[:, k, :] = z_k,  z_0 is the initial phenotype (Z0)
        x_k = z_k - z_{k-1}  stored in X[:, k-1, :]
    """

    X = extract_mutational_vectors(Z)
    # X shape : (replicates, steps, n)
    # X[:, k-1, :] = x_k

    replicates = Z.shape[0]
    steps = X.shape[1]   # number of mutation steps

    colors = plt.cm.viridis(np.linspace(0, 1, replicates))

    # ====================================================
    # 1. angle(x(t), x(t+1))  pour t = 1,...,steps-1
    # ====================================================
    # X[:, t-1, :] = x_t,  X[:, t, :] = x_{t+1}
    # t varie de 1 à steps-1  =>  indices Python 0 à steps-2

    t_consec = np.arange(1, steps)          # t = 1,...,steps-1  (axe x : indice de la 1ère mutation)

    angles_consec = np.zeros((replicates, steps - 1))  # shape (replicates, steps-1)

    for t_idx in range(steps - 1):           # t_idx = t-1,  t = t_idx+1
        a = X[:, t_idx, :]                   # x(t)
        b = X[:, t_idx + 1, :]              # x(t+1)
        angles_consec[:, t_idx] = compute_angle(a, b)

    plt.figure(figsize=(7, 5))
    for r in range(replicates):
        plt.plot(t_consec, np.degrees(angles_consec[r]),
                 marker="o", linestyle="-", color=colors[r])
    plt.xlabel("Step t")
    plt.ylabel("Angle x(t) ∧ x(t+1)  (degrees)")
    plt.title("Angle between consecutive mutation vectors x(t) and x(t+1)")
    plt.grid(True)
    plt.show()

    # ====================================================
    # 2. angle(x(t), Z(t-1))  pour t = 1,...,steps
    # Z(t-1) est le phénotype AVANT la mutation x(t)
    # ====================================================
    # x_t = X[:, t-1, :],  z_{t-1} = Z[:, t-1, :]

    t_xz = np.arange(1, steps + 1)          # t = 1,...,steps

    angles_xz = np.zeros((replicates, steps))

    for t_idx in range(steps):               # t_idx = t-1
        a = X[:, t_idx, :]                   # x(t)   = x_{t_idx+1}
        b = Z[:, t_idx, :]                   # Z(t-1) = z_{t_idx}
        angles_xz[:, t_idx] = compute_angle(a, b)

    plt.figure(figsize=(7, 5))
    for r in range(replicates):
        plt.plot(t_xz, np.degrees(angles_xz[r]),
                 marker="o", linestyle="-", color=colors[r])
    plt.xlabel("Step t")
    plt.ylabel("Angle [x(t), Z(t-1)]  (degrees)")
    plt.title("Angle between mutation x(t) and phenotype Z(t-1) before mutation")
    plt.grid(True)
    plt.show()

    # ====================================================
    # 3. angle(Z(t), Z(0))  pour t = 1,...,steps
    # ====================================================

    t_zz0 = np.arange(1, steps + 1)

    angles_zz0 = np.zeros((replicates, steps))

    Z0 = Z[:, 0, :]                          # shape (replicates, n) — phénotype initial

    for t_idx in range(1, steps + 1):        # t = 1,...,steps
        a = Z[:, t_idx, :]                   # Z(t)
        b = Z0                               # Z(0)
        angles_zz0[:, t_idx - 1] = compute_angle(a, b)

    plt.figure(figsize=(7, 5))
    for r in range(replicates):
        plt.plot(t_zz0, np.degrees(angles_zz0[r]),
                 marker="o", linestyle="-", color=colors[r])
    plt.xlabel("Step t")
    plt.ylabel("Angle [Z(t), Z(0)]  (degrees)")
    plt.title("Angle between current phenotype Z(t) and initial phenotype Z(0)")
    plt.grid(True)
    plt.show()

    # ====================================================
    # 4. angle(x(1), x(t))  pour t = 2,...,steps
    # ====================================================

    t_x1xt = np.arange(2, steps + 1)        # t = 2,...,steps

    angles_x1xt = np.zeros((replicates, steps - 1))

    x1 = X[:, 0, :]                         # x(1) = X[:, 0, :], shape (replicates, n)

    for t_idx in range(1, steps):            # t_idx = t-1,  t = t_idx+1 >= 2
        b = X[:, t_idx, :]                   # x(t)
        angles_x1xt[:, t_idx - 1] = compute_angle(x1, b)

    plt.figure(figsize=(7, 5))
    for r in range(replicates):
        plt.plot(t_x1xt, np.degrees(angles_x1xt[r]),
                 marker="o", linestyle="-", color=colors[r])
    plt.xlabel("Step t")
    plt.ylabel("Angle [x(1), x(t)]  (degrees)")
    plt.title("Angle between first mutation x(1) and subsequent mutation x(t)")
    plt.grid(True)
    plt.show()

In [ ]:
def plot_Eij_median_sd(E: np.ndarray, i_fixed: int = None):
    """
    Plot median ± 2 SD of E_ij across replicates (math convention, 1-based indexing).

    Parameters
    ----------
    E : ndarray
        Shape (replicates, steps)

        E[r, j-1] = E_ij for replicate r and mutation index j (1-based)

    i_fixed : int
        Fixed mutation index i (1-based, in {1,...,steps})

    Notes
    -----
    - j runs from 1 to steps (mutation indices, not phenotype indices)
    - Blue: j <= i  (contingency region)
    - Red:  j > i   (entrenchment region)
    """

    replicates, steps = E.shape

    if i_fixed is None:
        raise ValueError("i_fixed must be provided (1-based indexing)")

    if i_fixed < 1 or i_fixed > steps:
        raise ValueError(f"i_fixed must be in [1, {steps}]")

    # j in math convention
    j = np.arange(1, steps + 1)

    median = np.median(E, axis=0)
    sd = np.std(E, axis=0)

    # Masks (math convention)
    mask_blue = j <= i_fixed
    mask_red  = j > i_fixed

    plt.figure(figsize=(8, 5))

    # j <= i (contingency)
    plt.errorbar(
        j[mask_blue],
        median[mask_blue],
        yerr=2 * sd[mask_blue],
        fmt='o-',
        color='blue',
        ecolor='lightblue',
        elinewidth=2,
        capsize=3,
        label=f'j ≤ i={i_fixed}'
    )

    # j > i (entrenchment)
    plt.errorbar(
        j[mask_red],
        median[mask_red],
        yerr=2 * sd[mask_red],
        fmt='o-',
        color='red',
        ecolor='salmon',
        elinewidth=2,
        capsize=3,
        label=f'j > i={i_fixed}'
    )

    # Connecting red line between last blue and first red point
    if i_fixed < steps:
        plt.plot(
            [i_fixed, i_fixed + 1],
            [median[i_fixed - 1], median[i_fixed]],
            color='red',
            linewidth=2
        )

    plt.xlabel("Mutation index j (1-based)")
    plt.ylabel(r"$E_{ij}$")

    plt.title(f"Contingency / entrenchment (fixed i = {i_fixed})")

    plt.axvline(i_fixed, color='black', linestyle='--', alpha=0.5)

    plt.grid(True)
    plt.legend()

    plt.show()

## IV.2 Testing of adaptative walk generator

In [ ]:
# ============================================================
# Example usage for forwars adaptive walk simulation
# ============================================================
STEPS = 10
Z = simulate_adaptive_walk(
    n=25,
    sigma_m=0.05,
    Ne=10**5,
    steps=STEPS,
    replicates=10,
    initial_size=2,
    MH_size=15000,
    kimura=True,
    only_beneficial=True
)
plot_mutation_and_distance(Z)
plot_angle(Z)
E=Eij_table_fixed_i(Z, i=np.floor(STEPS/2).astype(int))  # i = middle step
plot_Eij_median_sd(E, i_fixed=np.floor(STEPS/2).astype(int))


In [ ]:
# ============================================================
# Example usage
# ============================================================
Z = simulate_adaptive_walk(
    n=5,
    sigma_m=0.05,
    Ne=10**7,
    steps=16,
    replicates=10,
    initial_size=1.0,
    single_first_mutation=True
)
plot_mutation_and_distance(Z)
plot_angle(Z)
E=Eij_table_fixed_i(Z, i=8)
plot_Eij_median_sd(E, i_fixed=8)


# V Adaptative Walks in Regime I-II

In [ ]:
"""
 Contingency and entrenchment along adaptive walks.

Strategy: simulate walks in batches, accumulate valid windows until
min_valid is reached. d_start and steps_long are fixed manually per
(n, d_target) in two dictionaries. No adaptive restarting.

Epistasis is normalized by mean fitness gain E[s_i] at the center step,
making panels comparable across (n, d_target) combinations.

Center step i* is plotted in black (E_{i,i} = 0 by definition).
15 contingency points (j < i) in blue, 15 entrenchment points (j > i) in red.
To achieve symmetry: steps = 2*WINDOW + 1 = 33, i_fixed = WINDOW + 1 = 17.

Grid:
  rows    : n = 5, 50
  columns : d_target = 2.0, 0.2, 0.07
  sti = 0.1 fixed, sigma = sti/sqrt(n)
  kimura = False, only_beneficial = True
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import norm as scipy_norm



# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":      "serif",
    "mathtext.fontset": "dejavuserif",
    "font.size":        11,
    "axes.labelsize":   12,
    "axes.linewidth":   0.8,
    "xtick.direction":  "in",
    "ytick.direction":  "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "legend.frameon":   False,
    "legend.fontsize":  9,
})

ONE_MM = 1 / 25.4
if True: 
    #STI       = 0.1
    #N_DIMS    = [10, 100]
    #D_TARGETS = [2.5,0.25, 0.07]
    
    STI       = 0.1
    N_DIMS    = [5,10, 100]
    D_TARGETS = [5,0.5, 0.05]

    # steps = 2*WINDOW + 1 gives WINDOW steps before i_fixed,
    # i_fixed itself, and WINDOW steps after i_fixed.
    # With WINDOW=15: i_fixed=16, steps=31, window on each side = 15.
    WINDOW    = 15           # steps before and after i_fixed
    STEPS_WIN = 2 * WINDOW   # = 30 steps total (+ z_0 = 31 phenotype vectors)
    I_FIXED   = WINDOW + 1   # = 16, 1-based center in extracted window

    NE        = 10**5
    EPS_REL   = 0.1         # |d_{i*} - d_target| < eps_rel * d_target
    MIN_VALID = 100         # accumulate until this many valid windows
    BATCH     = 5           # replicates per simulation batch

    COLOUR_CONT   = "#0072b5"   # blue  — contingency (j < i)
    COLOUR_ENTR   = "#bc3c29"   # red   — entrenchment (j > i)
    COLOUR_CENTER = "black"     # black — center point (j = i, E=0)

    # ── Manually calibrated starting distances per (n, d_target) ──────────────────
    # d_start: initial distance for long walks (must be > d_target)
    # steps_long: total walk length (must be > 2*WINDOW + 1 to allow window)

    D_START_5 = {
        5:  6.9,
        0.5:  1.5,
        0.05: 1,
    }
    D_START_10 = {
        5:  6,
        0.5:  1.2,
        0.05: 0.75,
    }
    D_START_100 = {
        5:  5.2,
        0.5:  0.62,
        0.05: 0.1,
    }
    STEPS_LONG = {
        5:  50,
        0.5:  50,
        0.05: 50,
    }   

if False:
    # ── Simulation parameters ─────────────────────────────────────────────────────
    STI       = 0.1
    N_DIMS    = [5, 50]
    D_TARGETS = [2.0, 0.2, 0.07]

    # steps = 2*WINDOW + 1 gives WINDOW steps before i_fixed,
    # i_fixed itself, and WINDOW steps after i_fixed.
    # With WINDOW=15: i_fixed=16, steps=31, window on each side = 15.
    WINDOW    = 15           # steps before and after i_fixed
    STEPS_WIN = 2 * WINDOW   # = 30 steps total (+ z_0 = 31 phenotype vectors)
    I_FIXED   = WINDOW + 1   # = 16, 1-based center in extracted window

    NE        = 10**7
    EPS_REL   = 0.02         # |d_{i*} - d_target| < eps_rel * d_target
    MIN_VALID = 30          # accumulate until this many valid windows
    BATCH     = 5           # replicates per simulation batch

    COLOUR_CONT   = "#0072b5"   # blue  — contingency (j < i)
    COLOUR_ENTR   = "#bc3c29"   # red   — entrenchment (j > i)
    COLOUR_CENTER = "black"     # black — center point (j = i, E=0)

    # ── Manually calibrated starting distances per (n, d_target) ──────────────────
    # d_start: initial distance for long walks (must be > d_target)
    # steps_long: total walk length (must be > 2*WINDOW + 1 to allow window)
    D_START_5 = {
        2.0:  2.6,
        0.2:  0.9,
        0.07: 0.6,
    }
    D_START_50 = {
        2.0:  2.25,
        0.2:  0.45,
        0.07: 0.17,
    }
    STEPS_LONG = {
        2.0:  50,
        0.2:  50,
        0.07: 50,
    }


# Densities and summary statistics
# --------------------------------------------------------------------------
def phi(u):
    """Standard normal density."""
    return norm.pdf(u)

def phibar(x):
    """Upper-tail probability 1 - Phi(x)."""
    return norm.sf(x)

def D(x):
    """Normalising constant of the fixed-substitution density: phi(x) - x*(1 - Phi(x))."""
    return phi(x) - x * phibar(x)

def lambda_mills(x):
    """Mean projection of beneficial mutations, E[U | U > x] = phi(x) / Phibar(x)."""
    return phi(x) / phibar(x)

def M_fixed(x):
    """Mean projection of fixed substitutions, E_fix[U | x] = Phibar(x) / D(x)."""
    return phibar(x) / D(x)



# ── Compute E_{i_fixed, j} from extracted windows ────────────────────────────

def compute_Eij_from_window(Z_windows, i_fixed):
    """
    Compute E_{i_fixed, j} for all j = 1..steps from Z_windows.

    Mathematical definition:
        E_{i,j} = <x_i, z_{i-1} - z_{j-1}>  if i >= j  (contingency)
        E_{i,j} = <x_i, z_i    - z_j   >    if i <  j  (entrenchment)
        E_{i,i} = 0  by definition

    where x_i = z_i - z_{i-1} is the i-th mutational vector.

    Parameters
    ----------
    Z_windows : ndarray, shape (n_rep, STEPS_WIN+1, n)
                Z_windows[r, k, :] = z_k (0-based k in window)
    i_fixed   : int, 1-based mutation index (center = WINDOW+1)

    Returns
    -------
    E : ndarray, shape (n_rep, STEPS_WIN)
        E[r, j-1] = E_{i_fixed, j}, j=1..STEPS_WIN (1-based)
    """
    n_rep, steps_plus1, dim = Z_windows.shape
    steps = steps_plus1 - 1   # = STEPS_WIN = 2*WINDOW

    if i_fixed < 1 or i_fixed > steps:
        raise ValueError(f"i_fixed={i_fixed} must be in [1, {steps}]")

    # Mutational vector x_i = z_i - z_{i-1} (0-based in window)
    x_i = Z_windows[:, i_fixed, :] - Z_windows[:, i_fixed - 1, :]
    # shape: (n_rep, dim)

    E = np.zeros((n_rep, steps))

    for j in range(1, steps + 1):
        if j == i_fixed:
            # E_{i,i} = 0 by definition — leave zeros
            continue
        elif i_fixed >= j:
            # Contingency: <x_i, z_{i-1} - z_{j-1}>
            delta = (Z_windows[:, i_fixed - 1, :]
                     - Z_windows[:, j - 1, :])
        else:
            # Entrenchment: <x_i, z_i - z_j>
            delta = Z_windows[:, i_fixed, :] - Z_windows[:, j, :]

        E[:, j - 1] = np.einsum('rn,rn->r', x_i, delta)

    return E


def compute_mean_si(Z_windows, i_fixed):
    """
    Compute mean log-fitness gain of substitution i_fixed across replicates.

    s_i = w(z_i) - w(z_{i-1}) = (||z_{i-1}||^2 - ||z_i||^2) / 2 > 0

    Used to normalize E_{i,j} for comparability across panels.

    Parameters
    ----------
    Z_windows : ndarray, shape (n_rep, STEPS_WIN+1, n)
    i_fixed   : int, 1-based

    Returns
    -------
    mean_si : float, mean fitness gain at step i_fixed
    """
    norm2_before = np.sum(Z_windows[:, i_fixed - 1, :]**2, axis=1)
    norm2_after  = np.sum(Z_windows[:, i_fixed,     :]**2, axis=1)
    s_i = (norm2_before - norm2_after) / 2.0   # > 0 for beneficial
    return np.mean(s_i)


# ── Accumulate valid windows batch by batch ───────────────────────────────────

def run_and_extract(n, d_target, sti, window, steps_win, Ne,
                    eps_rel, min_valid, batch=BATCH):
    """
    Simulate adaptive walks in batches and accumulate valid windows until
    min_valid is reached.

    d_start and steps_long are read from D_START_5/D_START_50 and
    STEPS_LONG — fixed manually, never changed between batches.
    All valid windows from previous batches are kept.

    A window is valid if:
      - i* (step closest to d_target) lies in valid_range, i.e. has
        at least `window` steps on each side within the simulated walk
      - |d_{i*} - d_target| < eps_rel * d_target

    The extracted window has steps_win + 1 = 2*window + 1 phenotype
    vectors: z_{i*-window}, ..., z_{i*}, ..., z_{i*+window}.

    Parameters
    ----------
    n         : phenotypic dimensionality
    d_target  : target distance to optimum at center step i*
    sti       : fixed typical mutational norm (sigma = sti/sqrt(n))
    window    : half-window size (steps before and after i*)
    steps_win : total steps in extracted window = 2*window
    Ne        : effective population size
    eps_rel   : relative tolerance for distance matching
    min_valid : stop when this many valid windows accumulated
    batch     : replicates per simulate_adaptive_walk call

    Returns
    -------
    Z_windows : ndarray, shape (min_valid, steps_win+1, n)
    d_center  : ndarray, shape (min_valid,)
    """
    sigma      = sti / np.sqrt(n)
    eps        = eps_rel * d_target
    if n==5:
        d_start    = D_START_5[d_target]
    elif n==50:
        d_start    = D_START_50[d_target]
    elif n==10:
        d_start    = D_START_10[d_target]
    elif n==100:
        d_start    = D_START_100[d_target]
    steps_long = STEPS_LONG[d_target]

    # Sanity checks
    if steps_long < steps_win + 1:
        raise ValueError(
            f"steps_long={steps_long} too short: need >= {steps_win+1}."
        )

    # valid_range: 0-based step indices with window on each side
    valid_range = np.arange(window, steps_long - window + 1)

    # Accumulators
    Z_windows_list = []
    d_center_list  = []
    n_valid        = 0
    n_batches      = 0
    n_simulated    = 0

    print(f"\n  n={n}, d_target={d_target:.4f}, "
          f"d_start={d_start:.4f}, steps_long={steps_long}, "
          f"window={window}, eps={eps:.5f}",
          flush=True)

    while n_valid < min_valid:

        n_batches   += 1
        n_simulated += batch

        # Simulate one batch of long walks
        # Z shape: (batch, steps_long+1, n)
        Z = simulate_adaptive_walk(
            n               = n,
            sigma_m         = sigma,
            Ne              = Ne,
            steps           = steps_long,
            replicates      = batch,
            initial_size    = d_start,
            kimura          = True,
            only_beneficial = True,
            exact           = True,
            verbose         = False,
        )

        # Distance to optimum: shape (batch, steps_long+1)
        d_traj = np.linalg.norm(Z, axis=2)

        # Find i* = step closest to d_target within valid_range
        d_in_range   = d_traj[:, valid_range]
        i_star_local = np.argmin(np.abs(d_in_range - d_target), axis=1)
        i_star       = valid_range[i_star_local]   # 0-based in full traj

        # Distance at i* for each replicate
        d_at_istar = d_traj[np.arange(batch), i_star]

        # Validity: distance close enough to target
        ok   = np.abs(d_at_istar - d_target) < eps
        n_ok = ok.sum()

        # Diagnostic
        rate = n_ok / batch
        if rate < 0.05:
            print(
                f"    WARNING batch {n_batches}: rate={rate:.1%}, "
                f"d_at_istar mean={d_at_istar.mean():.4f} "
                f"(target={d_target:.4f}). "
                f"Adjust D_START or EPS_REL.",
                flush=True
            )

        # Extract and accumulate valid windows
        idx_ok    = np.where(ok)[0]
        i_star_ok = i_star[ok]

        for k, r in enumerate(idx_ok):
            if n_valid >= min_valid:
                break
            i0 = i_star_ok[k] - window     # 0-based start of window
            i1 = i_star_ok[k] + window + 1  # 0-based end of window
            Z_windows_list.append(Z[r, i0:i1, :].copy())
            d_center_list.append(d_at_istar[idx_ok[k]])
            n_valid += 1

        print(
            f"    Batch {n_batches:4d}: "
            f"{n_ok:3d}/{batch} valid "
            f"(rate={rate:.0%}), "
            f"total={n_valid}/{min_valid}",
            flush=True
        )

    # Stack into arrays
    Z_windows = np.stack(Z_windows_list, axis=0)
    d_center  = np.array(d_center_list)

    print(
        f"  Done: {n_batches} batches, {n_simulated} walks simulated.\n"
        f"  d_center: mean={d_center.mean():.5f} "
        f"+/- {d_center.std():.5f} "
        f"(target={d_target:.5f})",
        flush=True
    )

    return Z_windows, d_center




def epistasis_regime_I_normalized(x, sigma, d, mean_si, sti=STI):
    """
    Regime I prediction for E[E_{i,i+/-1}] / mean_si.
    Both numerator and denominator are in log-fitness units.

    E[E_{i,i+1}] ~ sigma^2 * M(x) * (2x - M(x))
    normalized by empirical mean_si (mean log-fitness gain).
    """
    lam = M_fixed(x)
    return (sigma**2 * lam * (2*x - lam)) / mean_si


def epistasis_regime_II_normalized(y, mean_si, sti=STI):
    """
    Regime II prediction for E[E_{i,i+/-1}] / mean_si.
    Both in log-fitness units.

    E[E_{i,i+1}] ~ (sti^2 / 4y^2) * (1 + 2y^2 - sqrt(1+4y^4))^2
    normalized by empirical mean_si.
    """
    term = 1.0 + 2.0*y**2 - np.sqrt(1.0 + 4.0*y**4)
    return (sti**2 / (4.0*y**2)) * term**2 / mean_si


def plot_Eij_panel(ax, E, i_fixed, window, mean_si,
                   x_val, y_val, sigma,
                   label_x=True, label_y=True):
    """
    Plot mean +/- 2 SE of E_{i_fixed, j} / mean_si for all j.

    Overlaid analytical predictions as thin horizontal dashed lines:
      - Regime I  (always shown): blue dashed for contingency side,
                                  red dashed for entrenchment side
      - Regime II (shown if y > 0.3): same colors, dotted

    Both E_{i,j} and mean_si are in log-fitness units, so the ratio
    is dimensionless. Y-axis label reflects this:
    mean log-fitness epistasis / mean log-fitness gain.

    Parameters
    ----------
    ax      : matplotlib Axes
    E       : ndarray, shape (n_rep, 2*window)
    i_fixed : int, 1-based center index
    window  : int
    mean_si : float, mean log-fitness gain at step i_fixed
    x_val   : float, dimensionless Fisher parameter x = sti*sqrt(n)/(2d)
    y_val   : float, y = sti/(2d)
    sigma   : float, per-axis mutational std
    label_x : bool
    label_y : bool
    """
    n_rep, steps = E.shape

    j_vals = np.arange(1, steps + 1)
    x_vals = j_vals - i_fixed

    # Normalize E by mean log-fitness gain
    E_norm = E / mean_si
    mean_E = np.mean(E_norm, axis=0)
    se_E   = np.std(E_norm, axis=0, ddof=1) / np.sqrt(n_rep)

    mask_blue = j_vals < i_fixed
    mask_red  = j_vals > i_fixed

    # ── Contingency (blue) ────────────────────────────────────────────────────
    ax.errorbar(
        x_vals[mask_blue], mean_E[mask_blue],
        yerr=2 * se_E[mask_blue],
        fmt='o-', color=COLOUR_CONT, ecolor='lightblue',
        elinewidth=2, capsize=3, markersize=4, linewidth=1.5,
        label=r'$j < i$ (contingency)', zorder=3,
    )

    # ── Entrenchment (red) ────────────────────────────────────────────────────
    ax.errorbar(
        x_vals[mask_red], mean_E[mask_red],
        yerr=2 * se_E[mask_red],
        fmt='o-', color=COLOUR_ENTR, ecolor='salmon',
        elinewidth=2, capsize=3, markersize=4, linewidth=1.5,
        label=r'$j > i$ (entrenchment)', zorder=3,
    )

    # ── Center point (black) ──────────────────────────────────────────────────
    ax.plot(0, 0, 'o', color=COLOUR_CENTER, markersize=5, zorder=5,
            label=r'$j = i$')

    # ── Connecting lines through center ───────────────────────────────────────
    ax.plot([x_vals[mask_blue][-1], 0],
            [mean_E[mask_blue][-1], 0],
            color=COLOUR_CONT, linewidth=1.5, zorder=2)
    ax.plot([0, x_vals[mask_red][0]],
            [0, mean_E[mask_red][0]],
            color=COLOUR_ENTR, linewidth=1.5, zorder=2)

    # ── Analytical predictions ────────────────────────────────────────────────
    # Regime I prediction (always shown)
    pred_I = epistasis_regime_I_normalized(x_val, sigma, 
                                           STI/(2*y_val), mean_si)

    # Horizontal dashed lines over contingency and entrenchment ranges
    x_blue_range = [x_vals[mask_blue][0],  x_vals[mask_blue][-1]+1]
    x_red_range  = [x_vals[mask_red][0],   x_vals[mask_red][-1]+1]

    if abs(pred_I) < 6: # only plot if not too large (avoid clutter)
        #ax.hlines(pred_I, x_blue_range[0], x_blue_range[1],colors=COLOUR_CONT, linewidths=1.2, linestyles='--',label=r'Regime I pred.', zorder=4, alpha=0.8)
        ax.hlines(pred_I, x_red_range[0], x_red_range[1],
                colors=COLOUR_ENTR, linewidths=0.8, linestyles='--',
                zorder=1, alpha=0.9)

        # Label at the right end of the line
        #ax.text(x_red_range[1]-2.5, pred_I,"Regime I", fontsize=5, color=COLOUR_ENTR,va='center', ha='left')

    # Regime II prediction (only if y > 0.3)
    if y_val > 0.3:
        pred_II = epistasis_regime_II_normalized(y_val, mean_si)

        #ax.hlines(pred_II, x_blue_range[0], x_blue_range[1], colors=COLOUR_CONT, linewidths=1.2, linestyles=':',label=r'Regime II pred.', zorder=4, alpha=0.8)
        ax.hlines(pred_II, x_red_range[0], x_red_range[1],
                  colors=COLOUR_ENTR, linewidths=0.8, linestyles=':',
                  zorder=1, alpha=0.9)
                # Label at the right end of the line
        #ax.text(x_red_range[1] + 0.3, pred_II,"Regime II", fontsize=7, color=COLOUR_ENTR,va='center', ha='left')

    # ── Reference lines ───────────────────────────────────────────────────────
    ax.axhline(0.0, color='black', lw=0.8, zorder=1)
    ax.axvline(0.0, color='black', lw=0.8, ls='--', alpha=0.5, zorder=1)
    ax.grid(True, alpha=0.3)

    # ── Axis labels ───────────────────────────────────────────────────────────
    if label_x:
        ax.set_xlabel(r"Step $j - i$", labelpad=4)
    if label_y:
        ax.set_ylabel(
            r"$\mathcal{E}_{i,j}\,/\,\mathbb{E}[\Delta_i]$",
            labelpad=4)

    ax.set_xlim(-window - 0.5, window + 0.5)
    #ax.legend(fontsize=7, loc='upper left')

# ── Save figure ───────────────────────────────────────────────────────────────

def save_fig(fig, n, d_target, sti):
    """Save as PDF and TIFF. Filename encodes all parameters."""
    fname = (f"fig3_Eij"
             f"_n{n}"
             f"_d{str(d_target).replace('.', 'p')}"
             f"_sti{str(sti).replace('.', 'p')}")
    fig.savefig(fname + ".pdf",  format="pdf",  bbox_inches="tight")
    fig.savefig(fname + ".tiff", format="tiff", dpi=1200,
                bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {fname}.pdf / .tiff")


# ── Main loop ─────────────────────────────────────────────────────────────────
for n in N_DIMS:
    for d_target in D_TARGETS:

        sigma = STI / np.sqrt(n)
        y_val = STI / (2 * d_target)
        x_val = STI * np.sqrt(n) / (2 * d_target)

        print(f"\n{'='*60}", flush=True)
        print(f"n={n}, d_target={d_target}, "
              f"y={y_val:.3f}, x={x_val:.3f}",
              flush=True)

        # ── Accumulate valid windows ──────────────────────────────────────────
        Z_windows, d_center = run_and_extract(
            n         = n,
            d_target  = d_target,
            sti       = STI,
            window    = WINDOW,
            steps_win = STEPS_WIN,
            Ne        = NE,
            eps_rel   = EPS_REL,
            min_valid = MIN_VALID,
            batch     = BATCH,
        )

        # ── Compute E_{I_FIXED, j} and normalize by mean fitness gain ─────────
        E       = compute_Eij_from_window(Z_windows, i_fixed=I_FIXED)
        mean_si = compute_mean_si(Z_windows, i_fixed=I_FIXED)

        print(f"  mean_si = {mean_si:.6f}", flush=True)

        # ── Plot ──────────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(90 * ONE_MM, 80 * ONE_MM))

        plot_Eij_panel(
            ax      = ax,
            E       = E,
            i_fixed = I_FIXED,
            window  = WINDOW,
            mean_si = mean_si,
            x_val   = x_val,
            y_val   = y_val,
            sigma   = sigma,
            label_x = True,
            label_y = True,
        )

        # Parameter annotation
        if False:
            ax.text(
                0.97, 0.03,
                rf"$n={n}$, $d \approx {d_target}$, $\tilde{{\sigma}}={STI}$"
                "\n"
                rf"$y={y_val:.2f}$, $x={x_val:.2f}$"
                "\n"
                rf"$d_i$: {d_center.mean():.4f} $\pm$ {d_center.std():.4f}"
                "\n"
                rf"$\mathbb{{E}}[s_i] = {mean_si:.5f}$",
                transform=ax.transAxes, fontsize=7,
                ha='right', va='bottom',
                bbox=dict(fc='white', ec='lightgray', alpha=0.8, pad=3),
            )

        save_fig(fig, n, d_target, STI)

print("\nAll figures saved.")

In [ ]:
STI       = 0.1
N_DIMS    = [5,10, 100]
D_TARGETS = [5,0.5, 0.05]

for n in N_DIMS:
    for d in D_TARGETS:
        sigma = STI / np.sqrt(n)
        y_val = STI / (2 * d)
        x_val = STI * np.sqrt(n) / (2 * d)
        print(f"n={n}, d={d}, y={y_val:.3f}, x={x_val:.3f}")

In [ ]:
STI       = 0.1
N_DIMS    = [5,10, 100]
Ne=10**5


for n in N_DIMS:
    d_eq=np.sqrt(n / (4.0 * Ne))
    sigma = STI / np.sqrt(n)
    y_val = STI / (2 * d_eq)
    x_val = STI * np.sqrt(n) / (2 * d_eq)
    print(f"n={n}, d={d_eq}, y={y_val:.3f}, x={x_val:.3f}")

# VI Contingency and Entrenchment at MSDE

In [ ]:
"""
Figure 4 — Contingency and entrenchment at mutation-selection-drift equilibrium (MSDE).

Three panels: n = 5, 10, 100, with sti = 0.1 fixed.
Starting phenotype: random vector renormalized to d_eq = sqrt(n / Ne).
Walk length: 35 steps. Focal substitution: step i = 18 (center).
Y-axis: Ne * E_{i,j} (dimensionless, comparable to Shah et al. 2015).

Convention: sigma = sti / sqrt(n), Ne fixed, only_beneficial = False,
kimura = True (fixation probability p(s) used).
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":      "serif",
    "mathtext.fontset": "dejavuserif",
    "font.size":        11,
    "axes.labelsize":   12,
    "axes.linewidth":   0.8,
    "xtick.direction":  "in",
    "ytick.direction":  "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "legend.frameon":   False,
    "legend.fontsize":  9,
})

ONE_MM = 1 / 25.4

# ── Parameters ────────────────────────────────────────────────────────────────
STI      = 0.1
N_DIMS   = [5, 10,20,50, 100]
NE       = 10**5
STEPS    = 35           # total walk length
WINDOW   = 17           # steps before and after focal substitution
I_FIXED  = WINDOW + 1   # = 18, 1-based center
N_REP    = 10          # replicates per n
BATCH    = 5           # replicates per simulation batch

COLOUR_CONT   = "#0072b5"
COLOUR_ENTR   = "#bc3c29"
COLOUR_CENTER = "black"


# ── Equilibrium distance ──────────────────────────────────────────────────────

def d_eq(n, Ne):
    """
    Mean distance to optimum at MSDE: d_eq = sqrt(n / (2*Ne)).
    From Tenaillon et al. 2007: E[||z||^2] = n / (2*Ne) at stationarity
    under Gaussian fitness W(z) = exp(-||z||^2/2).
    """
    return np.sqrt(n / 2*Ne)


# ── Random initial phenotype at d_eq ─────────────────────────────────────────

def random_phenotype_at_deq(n, Ne, rng):
    """
    Draw a random unit vector in R^n and rescale to d_eq = sqrt(n/2Ne).
    This samples uniformly on the sphere of radius d_eq.
    """
    z = rng.standard_normal(n)
    z =(z / np.linalg.norm(z)) * d_eq(n, Ne)
    return z


# ── Compute E_{i_fixed, j} from window ───────────────────────────────────────

def compute_Eij_from_window(Z_windows, i_fixed):
    """
    Compute E_{i_fixed, j} for all j = 1..steps from Z_windows.

    Mathematical definition:
        E_{i,j} = <x_i, z_{i-1} - z_{j-1}>  if i >= j  (contingency)
        E_{i,j} = <x_i, z_i    - z_j   >    if i <  j  (entrenchment)
        E_{i,i} = 0  by definition

    Parameters
    ----------
    Z_windows : ndarray, shape (n_rep, 2*WINDOW+1, n)
    i_fixed   : int, 1-based

    Returns
    -------
    E : ndarray, shape (n_rep, 2*WINDOW)
    """
    n_rep, steps_plus1, dim = Z_windows.shape
    steps = steps_plus1 - 1

    x_i = Z_windows[:, i_fixed, :] - Z_windows[:, i_fixed - 1, :]
    E   = np.zeros((n_rep, steps))

    for j in range(1, steps + 1):
        if j == i_fixed:
            continue
        elif i_fixed >= j:
            delta = (Z_windows[:, i_fixed - 1, :]
                     - Z_windows[:, j - 1, :])
        else:
            delta = Z_windows[:, i_fixed, :] - Z_windows[:, j, :]
        E[:, j - 1] = np.einsum('rn,rn->r', x_i, delta)

    return E


# ── Simulate MSDE walks and collect windows ───────────────────────────────────

def collect_msde_windows(n, sti, Ne, steps, window, i_fixed,
                          n_rep, batch):
    """
    Simulate MSDE adaptive walks in batches and collect n_rep windows
    of length 2*window + 1 centered on step i_fixed.

    Starting phenotype: random vector on sphere of radius d_eq = sqrt(n/Ne).
    Uses kimura=True and only_beneficial=False (MSDE regime).

    Parameters
    ----------
    n        : phenotypic dimensionality
    sti      : fixed typical mutational norm
    Ne       : effective population size
    steps    : total walk length (must be >= 2*window + 1)
    window   : half-window size
    i_fixed  : 1-based center step in extracted window
    n_rep    : number of windows to collect
    batch    : replicates per simulate_adaptive_walk call

    Returns
    -------
    Z_windows : ndarray, shape (n_rep, 2*window+1, n)
    d_center  : ndarray, shape (n_rep,) — distance at focal step
    """
    sigma  = sti / np.sqrt(n)
    d_init = d_eq(n, Ne)
    rng    = np.random.default_rng(42 + n)

    if steps < 2 * window + 1:
        raise ValueError(
            f"steps={steps} too short for window={window}. "
            f"Need >= {2*window+1}."
        )

    Z_windows_list = []
    d_center_list  = []
    n_collected    = 0
    n_batches      = 0

    print(f"\n  n={n}, d_eq={d_init:.5f}, sigma={sigma:.5f}", flush=True)

    while n_collected < n_rep:
        n_batches += 1

        # Generate random starting phenotypes on sphere of radius d_eq
        # by passing initial_size = d_init to simulate_adaptive_walk
        # (z_0 = (d_init, 0, ..., 0) by convention in simulate_adaptive_walk)
        # For MSDE we want a random direction — we run from (d_init, 0,...,0)
        # which is equivalent up to rotation invariance of the model.
        Z = simulate_adaptive_walk(
            n               = n,
            sigma_m         = sigma,
            Ne              = Ne,
            steps           = steps,
            replicates      = batch,
            initial_size    = d_init,
            kimura          = True,
            only_beneficial = False,   # MSDE: deleterious mutations can fix
            exact           = True,
            verbose         = False,
        )
        # Z shape: (batch, steps+1, n)

        # Extract window centered on i_fixed (0-based: i_fixed-1 in Z)
        # i_fixed is 1-based in the window, but here we take the full walk
        # and extract [i_fixed-1-window : i_fixed-1+window+1] (0-based in Z)
        i0 = i_fixed - 1 - window     # 0-based start in Z (step index)
        i1 = i_fixed - 1 + window + 1  # 0-based end in Z

        if i0 < 0 or i1 > steps + 1:
            raise ValueError(
                f"Window [{i0}, {i1}] out of bounds for steps={steps}. "
                f"Increase steps or decrease window."
            )

        # All replicates are valid (no filtering needed — MSDE is stationary)
        for r in range(batch):
            if n_collected >= n_rep:
                break
            Z_windows_list.append(Z[r, i0:i1, :].copy())
            d_center_list.append(
                np.linalg.norm(Z[r, i_fixed - 1, :])
            )
            n_collected += 1

        print(
            f"    Batch {n_batches:3d}: collected {n_collected}/{n_rep}",
            flush=True
        )

    Z_windows = np.stack(Z_windows_list, axis=0)
    d_center  = np.array(d_center_list)

    print(
        f"  d_center: mean={d_center.mean():.5f} "
        f"+/- {d_center.std():.5f} "
        f"(d_eq={d_init:.5f})",
        flush=True
    )

    return Z_windows, d_center


# ── Plot function ─────────────────────────────────────────────────────────────

def plot_msde_panel(ax, E, i_fixed, window, Ne,
                    label_x=True, label_y=True):
    """
    Plot mean +/- 2 SE of Ne * E_{i_fixed, j} for all j.

    Blue  (#0072b5): j < i_fixed (contingency)
    Red   (#bc3c29): j > i_fixed (entrenchment)
    Black dot at x=0: center point (E=0 by definition)

    Y-axis: Ne * E_{i,j} (dimensionless, as in Shah et al. 2015).

    Parameters
    ----------
    ax      : matplotlib Axes
    E       : ndarray, shape (n_rep, 2*window)
    i_fixed : int, 1-based
    window  : int
    Ne      : float, effective population size (for scaling)
    label_x : bool
    label_y : bool
    """
    n_rep, steps = E.shape

    j_vals = np.arange(1, steps + 1)
    x_vals = j_vals - i_fixed

    # Scale by Ne
    E_scaled = NE * E
    mean_E   = np.mean(E_scaled, axis=0)
    se_E     = np.std(E_scaled, axis=0, ddof=1) / np.sqrt(n_rep)

    mask_blue = j_vals < i_fixed
    mask_red  = j_vals > i_fixed

    # Contingency (blue)
    ax.errorbar(
        x_vals[mask_blue], mean_E[mask_blue],
        yerr=2 * se_E[mask_blue],
        fmt='o-', color=COLOUR_CONT, ecolor='lightblue',
        elinewidth=2, capsize=3, markersize=4, linewidth=1.5,
        label=r'$j < i$ (contingency)', zorder=3,
    )

    # Entrenchment (red)
    ax.errorbar(
        x_vals[mask_red], mean_E[mask_red],
        yerr=2 * se_E[mask_red],
        fmt='o-', color=COLOUR_ENTR, ecolor='salmon',
        elinewidth=2, capsize=3, markersize=4, linewidth=1.5,
        label=r'$j > i$ (entrenchment)', zorder=3,
    )

    # Center point (black)
    ax.plot(0, 0, 'o', color=COLOUR_CENTER, markersize=5, zorder=5,
            label=r'$j = i$')

    # Connecting lines through center
    ax.plot([x_vals[mask_blue][-1], 0],
            [mean_E[mask_blue][-1], 0],
            color=COLOUR_CONT, linewidth=1.5, zorder=2)
    ax.plot([0, x_vals[mask_red][0]],
            [0, mean_E[mask_red][0]],
            color=COLOUR_ENTR, linewidth=1.5, zorder=2)

    # Reference lines
    ax.axhline(0.0, color='black', lw=0.8, zorder=1)
    ax.axvline(0.0, color='black', lw=0.8, ls='--', alpha=0.5, zorder=1)
    ax.grid(True, alpha=0.3)

    if label_x:
        ax.set_xlabel(r"Step $j - i$", labelpad=4)
    if label_y:
        ax.set_ylabel(
            r"$N_e\,\mathcal{E}_{i,j}$",
            labelpad=4)

    ax.set_xlim(-window - 0.5, window + 0.5)
    ax.legend(fontsize=8, loc='upper left')


# ── Save figure ───────────────────────────────────────────────────────────────

def save_fig(fig, n, sti, Ne):
    """Save as PDF and TIFF."""
    fname = (f"fig4_MSDE_Eij"
             f"_n{n}"
             f"_sti{str(sti).replace('.', 'p')}"
             f"_Ne{int(Ne)}")
    fig.savefig(fname + ".pdf",  format="pdf",  bbox_inches="tight")
    fig.savefig(fname + ".tiff", format="tiff", dpi=1200,
                bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {fname}.pdf / .tiff")


# ── Main: one figure per n ────────────────────────────────────────────────────
for n in N_DIMS:

    d_eq_n = d_eq(n, NE)
    sigma  = STI / np.sqrt(n)
    y_val  = STI / (2 * d_eq_n)
    x_val  = STI * np.sqrt(n) / (2 * d_eq_n)

    print(f"\n{'='*60}")
    print(f"n={n}, d_eq={d_eq_n:.5f}, sigma={sigma:.5f}, "
          f"y={y_val:.3f}, x={x_val:.3f}",
          flush=True)

    # Collect MSDE windows
    Z_windows, d_center = collect_msde_windows(
        n       = n,
        sti     = STI,
        Ne      = NE,
        steps   = STEPS,
        window  = WINDOW,
        i_fixed = I_FIXED,
        n_rep   = N_REP,
        batch   = BATCH,
    )

    # Compute E_{I_FIXED, j}
    E = compute_Eij_from_window(Z_windows, i_fixed=I_FIXED)

    # Plot
    fig, ax = plt.subplots(figsize=(90 * ONE_MM, 80 * ONE_MM))

    plot_msde_panel(
        ax      = ax,
        E       = E,
        i_fixed = I_FIXED,
        window  = WINDOW,
        Ne      = NE,
        label_x = True,
        label_y = True,
    )

    # Parameter annotation
    ax.text(
        0.97, 0.03,
        rf"$n={n}$, $\tilde{{\sigma}}={STI}$, $N_e={NE:.0e}$"
        "\n"
        rf"$d_{{eq}}={d_eq_n:.4f}$"
        "\n"
        rf"$y={y_val:.2f}$, $x={x_val:.2f}$"
        "\n"
        rf"$d_i$: {d_center.mean():.4f} $\pm$ {d_center.std():.4f}",
        transform=ax.transAxes, fontsize=7,
        ha='right', va='bottom',
        bbox=dict(fc='white', ec='lightgray', alpha=0.8, pad=3),
    )

    save_fig(fig, n, STI, NE)

print("\nAll MSDE figures saved.")

In [ ]:
"""
Figure 4 — Contingency and entrenchment at mutation-selection-drift equilibrium (MSDE).

Single figure with three curves: n = 5, 10, 100.
Y-axis: Ne * E_{i,j} (dimensionless).
Shaded bands: +/- 2 SE across replicates.
Walk length: 100 steps. Focal substitution: step i = 51 (center).
Window: 50 steps before and after focal substitution.

Convention: sigma = sti/sqrt(n), kimura=True, only_beneficial=False.
Starting phenotype: z_0 = (d_eq, 0, ..., 0), d_eq = sqrt(n/Ne).
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Global style ──────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family":      "serif",
    "mathtext.fontset": "dejavuserif",
    "font.size":        11,
    "axes.labelsize":   12,
    "axes.linewidth":   0.8,
    "xtick.direction":  "in",
    "ytick.direction":  "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "legend.frameon":   False,
    "legend.fontsize":  9,
})

ONE_MM = 1 / 25.4

# ── Parameters ────────────────────────────────────────────────────────────────
STI    = 0.1
NE     = 10**5
STEPS  = 38          # total walk length
WINDOW = 15          # steps before and after focal step
I_FIXED = WINDOW + 1  # = 51, 1-based center
N_REP  = 50          # replicates per n
BATCH  = 5

N_DIMS = [5, 10, 50, 100]

# Color palette
COLOURS = {
    5:   "#bc3c29",
    10:"#e18727",
    50:  "#0072b5",
    100: "#20854e",
}


# ── Equilibrium distance ──────────────────────────────────────────────────────

def d_eq(n, Ne):
    """Mean distance to optimum at MSDE: sqrt(n/2Ne)."""
    return np.sqrt(n / (2*Ne))


# ── Compute E_{i_fixed, j} from window ───────────────────────────────────────

def compute_Eij_from_window(Z_windows, i_fixed):
    """
    Compute E_{i_fixed, j} for all j = 1..steps from Z_windows.

        E_{i,j} = <x_i, z_{i-1} - z_{j-1}>  if i >= j
        E_{i,j} = <x_i, z_i    - z_j   >    if i <  j
        E_{i,i} = 0

    Parameters
    ----------
    Z_windows : ndarray, shape (n_rep, 2*WINDOW+1, n)
    i_fixed   : int, 1-based

    Returns
    -------
    E : ndarray, shape (n_rep, 2*WINDOW)
    """
    n_rep, steps_plus1, dim = Z_windows.shape
    steps = steps_plus1 - 1

    x_i = Z_windows[:, i_fixed, :] - Z_windows[:, i_fixed - 1, :]
    E   = np.zeros((n_rep, steps))

    for j in range(1, steps + 1):
        if j == i_fixed:
            continue
        elif i_fixed >= j:
            delta = (Z_windows[:, i_fixed - 1, :]
                     - Z_windows[:, j - 1, :])
        else:
            delta = Z_windows[:, i_fixed, :] - Z_windows[:, j, :]
        E[:, j - 1] = np.einsum('rn,rn->r', x_i, delta)

    return E


# ── Collect MSDE windows ──────────────────────────────────────────────────────

def collect_msde_windows(n, sti, Ne, steps, window, i_fixed, n_rep, batch):
    """
    Simulate MSDE walks in batches and collect n_rep windows of length
    2*window+1 centered on step i_fixed.

    No filtering: all replicates are valid (stationary regime).

    Parameters
    ----------
    n       : phenotypic dimensionality
    sti     : fixed typical mutational norm
    Ne      : effective population size
    steps   : total walk length
    window  : half-window size
    i_fixed : 1-based center step in extracted window
    n_rep   : windows to collect
    batch   : replicates per simulation call

    Returns
    -------
    Z_windows : ndarray, shape (n_rep, 2*window+1, n)
    d_center  : ndarray, shape (n_rep,)
    """
    sigma  = sti / np.sqrt(n)
    d_init = d_eq(n, Ne)

    # 0-based window boundaries in full Z trajectory
    i0 = i_fixed - 1 - window
    i1 = i_fixed - 1 + window + 1

    if i0 < 0 or i1 > steps + 1:
        raise ValueError(
            f"Window [{i0},{i1}] out of bounds for steps={steps}."
        )

    Z_windows_list = []
    d_center_list  = []
    n_collected    = 0
    n_batches      = 0

    print(f"  n={n:3d}, d_eq={d_init:.6f}, sigma={sigma:.6f}",
          flush=True)

    while n_collected < n_rep:
        n_batches += 1

        Z = simulate_adaptive_walk(
            n               = n,
            sigma_m         = sigma,
            Ne              = Ne,
            steps           = steps,
            replicates      = batch,
            initial_size    = d_init,
            kimura          = True,
            only_beneficial = False,   # MSDE: deleterious mutations can fix
            exact           = True,
            verbose         = False,
        )
        # Z shape: (batch, steps+1, n)

        for r in range(batch):
            if n_collected >= n_rep:
                break
            Z_windows_list.append(Z[r, i0:i1, :].copy())
            d_center_list.append(np.linalg.norm(Z[r, i_fixed - 1, :]))
            n_collected += 1

        print(f"    Batch {n_batches:3d}: {n_collected}/{n_rep}",
              flush=True)

    Z_windows = np.stack(Z_windows_list, axis=0)
    d_center  = np.array(d_center_list)

    print(
        f"  d_center: mean={d_center.mean():.5f} "
        f"+/- {d_center.std():.5f} "
        f"(d_eq={d_init:.5f})",
        flush=True
    )
    return Z_windows, d_center


# ── Main: simulate all n ──────────────────────────────────────────────────────
results = {}   # results[n] = (x_vals, mean_E_scaled, se_E_scaled)

for n in N_DIMS:
    print(f"\n{'='*60}\nn={n}", flush=True)

    Z_windows, d_center = collect_msde_windows(
        n       = n,
        sti     = STI,
        Ne      = NE,
        steps   = STEPS,
        window  = WINDOW,
        i_fixed = I_FIXED,
        n_rep   = N_REP,
        batch   = BATCH,
    )

    E = compute_Eij_from_window(Z_windows, i_fixed=I_FIXED)

    # Scale by Ne and compute stats
    E_scaled = NE * E                                      # (n_rep, 2*WINDOW)
    mean_E   = np.mean(E_scaled, axis=0)
    se_E     = np.std(E_scaled, axis=0, ddof=1) / np.sqrt(N_REP)

    # x-axis: relative step j - i
    j_vals = np.arange(1, 2 * WINDOW + 1)
    x_vals = j_vals - I_FIXED

    results[n] = (x_vals, mean_E, se_E)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(130 * ONE_MM, 95 * ONE_MM))

for n in N_DIMS:
    x_vals, mean_E, se_E = results[n]
    col = COLOURS[n]

    j_vals    = np.arange(1, 2 * WINDOW + 1)
    mask_blue = j_vals < I_FIXED
    mask_red  = j_vals > I_FIXED
    x_blue    = x_vals[mask_blue]
    x_red     = x_vals[mask_red]

    # ── Contingency (left, negative x) ────────────────────────────────────────
    ax.plot(x_blue, mean_E[mask_blue],
            color=col, lw=1.5, zorder=3)
    ax.fill_between(x_blue,
                    mean_E[mask_blue] - 2 * se_E[mask_blue],
                    mean_E[mask_blue] + 2 * se_E[mask_blue],
                    color=col, alpha=0.20, zorder=2)

    # ── Entrenchment (right, positive x) ─────────────────────────────────────
    ax.plot(x_red, mean_E[mask_red],
            color=col, lw=1.5, zorder=3,
            label=rf'$n = {n}$')
    ax.fill_between(x_red,
                    mean_E[mask_red] - 2 * se_E[mask_red],
                    mean_E[mask_red] + 2 * se_E[mask_red],
                    color=col, alpha=0.20, zorder=2)

    # ── Connecting line through center (0, 0) ─────────────────────────────────
    ax.plot([x_blue[-1], 0], [mean_E[mask_blue][-1], 0],
            color=col, lw=1.5, zorder=3)
    ax.plot([0, x_red[0]],  [0, mean_E[mask_red][0]],
            color=col, lw=1.5, zorder=3)

# ── Center point ──────────────────────────────────────────────────────────────
ax.plot(0, 0, 'o', color='black', markersize=5, zorder=5)

# ── Reference lines ───────────────────────────────────────────────────────────
ax.axhline(0.0, color='black', lw=0.8, zorder=1)
ax.axvline(0.0, color='black', lw=0.8, ls='--', alpha=0.5, zorder=1)
ax.grid(True, alpha=0.3)

# ── Labels and legend ─────────────────────────────────────────────────────────
ax.set_xlabel(r"Step $j - i$", labelpad=4)
ax.set_ylabel(r"$N_e\,\mathcal{E}_{i,j}$", labelpad=4)
ax.set_xlim(-WINDOW - 0.5, WINDOW + 0.5)
ax.legend(loc='upper left', fontsize=9,
          title=rf"$\tilde{{\sigma}}={STI}$, $N_e={NE:.0e}$",
          title_fontsize=8)


# ── Save ──────────────────────────────────────────────────────────────────────
fname = f"fig4_MSDE_NeEij_sti{str(STI).replace('.','p')}_Ne{int(NE)}"
fig.savefig(fname + ".pdf",  format="pdf",  bbox_inches="tight")
fig.savefig(fname + ".tiff", format="tiff", dpi=1200, bbox_inches="tight")
plt.close(fig)

print(f"\nSaved: {fname}.pdf / .tiff")

In [ ]:
"""
Figure — Contingency and entrenchment at mutation-selection-drift equilibrium (MSDE).
 
What is plotted
---------------
Generalized epistasis N_e * E_{i,j} between a focal substitution delta_i and the
substitutions j that precede (j < i, contingency) or follow (j > i,
entrenchment) it, along substitution trajectories at MSDE, for several n:
 
    j < i :  E_{i,j} =  <delta_i, z_{i-1} - z_{j-1}>
    j > i :  E_{i,j} = -<delta_i, z_j   - z_i    >
    j = i :  E_{i,i} = 0
 
Stationary state
----------------
With the exact Kimura fixation probability used in the simulations,
    pi(s) = (1 - e^{-2s}) / (1 - e^{-4 N_e s}),
detailed balance gives pi(s) / pi(-s) = exp[(4N_e - 2) s], so the
time-stationary phenotype distribution is
    p(z) ∝ W(z)^{4N_e - 2} = exp[-(2N_e - 1) ||z||^2],
i.e. z ~ N(0, I / (2(2N_e - 1))), with
    E||z||^2 = n / (2(2N_e - 1)) ≈ n / (4 N_e),   E[log W] ≈ -n / (8 N_e).
(The expression d_eq = sqrt(n / 2N_e) corresponds to the haploid fixation
probability (1 - e^{-2s}) / (1 - e^{-2 N_e s}).)
 
Each trajectory starts at distance d_eq = sqrt(n / 4N_e) from the optimum, then
runs BURN_IN substitutions before the window. The burn-in lets the distance
relax to the stationary law of the substitution (jump) chain, which is what is
sampled when recording successive substitutions (substitutions occur on
average somewhat farther from the optimum than the time-averaged phenotype).
The distance along the trajectories is printed as a stationarity check.
 
One window per trajectory: windows are independent, and the bands are
+- 2 SE across trajectories.
 
Dependencies (earlier notebook cells)
-------------------------------------
    simulate_adaptive_walk  — walk simulator (only_beneficial=False allows
                              deleterious substitutions to fix)
"""
 
from __future__ import annotations
 
import os
from typing import Dict, List, Tuple
 
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
 
FloatArray = NDArray[np.float64]
 
# =============================================================================
# 1. Parameters
# =============================================================================
STI: float = 0.1                   # sigma_tilde; sigma = STI / sqrt(n)
NE: float = 1e5                    # effective population size
N_DIMS: List[int] = [5, 10, 50, 100]
WINDOW: int = 15                   # substitutions shown on EACH side of the focal one
BURN_IN: int = 100                  # substitutions discarded before the window
N_REP: int = 50                    # independent trajectories (one window each) per n
SEED: int = 2026
CACHE_FILE: str = "fig_msde_windows.npz"
OUT_STEM: str = "fig_msde_NeEij"
 
# Window geometry: phenotypes z_0 ... z_{2W+1}, substitutions j = 1 ... 2W+1,
# focal substitution i = W + 1  ->  W points on each side.
I_FOCAL: int = WINDOW + 1
STEPS_TOTAL: int = BURN_IN + 2 * WINDOW + 1      # substitutions simulated per trajectory
 
COLOURS: Dict[int, str] = {5: "#bc3c29", 10: "#e18727", 50: "#0072b5", 100: "#20854e"}
 
 
def d_eq(n: int, ne: float = NE) -> float:
    """Root-mean-square stationary distance to the optimum, sqrt(n / 4N_e)."""
    return np.sqrt(n / (4.0 * ne))
 
 
# =============================================================================
# 2. Simulation
# =============================================================================
def simulate_msde_windows(n: int, n_rep: int = N_REP, seed: int = SEED,
                          verbose: bool = True) -> Tuple[FloatArray, FloatArray]:
    """Simulate n_rep stationary trajectories and return one window per trajectory.
 
    Every trajectory starts from z_0 = (d_eq, 0, ..., 0) with d_eq = sqrt(n / 4N_e)
    (by isotropy, only the norm of z_0 matters); the BURN_IN substitutions
    preceding the window let the distance relax to its stationary distribution.
 
    Returns
    -------
    Z_windows : array, shape (n_rep, 2W+2, n)
        Phenotypes z_{B} ... z_{B+2W+1} of each trajectory (B = BURN_IN).
    d_path : array, shape (n_rep, STEPS_TOTAL + 1)
        Distance to the optimum along each full trajectory (stationarity check).
    """
    np.random.seed(seed)                      # simulator uses the global NumPy RNG
    Z = simulate_adaptive_walk(
        n=n, sigma_m=STI / np.sqrt(n), Ne=NE, steps=STEPS_TOTAL, replicates=n_rep,
        initial_size=d_eq(n), kimura=True, only_beneficial=False, exact=True,
        verbose=False,
    )                                                         # (n_rep, STEPS_TOTAL+1, n)
    if verbose:
        print(f"  n={n}: {n_rep} trajectories done", flush=True)
    return Z[:, BURN_IN: BURN_IN + 2 * WINDOW + 2].copy(), np.linalg.norm(Z, axis=2)
 
 
# =============================================================================
# 3. Generalized epistasis
# =============================================================================
def generalized_epistasis(Z: FloatArray, i: int = I_FOCAL) -> FloatArray:
    """E_{i,j} for j = 1 ... 2W+1; Z has shape (n_rep, 2W+2, n)."""
    delta_i = Z[:, i, :] - Z[:, i - 1, :]
    proj = np.einsum("rkn,rn->rk", Z, delta_i)               # <delta_i, z_k>
    j = np.arange(1, Z.shape[1])
    E = np.zeros((Z.shape[0], j.size))
    before, after = j < i, j > i
    E[:, before] = proj[:, [i - 1]] - proj[:, j[before] - 1]
    E[:, after] = proj[:, [i]] - proj[:, j[after]]
    return E
 
 
# =============================================================================
# 4. Plot
# =============================================================================
def plot_msde(results: Dict[int, Tuple[FloatArray, FloatArray]], outfile: str = OUT_STEM) -> None:
    """N_e E_{i,j} (mean +- 2 SE) against j - i, one curve per n."""
    mpl.rcParams.update({
        "font.family": "serif", "mathtext.fontset": "cm", "font.size": 8,
        "axes.labelsize": 9, "axes.linewidth": 0.7,
        "xtick.direction": "in", "ytick.direction": "in",
        "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "legend.frameon": False, "legend.fontsize": 8,
    })
    one_mm = 1 / 25.4
    fig, ax = plt.subplots(figsize=(110 * one_mm, 80 * one_mm), layout="constrained")
    k = np.arange(1, 2 * WINDOW + 2) - I_FOCAL                # j - i, focal at 0 with E = 0
    for n, (mean, se) in results.items():
        m = mean.copy(); m[k == 0] = 0.0
        ax.plot(k, m, "-", color=COLOURS[n], lw=1.2, label=rf"$n={n}$", zorder=3)
        ax.fill_between(k, m - 2 * se, m + 2 * se, color=COLOURS[n], alpha=0.2, lw=0, zorder=2)
    ax.plot(0, 0, "o", color="black", ms=3.5, zorder=5)
    ax.axhline(0, color="black", lw=0.6)
    ax.axvline(0, color="black", lw=0.6, ls=":")
    ax.set_xlim(-WINDOW - 0.5, WINDOW + 0.5)
    ax.set_xlabel(r"$j-i$")
    ax.set_ylabel(r"$N_e\,\mathbb{E}[\mathcal{E}_{i,j}]$")
    ax.legend(loc="upper left", title=rf"$\tilde\sigma={STI}$, $N_e=10^{{{int(np.log10(NE))}}}$",
              title_fontsize=7.5)
    fig.savefig(f"{outfile}.pdf", bbox_inches="tight")
    fig.savefig(f"{outfile}.tiff", dpi=1200, bbox_inches="tight",
                pil_kwargs={"compression": "tiff_lzw"})
    fig.savefig(f"{outfile}.png", dpi=250, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {outfile}.pdf / .tiff / .png")
 
 
# =============================================================================
# 5. Driver
# =============================================================================
if __name__ == "__main__" or "get_ipython" in globals():
    params = np.array([STI, NE, WINDOW, BURN_IN, N_REP, SEED])
    cache: Dict[str, FloatArray] = {}
    if os.path.exists(CACHE_FILE):
        with np.load(CACHE_FILE) as f:
            if np.array_equal(f["params"], params):
                cache = dict(f)
    results: Dict[int, Tuple[FloatArray, FloatArray]] = {}
    for k_n, n in enumerate(N_DIMS):
        if f"Z_{n}" not in cache:
            print(f"\nn={n}: d_eq = {d_eq(n):.5f}, y_eq = {STI / (2 * d_eq(n)):.2f}", flush=True)
            cache[f"Z_{n}"], cache[f"d_{n}"] = simulate_msde_windows(n, seed=SEED + k_n)
        Z, d_path = cache[f"Z_{n}"], cache[f"d_{n}"]
        rms_start = np.sqrt(np.mean(d_path[:, 0] ** 2))
        rms_win = np.sqrt(np.mean(d_path[:, BURN_IN:] ** 2))
        print(f"n={n}: RMS distance at start {rms_start:.5f}, over windows {rms_win:.5f} "
              f"(d_eq = {d_eq(n):.5f})")
        E = NE * generalized_epistasis(Z)
        results[n] = (E.mean(axis=0), E.std(axis=0, ddof=1) / np.sqrt(E.shape[0]))
    np.savez_compressed(CACHE_FILE, params=params, **cache)
    plot_msde(results)

# VII Theoretical results for pairwise epistasis among sucessive mutations

## VII.1 Theoretical Integral

In [ ]:
# ============================================================
# Mean pairwise epistasis  E_C[ E_{i,i+1} ] = -E_C[<delta_i, delta_{i+1}>]
# between two CONSECUTIVE substitutions in Gaussian FGM.
#
#   weight='beneficial' : C = B  (mutations beneficial only)         -> Lambda(x)
#   weight='fixation'   : C = F  (Haldane fixation weight pi(s)~2s)  -> M(x)
#
# Fixation = multiply the mutational density by s = -r d cos(theta) - r^2/2.
# On the analytic radial (gamma) moments this just RAISES the moment orders
# by +1 relative to the beneficial-only case. Same integration method,
# same cost. Validated nan-free and accurate for n = 5, 10, 100.
# ============================================================

from numpy.polynomial.legendre import leggauss
from scipy.special import gammainc, gamma as sc_gamma
from scipy.interpolate import interp1d

_N_GL = 50   # 50 GL points -> error ~1e-12 for smooth integrands

def _gl_nodes_weights(a, b, n=_N_GL):
    xi, wi = leggauss(n)
    return 0.5*(b-a)*xi + 0.5*(b+a), 0.5*(b-a)*wi

def _gamma_moment(power, r_max, sigma):
    """int_0^{r_max} r^power exp(-r^2/(2 sigma^2)) dr  (vectorised over r_max)."""
    s = (power + 1) / 2
    u = np.asarray(r_max)**2 / (2 * sigma**2)
    return (sigma**(power+1)) * (2**((power-1)/2)) * sc_gamma(s) * gammainc(s, u)


def _E_r_cos_phi_single(d, sigma, n, weight='fixation', n_gl=_N_GL):
    """Q_C(d) = E_C[r cos(phi) | d], the mean radial projection of the
       NEXT substitution sampled from distance d (< 0, points to optimum)."""
    if d <= 1e-12:
        return 0.0
    phi, wphi = _gl_nodes_weights(np.pi/2, np.pi, n_gl)
    cos_p, sin_p = np.cos(phi), np.sin(phi)     # cos_p < 0 on beneficial cone
    r_max   = -2 * d * cos_p
    sin_fac = sin_p**(n - 2)

    if weight == 'beneficial':                  # w = 1
        K2_nodes = sin_fac * _gamma_moment(n-1, r_max, sigma)
        num_nodes = sin_fac * cos_p * _gamma_moment(n, r_max, sigma)
    else:                                       # w = s = -r d cos - r^2/2  (orders +1)
        m_n  = _gamma_moment(n,   r_max, sigma)
        m_n1 = _gamma_moment(n+1, r_max, sigma)
        m_n2 = _gamma_moment(n+2, r_max, sigma)
        K2_nodes  = sin_fac * (-d*cos_p*m_n - 0.5*m_n1)
        num_nodes = sin_fac * (-d*cos_p*cos_p*m_n1 - 0.5*cos_p*m_n2)

    K2 = np.dot(wphi, K2_nodes)
    if abs(K2) < 1e-300:
        return np.nan
    return np.dot(wphi, num_nodes) / K2


def _build_E_rcos_interpolator(d, sigma, n, weight='fixation', n_grid=80, n_gl=_N_GL):
    """Precompute Q_C on Chebyshev nodes in (0, d) -> cubic interpolator."""
    k = np.arange(1, n_grid + 1)
    d_grid = 0.5 * d * (1 - np.cos(np.pi * (2*k - 1) / (2*n_grid)))
    d_grid = np.clip(d_grid, 1e-6*d, d*(1 - 1e-6))
    vals = np.array([_E_r_cos_phi_single(di, sigma, n, weight, n_gl) for di in d_grid])
    mask = np.isfinite(vals)
    if mask.sum() < 2:
        return lambda x: np.zeros_like(x)
    return interp1d(d_grid[mask], vals[mask], kind='cubic',
                    bounds_error=False, fill_value=0.0)


def mean_epistasis(d, sigma, n, weight='fixation',
                   n_gl=_N_GL, n_grid=80, return_ip=False):
    """
    Mean pairwise epistasis E_C[E_{i,i+1}] between consecutive substitutions.
    Returns -E_C[<delta_i, delta_{i+1}>] (set return_ip=True for the raw
    inner product). weight in {'beneficial','fixation'}.
    """
    interp_Q = _build_E_rcos_interpolator(d, sigma, n, weight, n_grid, n_gl)

    theta, wtheta = _gl_nodes_weights(np.pi/2 + 1e-10, np.pi - 1e-10, n_gl)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    r_max_t   = -2 * d * cos_t
    sin_fac_t = sin_t**(n - 2)

    # normalisation of the FIRST-substitution density (analytic)
    if weight == 'beneficial':
        K2_nodes = sin_fac_t * _gamma_moment(n-1, r_max_t, sigma)
    else:
        K2_nodes = sin_fac_t * (-d*cos_t*_gamma_moment(n, r_max_t, sigma)
                                - 0.5*_gamma_moment(n+1, r_max_t, sigma))
    K2_total = np.dot(wtheta, K2_nodes)
    if abs(K2_total) < 1e-300:
        return np.nan

    # numerator  E_C[ P_i * Q_C(d_i) ]  (inner GL over r; d_i couples r, theta)
    num_th = np.zeros(n_gl)
    for k in range(n_gl):
        ct, st, rmax = cos_t[k], sin_t[k], r_max_t[k]
        if rmax <= 1e-14:
            continue

        r_cap    = sigma * (np.sqrt(n) + 12.0)   # support effectif de r^{n-1} e^{-r^2/2σ^2}
        r_arr, wr = _gl_nodes_weights(0.0, min(rmax, r_cap), n_gl)
        r_arr = np.clip(r_arr, 1e-14, None)
        d_new = np.sqrt(d*d + 2*r_arr*d*ct + r_arr*r_arr)
        w     = 1.0 if weight == 'beneficial' else (-r_arr*d*ct - 0.5*r_arr*r_arr)
        base  = r_arr**(n-1) * np.exp(-r_arr*r_arr/(2*sigma**2)) * st**(n-2) * w
        P_i   = (r_arr*r_arr + r_arr*d*ct) / d_new
        num_th[k] = np.dot(wr, P_i * base * interp_Q(d_new))

    ip = np.dot(wtheta, num_th) / K2_total     # = E_C[<delta_i, delta_{i+1}>]
    return ip if return_ip else -ip            # mean epistasis = -inner product


In [ ]:
#Jeu de paramètres pour le test
SIGMA = 0.1
ds_test = [0.1,0.75,1.0, 2]
n_test = 15
results_sim = dict()
for d in ds_test:
    print(f"Testing d={d} ...")
    # Simulation of mutations and selection
    results_sim[d] = mean_epistasis(d, SIGMA, n_test)
    print(f"  E[<x_i, x_ip1>] = {results_sim[d]:.6f}")

## VII.2 Simulations par adaptative walk 

In [ ]:
# ============================================================
# E[<x_i, x_{i+1}>] — simulation using simulate_adaptive_walk
# 2 steps only, many replicates
# ============================================================

def E_scalar_product_simulation(d, sigma, n, n_rep=5000, Ne=10**5):
    """
    Estimate E[<x_i, x_{i+1}>] using simulate_adaptive_walk with 2 steps.
    
    For each replicate:
      - Start at z_0 = (d, 0, ..., 0)
      - Run 2 fixation steps
      - Record <x_1, x_2> = <z_1 - z_0, z_2 - z_1>
    
    Parameters
    ----------
    d     : initial distance to optimum
    sigma : std of each coordinate (sigma_m in the walk)
    n     : phenotypic dimension
    n_rep : number of replicates
    Ne    : effective population size (large → fixation prob ≈ 2s)
    """
    initial_size = d  # z_0 = (d, 0, ..., 0) by convention in simulate_adaptive_walk

    # Run the walk: 2 steps, n_rep replicates
    # Z has shape (n_rep, steps+1, n) = (n_rep, 3, n)
    #EXACT =false to compare with the integral 
    Z = simulate_adaptive_walk(
        n          = n,
        sigma_m    = sigma,
        Ne         = Ne,
        steps      = 2,
        replicates = n_rep,
        initial_size = initial_size,
        kimura=True,
        only_beneficial=True,
        exact=True,
        verbose=False

    )
    # Z[:, 0, :] = z_0,  Z[:, 1, :] = z_1,  Z[:, 2, :] = z_2

    # Compute mutations at each step
    x_i   = Z[:, 1, :] - Z[:, 0, :]   # shape (n_rep, n)
    x_ip1 = Z[:, 2, :] - Z[:, 1, :]   # shape (n_rep, n)

    # Scalar products <x_i, x_{i+1}> for each replicate
    scalar_products = np.sum(x_i * x_ip1, axis=1)  # shape (n_rep,)

    return np.mean(scalar_products)



In [ ]:
#Jeu de paramètres pour le test
SIGMA = 0.1
Ne = 1e5
ds_test = [0.75,1.0, 2]
n_test = 15
results_sim_walk = dict()
for d in ds_test:
    print(f"Testing d={d} with simulate_adaptive_walk ...")
    val = E_scalar_product_simulation(d, SIGMA, n_test, n_rep=20_000, Ne=Ne)
    results_sim_walk[d] = val
    print(f"  E[<x_i, x_ip1>] ≈ {val:.6f}")

## VII.3 $\epsilon$ as a function of y 

In [ ]:
# ============================================================================
# FIGURE A — Pairwise epistasis  E[ E_{i,i+1} ]  vs  y = sti/(2d),  n = 5,10,100
# Fixation weighting (pi(s) ~ 2s) '( Simulations with true fixation probabilities and Ne=10^5).  sti = 0.1 fixed.   

# Convention:  E_{i,i+1} = -<delta_i, delta_{i+1}> ;  sigma = sti/sqrt(n), d = sti/(2y)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

# ── Publication style ───────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10,
    "axes.linewidth": 0.8, "lines.linewidth": 1.6,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "xtick.major.size": 4, "ytick.major.size": 4,
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "xtick.minor.size": 2.2, "ytick.minor.size": 2.2,
    "legend.frameon": False, "legend.fontsize": 8.5,
    "savefig.dpi": 600,
})

# ── Parameters ──────────────────────────────────────────────────────────────
ONE_MM = 1 / 25.4
STI    = 0.1                       # fixed typical mutational norm (overall scale)
NE     = 10**5                    # large Ne -> Kimura fixation prob ~ 2s
N_DIMS = [5, 10, 100]

Y_MIN, Y_MAX = 0.005, 2.0
N_Y_THEORY   = 220
Y_VIOLINS    = [0.02, 0.08, 0.4, 1.0]
N_REP        =10000               # raise toward 1e4 for tighter means at n=100
RNG          = np.random.default_rng(0)  # reproducible

COLOURS      = {5: "#0072B5", 10: "#BC3C29", 100: "#20854E"}   # NEJM palette
JITTER_LOG   = {5: -0.075, 10: 0.0, 100: +0.075}
VIOLIN_WIDTH = 0.07

# ── Thin wrappers around the reused functions (NOT the integral/sampler) ─────
def epi_exact(y, n, sti=STI):
    """Exact mean epistasis from the fixation-weighted integral."""
    d, sigma = sti / (2.0 * y), sti / np.sqrt(n)
    return mean_epistasis(d, sigma, n, weight='fixation')

def simulate_epistasis_pairs(y, n, sti=STI, n_rep=N_REP, Ne=NE,rng=None):
    """2-step adaptive walk with fixation weighting; returns epistasis samples.
    Rosenblatt sampler"""
    d, sigma = sti / (2.0 * y), sti / np.sqrt(n)
    print(f"Simulating n={n}, y={y:.3f} -> d={d:.5f}, sigma={sigma:.5f} ...", flush=True)
    z=np.zeros(n)
    z[0] = d
    if rng is None:
        rng = np.random.default_rng(0)
    m1,m2,_=sample_two_successive_mutations_cdf(z, sigma, n_rep=n_rep, rng=rng, probability_of_fixation="exact",Ne=Ne)
    epistasie = -np.einsum("ij,ij->i", m1, m2)
    return epistasie

def draw_violin(ax, x_center, vals, color, marker_color=None,
                width_log=VIOLIN_WIDTH):
    """KDE violin (10-90 pct) on a log-x axis, with the sample MEAN +/- SEM
       (the quantity the theory curve predicts), not the median."""
    marker_color = marker_color or color
    w_lin      = x_center * (10**width_log - 1)
    v_lo, v_hi = np.percentile(vals, [10, 90])
    y_grid     = np.linspace(v_lo, v_hi, 300)
    dens       = gaussian_kde(vals, bw_method="scott")(y_grid)
    dens       = dens / dens.max() * w_lin
    ax.fill_betweenx(y_grid, x_center - dens, x_center + dens,
                     color=color, alpha=0.35, lw=0, zorder=4)
    ax.plot(x_center - dens, y_grid, color=color, lw=0.4, zorder=5)
    ax.plot(x_center + dens, y_grid, color=color, lw=0.4, zorder=5)
    m   = np.mean(vals)
    #sem = np.std(vals, ddof=1) / np.sqrt(len(vals))
    ax.errorbar(x_center, m, fmt="o", ms=3.5, color="white",
                mec=marker_color, mew=0.9, ecolor=marker_color,
                elinewidth=0.9, capsize=1.8, zorder=6)

# ── Compute ─────────────────────────────────────────────────────────────────
y_th = np.logspace(np.log10(Y_MIN), np.log10(Y_MAX), N_Y_THEORY)

print("Exact integrals (fixation)...", flush=True)
exact_curves = {n: np.array([epi_exact(y, n) for y in y_th]) for n in N_DIMS}

print("Simulations (kimura=True)...", flush=True)
rng = np.random.default_rng(0)
sim_data = {n: {yv: simulate_epistasis_pairs(yv, n, sti=STI, n_rep=20000, rng=rng)
                for yv in Y_VIOLINS} for n in N_DIMS}
# ── Plot ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(88 * ONE_MM, 82 * ONE_MM))
ax  = fig.add_axes([0.16, 0.15, 0.80, 0.80])

for n in N_DIMS:
    ax.plot(y_th, exact_curves[n], color=COLOURS[n], zorder=3, label=rf"$n={n}$")

ax.axhline(0.0, color="black", lw=0.6, zorder=1)

for n in N_DIMS:
    for yv in Y_VIOLINS:
        draw_violin(ax, yv * 10**JITTER_LOG[n], sim_data[n][yv], COLOURS[n])

ax.set_xscale("log")
ax.set_xlim(Y_MIN, Y_MAX)
ax.set_xlabel(r"$y = \tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$\mathcal{E}_{i,i+1}$")
ax.legend(loc="lower right", handlelength=1.4)
#ax.text(-0.20, 1.02, "A", transform=ax.transAxes, fontsize=13, fontweight="bold")

fig.savefig("fig_epistasis_A.pdf", bbox_inches="tight")
fig.savefig("fig_epistasis_A.tiff", dpi=600, bbox_inches="tight")
plt.close(fig)
print("Saved: fig_epistasis_A.pdf / .tiff")

In [ ]:
# ============================================================================
# FIGURE B — Pairwise epistasis, n = 20 : exact integral + Regime I (M(x))
#            + Regime II + regime-validity bands.   Fixation.  sti = 0.1.
#
# Reuses from Cell A:  epi_exact, simulate_epistasis_pairs, draw_violin,
#                      STI, NE, Y_MIN, Y_MAX, N_Y_THEORY, ONE_MM, style.
# Reuses from earlier cells:  mean_epistasis (with radial cap), simulate_adaptive_walk.
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# ── Parameters specific to this panel ───────────────────────────────────────
N_B          = 20
Y_VIOLINS_B  = [0.02, 0.08, 0.4, 1.0]
N_REP_B      =10000                       # n=20 needs only a few thousand
NE           = 10**5
# Regime-validity bands for n=20:
#   Regime I valid for  y << 1/sqrt(n) = 0.224 ;  sign change y* = 0.925/sqrt(20) ~ 0.207
#   Regime II valid for y = O(1)
Y_BOUND_I    = 0.22
Y_BOUND_II   = 0.5
MASK_II_MIN  = 0.2                       # only draw Regime II where it is defensible

COL_EXACT    = "#BC3C29"                  # exact integral (red)
COL_I        = "#0072B5"                  # Regime I
COL_II       = "#20854E"                  # Regime II
COL_VIOLIN   = "black"                    # <- black violins, distinct from red curve

# ── Special functions & asymptotics (fixation convention, E = -<.,.>) ───────
def M_fixed(x):
    """Fixation-weighted Gaussian mean  M(x) = 1/(Lambda(x) - x),
       Lambda(x) = phi(x)/(1-Phi(x)).  Replaces Lambda(x) under pi(s) ~ 2s."""
    return 1.0 / (norm.pdf(x) / norm.sf(x) - x)

def epistasis_regime_I(y, n, sti=STI):
    """Regime I (y << 1/sqrt(n)):  (sti^2/n) * M(x) * (2x - M(x)),  x = y*sqrt(n)."""
    x = y * np.sqrt(n)
    M = M_fixed(x)
    return (sti**2 / n) * M * (2.0 * x - M)

def epistasis_regime_II(y, sti=STI):
    """Regime II (y = O(1)):  (sti^2)/(4y^2) * (1+2y^2-sqrt(1+4y^4))^2.
       n -> infinity limit; finite-n correction is O(1/n)."""
    term = 1.0 + 2.0 * y**2 - np.sqrt(1.0 + 4.0 * y**4)
    return (sti**2) / (4.0 * y**2) * term**2

# ── Compute ─────────────────────────────────────────────────────────────────
y_th = np.logspace(np.log10(Y_MIN), np.log10(Y_MAX), N_Y_THEORY)

print("Exact integral (fixation), n=20...", flush=True)
exact_B = np.array([epi_exact(y, N_B) for y in y_th])
regI_B  = epistasis_regime_I(y_th, N_B)
regII_B = epistasis_regime_II(y_th)

print("Simulations (direct (r,theta) sampler, fixation)...", flush=True)
rng = np.random.default_rng(0)
sim_B = {yv: simulate_epistasis_pairs(yv, N_B, sti=STI, n_rep=N_REP_B,Ne=NE,
                                    rng=rng)
         for yv in Y_VIOLINS_B}

# ── Plot ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(88 * ONE_MM, 82 * ONE_MM))
ax  = fig.add_axes([0.16, 0.15, 0.80, 0.80])

# validity bands
ax.axvspan(Y_MIN,      Y_BOUND_I,  color=COL_I,  alpha=0.06, lw=0, zorder=0)
ax.axvspan(Y_BOUND_II, Y_MAX,      color=COL_II, alpha=0.06, lw=0, zorder=0)

# exact integral
ax.plot(y_th, exact_B, color=COL_EXACT, zorder=3, label="Exact integral")

# Regime I (drawn up to ~2x its validity for readability)
mI = y_th <= Y_BOUND_I * 2
ax.plot(y_th[mI], regI_B[mI], color=COL_I, lw=1.3, ls="--", zorder=3,
        label=r"Regime I: $\frac{\tilde\sigma^2}{n}\,M(x)\,(2x-M(x))$")

# Regime II 
mII = y_th >= MASK_II_MIN
ax.plot(y_th[mII], regII_B[mII], color=COL_II, lw=1.3, ls=":", zorder=3,
        label="Regime II")

ax.axhline(0.0, color="black", lw=0.6, zorder=1)

# black violins (mean +/- SEM marker)
for yv in Y_VIOLINS_B:
    draw_violin(ax, yv, sim_B[yv], COL_VIOLIN)

# regime labels inside the bands
ytop = 1.6 * np.nanmax(regII_B)
ax.text(np.sqrt(Y_MIN * Y_BOUND_I),  ytop, "Regime I",  color=COL_I,
        fontsize=8, ha="center", va="top")
ax.text(np.sqrt(Y_BOUND_II * Y_MAX), ytop, "Regime II", color=COL_II,
        fontsize=8, ha="center", va="top")

ax.set_xscale("log")
ax.set_xlim(Y_MIN, Y_MAX)
ax.set_xlabel(r"$y = \tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$\mathcal{E}_{i,i+1}$")
ax.set_title(rf"$n = {N_B}$")
#ax.legend(loc="lower right", handlelength=1.6)

fig.savefig("fig_epistasis_B.pdf", bbox_inches="tight")
fig.savefig("fig_epistasis_B.tiff", dpi=600, bbox_inches="tight")
plt.close(fig)
print("Saved: fig_epistasis_B.pdf / .tiff")

In [ ]:
# ============================================================================
# FIGURE C (compute) — Pairwise epistasis E[E_{i,i+1}] on the (n, y) plane.
# Fixation weighting.  REUSES mean_epistasis (integral NOT redefined here).
#
# mean_epistasis(d, sigma, n, weight='fixation') must include the radial cap
#   r_up = min(rmax, sigma*(sqrt(n)+12))   (fixes the small-y ripple).
#
# Convention:  E_{i,i+1} = -<delta_i, delta_{i+1}>  (mean_epistasis returns this).
#              sigma = sti/sqrt(n),  d = sti/(2y).
# With sti = 1 the plotted value equals the scale-free E/sti^2 = F(y,n): the
# phase diagram is universal (independent of the sti used in panels A/B).
# ============================================================================

import numpy as np

STI = 1.0     # overall scale only; the sign structure & boundary lines are sti-free

N_VALS = np.unique(np.round(
    np.logspace(np.log10(2), np.log10(100), 48)
).astype(int))
Y_VALS = np.logspace(np.log10(0.005), np.log10(1.5), 64)

def exact_epistasis(y, n, sti=STI):
    """Mean epistasis via the fixation-weighted integral (returns epistasis)."""
    d, sigma = sti / (2.0 * y), sti / np.sqrt(n)
    return mean_epistasis(d, sigma, n, weight='fixation')

# ── Compute grid ────────────────────────────────────────────────────────────
print(f"Grid: {len(N_VALS)} x {len(Y_VALS)} = {len(N_VALS)*len(Y_VALS)}", flush=True)
val_grid = np.full((len(Y_VALS), len(N_VALS)), np.nan)
for j, n in enumerate(N_VALS):
    print(f"  n={n}...", flush=True)
    for i, y in enumerate(Y_VALS):
        val_grid[i, j] = exact_epistasis(y, n)
print("Done.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors

# Takes `val_grid` (E[E_{i,i+1}] on Y_VALS x N_VALS) from the compute cell.
# Self-contained: defines its own signed-log transform.

# ── signed-log transform (compress dynamic range, keep the sign) ─────────────
LINTHRESH =5* 1e-3                       # |E| below this ~ linear; above ~ log
def signed_log(E, lin=LINTHRESH):
    E = np.asarray(E, float)
    return np.sign(E) * np.log10(1.0 + np.abs(E) / lin)
def signed_log_inv(z, lin=LINTHRESH):
    z = np.asarray(z, float)
    return np.sign(z) * lin * (10.0**np.abs(z) - 1.0)

z_grid = signed_log(val_grid)
z_min, z_max = np.nanmin(z_grid), np.nanmax(z_grid)

# ── symmetric discrete diverging colormap ───────────────────────────────────
# SAME number of levels K on each side of 0  -> 0 is centred on the colorbar
# and blue/red are mirror images (symmetric).  Raise K for a smoother gradient.
K = 16
neg_z  = np.linspace(z_min, 0.0, K + 1)[:-1]
pos_z  = np.linspace(0.0, z_max, K + 1)[1:]
levels = np.concatenate([neg_z, [0.0], pos_z])
n_bins = 2 * K

blues = plt.cm.Blues(np.linspace(0.95, 0.10, K))       # dark -> light  (toward 0)
reds  = plt.cm.Reds (np.linspace(0.10, 0.95, K))       # light -> dark  (away from 0)
cmap  = mcolors.ListedColormap(np.vstack([blues, reds]))   # mirrored intensities
norm  = mcolors.BoundaryNorm(levels, ncolors=n_bins)

# ── figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(130 * ONE_MM, 105 * ONE_MM))
N_mesh, Y_mesh = np.meshgrid(N_VALS, Y_VALS)
cf = ax.contourf(N_mesh, Y_mesh, z_grid, levels=levels, cmap=cmap, norm=norm)

# ── colorbar: round physical ticks, symmetric, log-spaced (no overlap) ──────
cb = plt.colorbar(cf, ax=ax, pad=0.02, label=r"$\mathcal{E}_{i,i+1}/\tilde{\sigma}^2$")
tick_E = np.array([-0.3, -0.1, -0.03, -0.01, -0.003, 0, 0.003, 0.01, 0.03, 0.1, 0.3])
lo, hi = signed_log_inv(z_min), signed_log_inv(z_max)
tick_E = tick_E[(tick_E >= lo) & (tick_E <= hi)]
cb.set_ticks(signed_log(tick_E))
cb.set_ticklabels(["0" if abs(v) < 1e-12 else f"{v:g}" for v in tick_E])

# ── boundary lines ──────────────────────────────────────────────────────────
X_STAR = 0.925368                      # fixation sign-change threshold
n_cont = np.linspace(N_VALS[0], N_VALS[-1], 1000)
def _seg(yv):
    m = (yv >= Y_VALS[0]) & (yv <= Y_VALS[-1]); return n_cont[m], yv[m]

xs, ys = _seg(X_STAR / np.sqrt(n_cont)); ax.plot(xs, ys, color="red",  lw=1.8, ls="-.")
xs, ys = _seg(2.0    / np.sqrt(n_cont)); ax.plot(xs, ys, color="blue", lw=1.4, ls="--")
ax.axhline(0.5, color="green", lw=1.4, ls=":")

# ── labels ──────────────────────────────────────────────────────────────────
n_lab = N_VALS[-1] * 0.9
ax.text(n_lab, X_STAR / np.sqrt(n_lab) * 0.7, r"$y = x_F^{*}/\sqrt{n}$",
        color="red",   fontsize=8, ha="right", va="bottom")
ax.text(n_lab, 2.0    / np.sqrt(n_lab) * 0.7, r"$y = 2/\sqrt{n}$",
        color="blue",  fontsize=8, ha="right", va="bottom")
ax.text(n_lab, 0.5 * 1.1, r"$y = 0.5$",
        color="green", fontsize=8, ha="right",  va="bottom")

# ── axes ────────────────────────────────────────────────────────────────────
ax.set_yscale("log")
ax.set_xlabel(r"$n$  (phenotypic complexity)", labelpad=4)
ax.set_ylabel(r"$y = \tilde{\sigma}/(2d)$",    labelpad=4)
ax.set_xlim(N_VALS[0], N_VALS[-1])
ax.set_ylim(Y_VALS[0], Y_VALS[-1])
ax.grid(alpha=0.12, which="both", color="gray")

fig.savefig("fig_epistasis_heatmap.pdf", format="pdf", bbox_inches="tight")
fig.savefig("fig_epistasis_heatmap.tiff", format="tiff", dpi=1200, bbox_inches="tight")
plt.close(fig)
print("Saved: fig_epistasis_heatmap.pdf / .tiff")

## VII.4 Convergence of mean epistasis in Regime I

In [ ]:
# ============================================================================
# FIGURE — Convergence of mean epistasis in Regime I
#
# At fixed x, the Regime-I prediction is
#       n E[eps] / sti^2  ->  M(x) (2x - M(x))   as n -> infinity.
# The quantity CHANGES SIGN at x_F* = 0.925 (negative = diminishing returns
# far from the optimum; positive near it), so the y-axis is SIGNED (symlog).
#
# Black solid line : asymptotic collapse M(x)(2x-M(x))
# Coloured points  : exact simulations at finite n (mean +/- SEM)
# Dotted lines     : connect simulations with the same n
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import norm

mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "font.size": 9, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "legend.frameon": False, "legend.fontsize": 8, "savefig.dpi": 600,
})
ONE_MM = 1 / 25.4
phi, Phibar = norm.pdf, norm.sf

# ── Parameters ──────────────────────────────────────────────────────────────
STI      = 0.1
NE       = 10**5
PROB     = "exact"
N_DIMS   = [5, 10, 100]
COLOURS  = {5: "#0072B5", 10: "#BC3C29", 100: "#20854E"}  # purple for 1000
MARKERS  = {5: 'o', 10: 's', 100: '^'}
N_REP    = 50_000              # mean converges fast; n=1000 is slow -> lower if needed
CHUNK    = 20_000               # cap memory at large n (arrays are (chunk, n))
X_GRID   = np.logspace(np.log10(0.1), np.log10(4.0), 10)
X_STAR   = 0.925368
LINTHRESH = 0.3                 # symlog: |value| below this shown linearly
RNG      = np.random.default_rng(0)

# ── Regime-I collapse target:  n E[eps]/sti^2 -> M(x)(2x - M(x)) ─────────────
_Lambda = lambda x: phi(x) / Phibar(x)
_M      = lambda x: 1.0 / (_Lambda(x) - x)
collapse_theory = lambda x: _M(x) * (2.0 * x - _M(x))

# ── Chunked mean and SEM of  n E[eps]/sti^2  at fixed (n, x) ─────────────────
def mean_sem_collapse(n, x, N, chunk=CHUNK):
    y = x / np.sqrt(n); d = STI / (2.0 * y); sigma = STI / np.sqrt(n)
    z0 = np.zeros(n); z0[0] = d
    s = s2 = 0.0; done = 0
    while done < N:
        m = min(chunk, N - done); done += m
        a, b, _ = sample_two_successive_mutations_cdf(
            z0, sigma, n_rep=m, rng=RNG, Ne=NE, probability_of_fixation=PROB)
        e = -np.einsum('ij,ij->i', a, b)
        s += e.sum(); s2 += (e * e).sum()
    mean = s / N
    sd   = np.sqrt(max(s2 / N - mean * mean, 0.0))
    scale = n / STI**2                                   # -> n E[eps]/sti^2
    return scale * mean, scale * sd / np.sqrt(N)

# ── Simulations at fixed x ──────────────────────────────────────────────────
results = {}
for n in N_DIMS:
    val, err = [], []
    for x in X_GRID:
        m, se = mean_sem_collapse(n, x, N_REP)
        val.append(m); err.append(se)
        print(f"n={n:4d} x={x:4.2f} (y={x/np.sqrt(n):.4f}): nE/sti^2={m:+.3f}", flush=True)
    results[n] = (np.array(val), np.array(err))

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(120 * ONE_MM, 92 * ONE_MM))

x_th = np.logspace(np.log10(X_GRID[0]*0.5), np.log10(X_GRID[-1]*2), 400)
ax.plot(x_th, collapse_theory(x_th), color="black", lw=1.8, zorder=4,
        label=r"$M(x)\,(2x-M(x))$")
ax.axhline(0.0, color="0.5", lw=0.6, zorder=1)
ax.axvline(X_STAR, color="0.5", lw=0.8, ls=":", zorder=1)     # sign change x_F*

for n in N_DIMS:
    c = COLOURS[n]; val, err = results[n]
    ax.plot(X_GRID, val, ls=":", lw=1.0, color=c, zorder=2)   # dotted connector
    ax.errorbar(X_GRID, val, yerr=err, fmt=MARKERS[n], ms=4.2, color=c,
                capsize=1.5, lw=0, elinewidth=0.8, zorder=3, label=rf"$n={n}$")

ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=LINTHRESH)
ax.set_xlim(X_GRID[0], X_GRID[-1])
ax.set_xlabel(r"$x = \sqrt{n}\,\tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$n\,\mathbb{E}[\mathcal{E}_{i,i+1}]/\tilde{\sigma}^2$")

# two legends (theory / n), as in the P_sign figure
leg1 = ax.legend(handles=[plt.Line2D([0], [0], color="black", lw=1.8,
                                     label=r"$M(x)(2x-M(x))$")], loc="upper left")
ax.add_artist(leg1)
handles_n = [plt.Line2D([0], [0], marker=MARKERS[n], linestyle=":", color=COLOURS[n],
                        markersize=4.5, label=rf"$n={n}$") for n in N_DIMS]
ax.legend(handles=handles_n, loc="lower right")

fig.tight_layout()
fig.savefig("fig_nEpsilon_vs_x.pdf", bbox_inches="tight")
fig.savefig("fig_nEpsilon_vs_x.tiff", dpi=1200, bbox_inches="tight")
plt.close(fig)
print("Saved: fig_nEpsilon_vs_x.pdf / .tiff")

# VIII Background dependance and Sign Epistasis

## VIII.1 Convergence of $nP_{sign}$

In [ ]:
# ============================================================================
# FIGURE — Convergence of P_sign in Regime I
#
# At fixed x, the Regime-I prediction is
#       n P_sign -> 2 x^2 Lambda(x) M(x)    as n -> infinity.
#
# Black solid line : asymptotic prediction
# Coloured points  : exact simulations at finite n (Wilson 95% CI)
# Dotted lines     : connect simulations with the same n
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import norm

# ── Style ─────────────────────────────────────────────────────────────────────
mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "font.size": 9, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "legend.frameon": False, "legend.fontsize": 8,
    "savefig.dpi": 600,
})

ONE_MM = 1 / 25.4
phi, Phibar = norm.pdf, norm.sf

# ── Parameters ────────────────────────────────────────────────────────────────
STI     = 0.1               # \tilde{\sigma}
NE      = 10**5
PROB    = "exact"
N_DIMS  = [5, 10, 100,1000]
N_DIMS  = [5, 10, 100]
COLOURS = {5: "#0072B5", 10: "#BC3C29", 100: "#20854E", 1000: "#7E57C2"}  # purple for 1000
COLOURS = {5: "#0072B5", 10: "#BC3C29", 100: "#20854E"}  

N_REP   = 100_000
X_GRID  = np.array([0.25, 0.35, 0.50, 0.70, 1.00, 1.40, 2.00])
X_GRID= np.logspace(np.log10(0.075), np.log10(5), 8)  # 0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.4,2.0,3.0,5.0
X_MIN, X_MAX = X_GRID[0], X_GRID[-1]
RNG     = np.random.default_rng(0)

# ── Regime-I theory ───────────────────────────────────────────────────────────
def _Lambda(x): return phi(x) / Phibar(x)                  # inverse Mills ratio
def _M(x):      return 1.0 / (_Lambda(x) - x)              # fixation-weighted mean
def nP_sign_theory(x): return 2.0 * x**2 * _Lambda(x) * _M(x)

# ── Wilson CI for a binomial proportion ───────────────────────────────────────
def wilson_ci(k, n, z_alpha=1.96):
    p_hat  = k / n
    denom  = 1 + z_alpha**2 / n
    centre = (p_hat + z_alpha**2 / (2 * n)) / denom
    half   = z_alpha * np.sqrt(p_hat * (1 - p_hat) / n + z_alpha**2 / (4 * n**2)) / denom
    return p_hat, centre - half, centre + half

# ── Sign-epistasis classifier for successive substitutions ────────────────────
def classify_sign(z0, a, b):
    w = lambda V: -0.5 * np.einsum("ij,ij->i", V, V)
    w0, wb, wab = w(z0), w(z0 + b), w(z0 + a + b)
    return int(((wb < w0) | (wb > wab)).sum())  # permissive OR adaptive reversal

# ── Simulations at fixed x ────────────────────────────────────────────────────
results = {}

for n in N_DIMS:
    sigma = STI / np.sqrt(n)                          # per-trait SD, with \tilde{\sigma}=STI fixed
    p_arr, lo_arr, hi_arr = [], [], []

    for x in X_GRID:
        d = STI * np.sqrt(n) / (2 * x)               # since x = sqrt(n) \tilde{\sigma} / (2d)
        z0 = np.zeros(n); z0[0] = d
        print(f"Simulating n={n:4d}, x={x:4.2f} -> d={d:.5f}, sigma={sigma:.5f} ...", flush=True)

        a, b, _ = sample_two_successive_mutations_cdf(
            z0, sigma, n_rep=N_REP, rng=RNG, Ne=NE, probability_of_fixation=PROB
        )

        k = classify_sign(np.broadcast_to(z0, a.shape), a, b)
        p, lo, hi = wilson_ci(k, N_REP)
        p_arr.append(p); lo_arr.append(lo); hi_arr.append(hi)

        print(f"n={n:4d}  x={x:4.2f}  y={x/np.sqrt(n):.4f} n_sign={k} P_sign={p:.5f}", flush=True)

    results[n] = (np.array(p_arr), np.array(lo_arr), np.array(hi_arr))

# ── Y-range ───────────────────────────────────────────────────────────────────
y_theory = nP_sign_theory(X_GRID)
y_all = [y_theory]
for n in N_DIMS:
    p, lo, hi = results[n]
    y_all += [n * np.clip(lo, 1e-12, None), n * hi]

y_all = np.concatenate(y_all)
y_min = 10 ** np.floor(np.log10(y_all[y_all > 0].min()))
y_max = 10 ** np.ceil(np.log10(y_all.max()))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(120 * ONE_MM, 90 * ONE_MM))

x_th = np.logspace(np.log10(X_MIN), np.log10(X_MAX), 400)
ax.plot(x_th, nP_sign_theory(x_th), color="black", lw=1.8,
        label=r"$2x^2\Lambda(x)M(x)$")

for n in N_DIMS:
    c = COLOURS[n]
    p, lo, hi = results[n]

    y = n * p
    yerr = np.vstack([n * (p - lo), n * (hi - p)])

    ax.plot(X_GRID, y, ls=":", lw=1.0, color=c, zorder=2)   # dotted connector
    ax.errorbar(X_GRID, y, yerr=yerr, fmt="o", ms=4.2, color=c,
                mfc=c, mec=c, capsize=1.5, lw=0, elinewidth=0.8,
                zorder=3, label=rf"$n={n}$")

# ── Axes ──────────────────────────────────────────────────────────────────────

ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(X_MIN, X_MAX)
ax.set_ylim(y_min, y_max)
ax.set_xlabel(r"$x=\sqrt{n}\,\tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$n\,P_{\mathrm{sign}}$")

# ── Legends ───────────────────────────────────────────────────────────────────
leg1 = ax.legend(handles=[plt.Line2D([0], [0], color="black", lw=1.8,
                                     label=r"$2x^2\Lambda(x)M(x)$")],
                 loc="upper left")
ax.add_artist(leg1)

handles_n = [plt.Line2D([0], [0], marker="o", linestyle=":", color=COLOURS[n],
                        markersize=4.5, label=rf"$n={n}$") for n in N_DIMS]
ax.legend(handles=handles_n, loc="lower right")

fig.tight_layout()
fig.savefig("fig_nPsign_vs_x.pdf", bbox_inches="tight")
fig.savefig("fig_nPsign_vs_x.tiff", dpi=1200, bbox_inches="tight")
plt.close(fig)

print("Saved: fig_nPsign_vs_x.pdf / .tiff")

## VIII.2 $P_{sign}(x)$

In [ ]:
# ============================================================================
# FIGURE — P_permissive and P_adaptive-reversal vs y (log-log), n = 20
#   coloured points = exact simulations (Wilson 95% CI)
#   dashed gray     = leading-order Regime-I asymptotic
#                     P_perm ~ P_rev ~ (x^2/n) Lambda(x) M(x)
#   x-axis jitter   = small multiplicative offset for readability
#
# REQUIRES: sample_two_successive_mutations_cdf(...) -> (delta1, delta2, z1)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import norm

mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "font.size": 9, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.top": True, "ytick.right": True,
    "legend.frameon": False, "legend.fontsize": 8, "savefig.dpi": 600,
})

ONE_MM = 1 / 25.4
phi, Phibar = norm.pdf, norm.sf

# ── Parameters ────────────────────────────────────────────────────────────────
STI     = 0.1
NE      = 10**5
PROB    = "exact"
N_B     = 20
N_REP   = 100_000
Y_GRID  = np.logspace(np.log10(0.01), np.log10(1.0), 10)
X_BOUND = 2.0
RNG     = np.random.default_rng(0)

# regime bands (identical to Figure B for consistency)
COL_I, COL_II = "#0072B5", "#20854E"
Y_BOUND_I, Y_BOUND_II = 0.22, 0.5

LABELS  = ["Permissive mutation", "Adaptive reversal"]
COLOURS = {"Permissive mutation": "#BC3C29", "Adaptive reversal": "#0072B5"}
JITTER  = {"Permissive mutation": 10**(-0.012), "Adaptive reversal": 10**(+0.012)}

# ── Regime-I theory for each class separately ────────────────────────────────
def _Lambda(x): return phi(x) / Phibar(x)
def _M(x):      return 1.0 / (_Lambda(x) - x)
def P_type_theory(x, n): return (x**2 / n) * _Lambda(x) * _M(x)

# ── Wilson CI ─────────────────────────────────────────────────────────────────
def wilson_ci(k, n, z_alpha=1.96):
    p_hat  = k / n
    denom  = 1 + z_alpha**2 / n
    centre = (p_hat + z_alpha**2 / (2 * n)) / denom
    half   = z_alpha * np.sqrt(p_hat * (1 - p_hat) / n + z_alpha**2 / (4 * n**2)) / denom
    return p_hat, centre - half, centre + half

# ── Classification ────────────────────────────────────────────────────────────
def classify_sign(z0, a, b):
    w = lambda V: -0.5 * np.einsum("ij,ij->i", V, V)
    w0, wb, wab = w(z0), w(z0 + b), w(z0 + a + b)
    return int((wb < w0).sum()), int((wb > wab).sum())   # permissive, reversal

# ── Simulations ───────────────────────────────────────────────────────────────
sigma = STI / np.sqrt(N_B)
k_perm = np.empty(len(Y_GRID), dtype=int)
k_rev  = np.empty(len(Y_GRID), dtype=int)

for i, y in enumerate(Y_GRID):
    d = STI / (2 * y)
    z0 = np.zeros(N_B); z0[0] = d

    a, b, _ = sample_two_successive_mutations_cdf(
        z0, sigma, n_rep=N_REP, rng=RNG, Ne=NE, probability_of_fixation=PROB
    )

    k_perm[i], k_rev[i] = classify_sign(np.broadcast_to(z0, a.shape), a, b)

    x = y * np.sqrt(N_B)
    print(
        f"n={N_B:2d}  y={y:.3f}  x={x:.3f}  "
        f"P_perm={k_perm[i]/N_REP:.4f}  P_rev={k_rev[i]/N_REP:.4f}",
        flush=True
    )




# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(120 * ONE_MM, 92 * ONE_MM))
x_lo, x_hi = Y_GRID[0] * 0.8, Y_GRID[-1] * 1.25

x_fine = np.linspace(0.03, X_BOUND, 200)
y_fine = x_fine / np.sqrt(N_B)
m = (y_fine >= Y_GRID[0]) & (y_fine <= Y_GRID[-1])

# regime-validity bands (behind everything)
ax.axvspan(x_lo,       Y_BOUND_I, color=COL_I,  alpha=0.06, lw=0, zorder=0)
ax.axvspan(Y_BOUND_II, x_hi,      color=COL_II, alpha=0.06, lw=0, zorder=0)


# theory (same for both classes at leading order)
ax.plot(y_fine[m], P_type_theory(x_fine[m], N_B), '--', color='gray', lw=1.2)

# data
for label, k in zip(LABELS, [k_perm, k_rev]):
    c = COLOURS[label]
    x_plot = Y_GRID * JITTER[label]
    ci = np.array([wilson_ci(ki, N_REP) for ki in k])
    p, lo, hi = ci[:, 0], ci[:, 1], ci[:, 2]
    yerr = np.vstack([p - lo, hi - p])

    ax.plot(x_plot, p, ':', color=c, lw=0.9, zorder=2)
    ax.errorbar(x_plot, p, yerr=yerr, fmt='o', ms=4, color=c, mfc=c, mec=c,
                capsize=1.5, lw=0, elinewidth=0.8, zorder=3, label=label)

# axes
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(Y_GRID[0]*0.75, Y_GRID[-1]*1.5); ax.set_ylim(1e-4, 1)
ax.set_xlabel(r"$y=\tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$P_{\mathrm{permissive}},\;P_{\mathrm{reversal}}$")

# legends
leg1 = ax.legend(loc="lower right", bbox_to_anchor=(1.0, 0.10))
ax.add_artist(leg1)

proxy = [
    plt.Line2D(
        [0], [0],
        color="gray",
        ls="--",
        lw=1.2,
        label=r"$\frac{x^2}{n}\Lambda(x)M(x)$"
    )
]

ax.legend(
    handles=proxy,
    loc="lower right",
    bbox_to_anchor=(1.0, 0.00),
    fontsize=8
)


# regime labels near the top of the bands
y_top = ax.get_ylim()[1]
ax.text(np.sqrt(x_lo * Y_BOUND_I),  y_top * 0.55, "Regime I",  color=COL_I,
        fontsize=8, ha="center", va="top")
ax.text(np.sqrt(Y_BOUND_II * x_hi), y_top * 0.55, "Regime II", color=COL_II,
        fontsize=8, ha="center", va="top")

ax.set_title(rf"$n = {N_B}$,  $\tilde{{\sigma}} = {STI}$")

fig.tight_layout()
fig.savefig("fig_Psign_types_log.pdf", bbox_inches="tight")
fig.savefig("fig_Psign_types_log.tiff", dpi=600, bbox_inches="tight")
plt.close(fig)

print("Saved: fig_Psign_types_log.pdf / .tiff")

## VIII.3 Magnitude of epistasis vs selective effect

In [ ]:
# ============================================================================
# FIGURE — Magnitude comparison at n = 20, vs y (log, 1e-2 .. 2), WITH the
#          Regime I / Regime II validity bands (same bounds/colours as Fig. B).
#   BLACK: violin of first-substitution selection s1 = Delta_a(z0);  square = mean.
#   RED  : violin of |epsilon|, epsilon = -<a,b>;  circle = max(sd eps, E|eps|).
#   dashed lines connect the markers; shaded bands mark the two regimes.
# REQUIRES: sample_two_successive_mutations_cdf(...) -> (delta1, delta2, z1)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import gaussian_kde

mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "font.size": 9, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "legend.frameon": False, "legend.fontsize": 8.5, "savefig.dpi": 600,
})
ONE_MM = 1 / 25.4

# ── Parameters ──────────────────────────────────────────────────────────────
STI    = 0.1
N_B    = 20
NE     = 10**5
PROB   = "exact"
N_REP  = 100_000
Y_GRID = np.logspace(-2.0, np.log10(2.0), 10)
JIT    = 0.05
COL_S  = "black"        # selection of first mutation
COL_E  = "#d62728"      # epistasis magnitude
# regime bands (identical to Figure B for consistency)
COL_I, COL_II = "#0072B5", "#20854E"
Y_BOUND_I, Y_BOUND_II = 0.22, 0.5
RNG    = np.random.default_rng(0)

# ── log-scale violin (KDE in log10, drawn on a log y-axis) ──────────────────
def log_violin(ax, x_center, vals, color, width_log=0.045, pct=(1, 99)):
    lv = np.log10(np.clip(vals, 1e-300, None))
    lo, hi = np.percentile(lv, pct)
    g = np.linspace(lo, hi, 200)
    dens = gaussian_kde(lv)(g); dens = dens / dens.max()
    w = x_center * (10**width_log - 1)
    ax.fill_betweenx(10**g, x_center - dens*w, x_center + dens*w, color=color, alpha=0.30, lw=0, zorder=3)
    ax.plot(x_center - dens*w, 10**g, color=color, lw=0.4, zorder=4)
    ax.plot(x_center + dens*w, 10**g, color=color, lw=0.4, zorder=4)

# ── Simulate ────────────────────────────────────────────────────────────────
sel, epi_abs, mag_epi, mean_sel,mean_epi = {}, {}, [], [], []
sigma = STI / np.sqrt(N_B)
for y in Y_GRID:
    d = STI / (2.0 * y); z0 = np.zeros(N_B); z0[0] = d
    a, b, _ = sample_two_successive_mutations_cdf(
        z0, sigma, n_rep=N_REP, rng=RNG, Ne=NE, probability_of_fixation=PROB)
    Delta_a_z = -(a @ z0) - 0.5 * np.einsum('ij,ij->i', a, a)
    eps = -np.einsum('ij,ij->i', a, b)
    sel[y] = Delta_a_z; epi_abs[y] = np.abs(eps)
    mean_sel.append(Delta_a_z.mean())
    mean_epi.append(np.abs(eps.mean()))
    mag_epi.append(np.sqrt(eps.std()**2 + np.mean(eps)**2))
    print(f"y={y:.3f} x={y*np.sqrt(N_B):.3f}: <Delta_1>={Delta_a_z.mean():.2e}  |eps| mag={mag_epi[-1]:.2e}  <eps>={mean_epi[-1]:.2e}", flush=True)
mean_sel = np.array(mean_sel); mag_epi = np.array(mag_epi)

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(135 * ONE_MM, 95 * ONE_MM))
x_lo, x_hi = Y_GRID[0] * 0.8, Y_GRID[-1] * 1.25

# regime-validity bands (behind everything)
ax.axvspan(x_lo,       Y_BOUND_I, color=COL_I,  alpha=0.06, lw=0, zorder=0)
ax.axvspan(Y_BOUND_II, x_hi,      color=COL_II, alpha=0.06, lw=0, zorder=0)

for y in Y_GRID:
    log_violin(ax, y * 10**(-JIT), sel[y],     COL_S)   # selection (black)
    log_violin(ax, y * 10**(+JIT), epi_abs[y], COL_E)   # |epistasis| (red)

ax.plot(Y_GRID * 10**(-JIT), mean_sel, 's--', color=COL_S, ms=5, lw=1.1, zorder=6,
        markeredgecolor="white", markeredgewidth=0.6, label=r"selection $\mathbb{E}[\Delta_{a}(z_0)]$")
ax.plot(Y_GRID * 10**(+JIT), mag_epi, 'o--', color=COL_E, ms=5, lw=1.1, zorder=6,
        markeredgecolor="white", markeredgewidth=0.6,
        label=r"epistasis magnitude $\sqrt{\mathrm{sd}\varepsilon^2 + \mathbb{E}(\varepsilon)^2}$")


ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(x_lo, x_hi)

# regime labels near the top of the bands
y_top = ax.get_ylim()[1]
ax.text(np.sqrt(x_lo * Y_BOUND_I),  y_top * 0.55, "Regime I",  color=COL_I,
        fontsize=8, ha="center", va="top")
ax.text(np.sqrt(Y_BOUND_II * x_hi), y_top * 0.55, "Regime II", color=COL_II,
        fontsize=8, ha="center", va="top")

ax.set_xlabel(r"$y = \tilde{\sigma}/(2d)$")
ax.set_ylabel(r"fitness effect (log-fitness units)")
ax.set_title(rf"$n = {N_B}$,  $\tilde{{\sigma}} = {STI}$")
ax.legend(loc="lower left")

fig.tight_layout()
fig.savefig("fig_magnitude_selection_vs_epistasis.pdf", bbox_inches="tight")
fig.savefig("fig_magnitude_selection_vs_epistasis.tiff", dpi=600, bbox_inches="tight")
plt.close(fig)
print("Saved: fig_magnitude_selection_vs_epistasis.pdf / .tiff")

In [ ]:
# ============================================================================
# FIGURE — Signed directional bias of epistasis  E[eps] / sd(eps)  vs y,
#          n = 5, 10, 100.  epsilon = -<a,b> (consecutive fixed substitutions).
#
# Reads directly:
#   * eps < 0 (diminishing returns) far from the optimum, |ratio| ~ 1/sqrt(n)
#     -> weak NEGATIVE bias, buried in pair-to-pair fluctuations (|.|<1 band);
#   * sign change at y* = x_F*/sqrt(n)  (thin coloured verticals);
#   * eps > 0 near the optimum, ratio grows past 1 -> strong POSITIVE bias.
# Signed y-axis on a symlog scale; +/-1 mark |mean| = fluctuations.
#
# REQUIRES: sample_two_successive_mutations_cdf(...) -> (delta1, delta2, z1)
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "font.size": 9, "axes.labelsize": 10, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "legend.frameon": False, "legend.fontsize": 8.5, "savefig.dpi": 600,
})
ONE_MM = 1 / 25.4

# ── Parameters ──────────────────────────────────────────────────────────────
STI      = 0.1
NE       = 10**5
PROB     = "exact"
N_DIMS   = [5, 10, 100]
COLOURS  = {5: "#0072B5", 10: "#BC3C29", 100: "#20854E"}
MARKERS  = {5: 'o', 10: 's', 100: '^'}
N_REP    = 100_000
Y_GRID   = np.logspace(-2.0, np.log10(2.0), 16)
X_STAR   = 0.925368
LINTHRESH = 0.05                      # symlog: |ratio| below this is shown linearly
RNG      = np.random.default_rng(0)

# ── Simulate signed ratio E[eps]/sd(eps) ────────────────────────────────────
ratio = {}
for n in N_DIMS:
    sigma = STI / np.sqrt(n); r = np.empty(len(Y_GRID))
    for i, y in enumerate(Y_GRID):
        d = STI / (2.0 * y); z0 = np.zeros(n); z0[0] = d
        a, b, _ = sample_two_successive_mutations_cdf(
            z0, sigma, n_rep=N_REP, rng=RNG, Ne=NE, probability_of_fixation=PROB)
        eps = -np.einsum('ij,ij->i', a, b)
        r[i] = eps.mean() / eps.std()
        print(f"n={n:3d} y={y:.3f}: E[eps]/sd={r[i]:+.3f}", flush=True)
    ratio[n] = r

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(135 * ONE_MM, 98 * ONE_MM))

# "fluctuation-dominated" band |ratio| < 1, and the +/-1 and 0 references
ax.axhspan(-1, 1, color="0.5", alpha=0.08, lw=0, zorder=0)
ax.axhline(0,  color="0.4", lw=0.7, zorder=1)
for hv in (1, -1):
    ax.axhline(hv, color="0.5", lw=0.8, ls="--", zorder=1)

for n in N_DIMS:
    c = COLOURS[n]
    ax.axvline(X_STAR / np.sqrt(n), color=c, ls=":", lw=0.8, alpha=0.4, zorder=1)  # y*(n)
    ax.plot(Y_GRID, ratio[n], color=c, lw=0.8, alpha=0.55, zorder=2)               # thin guide
    ax.plot(Y_GRID, ratio[n], MARKERS[n], ms=4.5, color=c, zorder=3, label=rf"$n={n}$")

ax.set_xscale("log")
ax.set_yscale("symlog", linthresh=LINTHRESH)
ax.set_xlim(Y_GRID[0] * 0.9, Y_GRID[-1] * 1.1)
y_bornes=8
ax.set_ylim(-y_bornes, y_bornes)
ax.set_xlabel(r"$y = \tilde{\sigma}/(2d)$")
ax.set_ylabel(r"$\mathbb{E}[\varepsilon] \,/\, \mathrm{sd}(\varepsilon)$")

# regime annotations
ax.text(Y_GRID[0]*1.15, 3.0, "strong positive directional bias",
        fontsize=7.5, color="0.25", va="center")
ax.text(Y_GRID[0]*1.15,-3.0, "strong negative directional bias",
        fontsize=7.5, color="0.25", va="center")

ax.text(Y_GRID[0]*1.15,0.1, "fluctuations dominate",
        fontsize=7.5, color="0.25", va="center")


ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig("fig_epistasis_signed_bias.pdf", bbox_inches="tight")
fig.savefig("fig_epistasis_signed_bias.tiff", dpi=600, bbox_inches="tight")
plt.close(fig)
print("Saved: fig_epistasis_signed_bias.pdf / .tiff")